In [2]:
# STEP 0 - setup: packages + Google Drive (outputs are saved to MyDrive/AsiaCoastGI)
!pip -q install rasterio pyogrio remotezip
import os, shutil
from google.colab import drive
# a failed mount can leave an ordinary local folder at the mount point -> remove it (never touches Drive)
if os.path.exists('/content/drive') and not os.path.ismount('/content/drive'):
    shutil.rmtree('/content/drive')
drive.mount('/content/drive', force_remount=True)
print("ismount:", os.path.ismount('/content/drive'))
for root in ['/content/drive/MyDrive', '/content/drive/MyDrive/AsiaCoastGI', '/content/drive/MyDrive/AsiaCoastGI/step1', '/content/drive/MyDrive/AsiaCoastGI/step2']:
    print(root, '->', sorted(os.listdir(root))[:40] if os.path.isdir(root) else 'MISSING')

Mounted at /content/drive
ismount: True
/content/drive/MyDrive -> ['AsiaCoastGI', 'Colab Notebooks']
/content/drive/MyDrive/AsiaCoastGI -> ['figs', 'step1', 'step2', 'step2_filtered', 'step3', 'step4', 'step5', 'step6', 'step8_figures.py']
/content/drive/MyDrive/AsiaCoastGI/step1 -> ['coastal_zone_EastAsia.gpkg', 'coastal_zone_SouthAsia.gpkg', 'coastal_zone_SoutheastAsia.gpkg', 'coastal_zone_WesternAsia.gpkg', 'country_region_assignment.csv', 'mask_EastAsia.tif', 'mask_SouthAsia.tif', 'mask_SoutheastAsia.tif', 'mask_WesternAsia.tif', 'quicklook_EastAsia.png', 'quicklook_SouthAsia.png', 'quicklook_SoutheastAsia.png', 'quicklook_WesternAsia.png', 'region_areas_km2.csv']
/content/drive/MyDrive/AsiaCoastGI/step2 -> ['areas30m.csv', 'frac_EastAsia_1990.tif', 'frac_EastAsia_2000.tif', 'frac_EastAsia_2010.tif', 'frac_EastAsia_2020.tif', 'frac_EastAsia_2022.tif', 'frac_SouthAsia_1990.tif', 'frac_SouthAsia_2000.tif', 'frac_SouthAsia_2010.tif', 'frac_SouthAsia_2020.tif', 'frac_SouthAsia_2022.tif

In [ ]:
# STEP 1 - 50 km marine coastal zone of Asia (no Earth Engine quota used)
"""STEP 1 (Colab) — 50 km marine coastal zone of Asia, 4 regions, 300 m EASE-Grid 2.0.
No Earth Engine compute. Inputs (open, citable):
  * GSHHG v2.3.7 full-resolution shoreline, level 1 = land/ocean boundary
    (Wessel & Smith 1996). Caspian/Aral are level-2 lakes -> not ocean.
  * Natural Earth 1:10m Admin-0 countries (SUBREGION = UN M49 sub-region).
Outputs -> OUT_DIR (Google Drive):
  mask_<Region>.tif  band1 = coastal land zone (1/0), band2 = seaward 20 km band
                     (filled with reclaimed land in Step 2)
  region_areas_km2.csv, quicklook_<Region>.png, coastal_zone_<Region>.gpkg (1.5 km)
"""
import os, subprocess, zipfile, time
import numpy as np, pandas as pd, geopandas as gpd
import rasterio
from rasterio.features import rasterize, shapes
from rasterio.transform import from_origin
from scipy import ndimage as ndi
from shapely.geometry import box, Polygon, shape
from shapely.ops import unary_union
import cv2, gc

def edt(nonzero):
    """Euclidean distance (m) from each nonzero cell to the nearest zero cell.
    OpenCV: float32 output, no index arrays -> ~4x less RAM than scipy."""
    return cv2.distanceTransform(nonzero.astype(np.uint8), cv2.DIST_L2, cv2.DIST_MASK_PRECISE) * RES

OUT_DIR = os.environ.get("OUT_DIR", "/content/drive/MyDrive/AsiaCoastGI/step1")
WORK = "/content/work"
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(WORK, exist_ok=True)

CRS = "EPSG:6933"          # EASE-Grid 2.0 global (cylindrical equal-area)
RES = 300.0
BUFFER_M, SEAWARD_M = 50_000, 20_000
REGIONS = {1: ("EastAsia", "Eastern Asia", (97, 17, 150, 47)),
           2: ("SouthAsia", "Southern Asia", (43, -1, 98, 40)),
           3: ("SoutheastAsia", "South-Eastern Asia", (91, -12, 142, 29)),
           4: ("WesternAsia", "Western Asia", (24, 11, 64, 44))}
# disputed island groups excluded (sovereignty unresolved, negligible area)
EXCLUDE_ADMIN = {"Spratly Islands", "Scarborough Reef"}
THRACE = Polygon([(25.5, 40.0), (26.2, 40.05), (26.7, 40.4), (27.5, 40.65), (28.9, 40.95),
                  (29.05, 41.2), (29.15, 41.25), (29.3, 42.3), (25.5, 42.3)])

def log(*a): print(time.strftime("%H:%M:%S"), *a, flush=True)

def fetch(urls, dest):
    """Download from the first working mirror; never keep a partial/empty file."""
    if os.path.exists(dest) and os.path.getsize(dest) > 1e5:
        return dest
    for url in ([urls] if isinstance(urls, str) else urls):
        log("download", url)
        r = subprocess.run(["wget", "-q", "--tries=3", "-O", dest, url])
        if r.returncode == 0 and os.path.getsize(dest) > 1e5:
            return dest
        if os.path.exists(dest): os.remove(dest)
    raise RuntimeError(f"all mirrors failed for {dest}")

# ---------------------------------------------------------------- inputs
gshhg_zip = fetch(["https://www.soest.hawaii.edu/pwessel/gshhg/gshhg-shp-2.3.7.zip",
                   "https://www.ngdc.noaa.gov/mgg/shorelines/data/gshhg/latest/gshhg-shp-2.3.7.zip"],
                  f"{WORK}/gshhg.zip")
ne_zip = fetch("https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip",
               f"{WORK}/ne_countries.zip")
with zipfile.ZipFile(gshhg_zip) as z:
    members = [m for m in z.namelist() if "GSHHS_shp/f/GSHHS_f_L1" in m]
    z.extractall(f"{WORK}/gshhg", members)
land_ll = gpd.read_file(f"{WORK}/gshhg/GSHHS_shp/f/GSHHS_f_L1.shp")
ne = gpd.read_file(f"zip://{ne_zip}")
log("GSHHG L1 polygons:", len(land_ll), "| NE countries:", len(ne))

sub_to_code = {v[1]: k for k, v in REGIONS.items()}
ne["region"] = ne["SUBREGION"].map(sub_to_code).fillna(0).astype(int)
ne.loc[ne["ADMIN"].isin(EXCLUDE_ADMIN), "region"] = 0
tr = ne["ADMIN"] == "Turkey"
ne.loc[tr, "geometry"] = ne.loc[tr, "geometry"].difference(THRACE)
assign = ne[ne.region > 0][["ADMIN", "SUBREGION", "region"]].sort_values(["region", "ADMIN"])
assign.to_csv(f"{OUT_DIR}/country_region_assignment.csv", index=False)
log("countries assigned:", assign.groupby("region").size().to_dict())

# ---------------------------------------------------------------- per region
rows = []
for code, (name, sub, (x0, y0, x1, y1)) in REGIONS.items():
    log("====", name)
    if os.path.exists(f"{OUT_DIR}/mask_{name}.tif") and os.path.exists(f"{OUT_DIR}/quicklook_{name}.png"):
        with rasterio.open(f"{OUT_DIR}/mask_{name}.tif") as s:
            z_ = s.read(1); sea_ = s.read(2)
        rows.append(dict(region=code, name=name, land_zone_km2=round(z_.sum() * (RES/1000)**2, 1),
                         seaward_band_km2=round(sea_.sum() * (RES/1000)**2, 1), grid_w=s.width, grid_h=s.height))
        log(name, "already done -> skipped"); del z_, sea_; continue
    bb_ll = box(x0 - 1, y0 - 1, x1 + 1, y1 + 1)
    land = land_ll[land_ll.intersects(bb_ll)].copy()
    land["geometry"] = land.geometry.intersection(bb_ll)
    land = land.to_crs(CRS)
    cty = ne[ne.intersects(bb_ll)].to_crs(CRS)
    gx0, gy0, gx1, gy1 = gpd.GeoSeries([box(x0, y0, x1, y1)], crs="EPSG:4326").to_crs(CRS).total_bounds
    gx0, gy1 = np.floor(gx0 / RES) * RES, np.ceil(gy1 / RES) * RES
    W, H = int(np.ceil((gx1 - gx0) / RES)), int(np.ceil((gy1 - gy0) / RES))
    T = from_origin(gx0, gy1, RES, RES)
    log(f"grid {W} x {H} = {W*H/1e6:.0f} M cells")

    L = rasterize(((g, 1) for g in land.geometry), (H, W), transform=T, fill=0, dtype="uint8",
                  all_touched=False)
    R = rasterize(((g, int(r)) for g, r in zip(cty.geometry, cty.region)), (H, W), transform=T,
                  fill=0, dtype="uint8")
    # NE vs GSHHG coastline mismatch: give unlabelled GSHHG land the nearest label (<= 6 km)
    for _ in range(20):
        gap = (L == 1) & (R == 0)
        if not gap.any():
            break
        R = np.where(gap, ndi.grey_dilation(R, size=3), R)
    # but never relabel land that NE assigns to a non-study country
    other = rasterize(((g, 1) for g, r in zip(cty.geometry, cty.region) if r == 0), (H, W),
                      transform=T, fill=0, dtype="uint8")
    R[other == 1] = 0

    d_ocean = edt(L == 1)   # land -> ocean
    zone = (L == 1) & (R == code) & (d_ocean <= BUFFER_M)
    del d_ocean
    d_land = edt(L == 0)    # ocean -> land
    # seaward band next to THIS region's zone only
    near = edt(~zone)
    sea = (L == 0) & (d_land <= SEAWARD_M) & (near <= SEAWARD_M)
    del d_land, near

    prof = dict(driver="GTiff", height=H, width=W, count=2, dtype="uint8", crs=CRS, transform=T,
                compress="deflate", tiled=True, blockxsize=512, blockysize=512, nodata=0,
                BIGTIFF="IF_SAFER")
    with rasterio.open(f"{OUT_DIR}/mask_{name}.tif", "w", **prof) as dst:
        dst.write(zone.astype("uint8"), 1); dst.write(sea.astype("uint8"), 2)
        dst.set_band_description(1, "coastal_land_zone_50km"); dst.set_band_description(2, "seaward_20km")
    km2 = zone.sum() * (RES / 1000) ** 2
    rows.append(dict(region=code, name=name, land_zone_km2=round(km2, 1),
                     seaward_band_km2=round(sea.sum() * (RES / 1000) ** 2, 1), grid_w=W, grid_h=H))
    log(f"{name}: coastal land zone = {km2:,.0f} km2")

    # quick-look + 1.5 km vector outline for maps
    f = 5
    small = zone[: H // f * f, : W // f * f].reshape(H // f, f, W // f, f).max(axis=(1, 3)).astype("uint8")
    Ts = from_origin(gx0, gy1, RES * f, RES * f)
    geoms = [shape(g) for g, v in shapes(small, mask=small == 1, transform=Ts)]
    gpd.GeoDataFrame({"region": [code], "name": [name]},
                     geometry=[unary_union(geoms)], crs=CRS).to_file(f"{OUT_DIR}/coastal_zone_{name}.gpkg")
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 8 * H / W))
    q = 20
    ax.imshow(zone[::q, ::q] * 2 + sea[::q, ::q], cmap="Greys", interpolation="nearest")
    ax.set_title(f"{name}: coastal land zone (black), seaward 20 km band (grey)"); ax.axis("off")
    fig.savefig(f"{OUT_DIR}/quicklook_{name}.png", dpi=120, bbox_inches="tight"); plt.close(fig)
    del L, R, other, zone, sea, land, cty; gc.collect()

pd.DataFrame(rows).to_csv(f"{OUT_DIR}/region_areas_km2.csv", index=False)
print(pd.DataFrame(rows).to_string(index=False))
log("STEP 1 DONE ->", OUT_DIR)


21:08:24 download https://www.soest.hawaii.edu/pwessel/gshhg/gshhg-shp-2.3.7.zip
21:09:34 download https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip
21:09:39 GSHHG L1 polygons: 179837 | NE countries: 258
21:09:39 countries assigned: {1: 8, 2: 10, 3: 11, 4: 22}
21:09:39 ==== EastAsia
21:09:41 grid 17047 x 10736 = 183 M cells
21:11:19 EastAsia: coastal land zone = 807,863 km2
21:11:22 ==== SouthAsia
21:11:23 grid 17690 x 16117 = 285 M cells
21:14:19 SouthAsia: coastal land zone = 515,868 km2
21:14:21 ==== SoutheastAsia
21:14:23 grid 16404 x 16892 = 277 M cells
21:17:16 SoutheastAsia: coastal land zone = 1,860,538 km2
21:17:19 ==== WesternAsia
21:17:19 grid 12866 x 12313 = 158 M cells
21:18:57 WesternAsia: coastal land zone = 580,817 km2
 region          name  land_zone_km2  seaward_band_km2  grid_w  grid_h
      1      EastAsia       807863.4          560860.0   17047   10736
      2     SouthAsia       515867.7          308542.7   17690   16117
      3 Southeas

In [ ]:
# STEP 2a - inspect GLC_FCS30D v2 on Zenodo (reads only the zip index + one tile, no full download)
!pip -q install remotezip
import re, rasterio
from remotezip import RemoteZip
REC = "https://zenodo.org/records/15063683/files/"
url = REC + "GLC_FCS30D_19852022maps_E120-E125.zip?download=1"
with RemoteZip(url) as z:
    infos = z.infolist()
    print(len(infos), "members; first 15:")
    for i in infos[:15]: print(f"  {i.filename}  {i.file_size/1e6:.0f} MB")
    tif = next(i for i in infos if i.filename.lower().endswith(".tif"))
    z.extract(tif, "/content/work/glc_probe")
p = "/content/work/glc_probe/" + tif.filename
with rasterio.open(p) as s:
    print(tif.filename, s.count, "bands", s.dtypes[0], s.crs, s.res, s.bounds)
    print("descriptions:", s.descriptions[:30])
    print("tags:", s.tags())

86 members; first 15:
  GLC_FCS30D_19852022maps_E120-E125/GLC_FCS30D_19852000_E120N0_5years_V1.1.tif  8 MB
  GLC_FCS30D_19852022maps_E120-E125/GLC_FCS30D_19852000_E120N10_5years_V1.1.tif  12 MB
  GLC_FCS30D_19852022maps_E120-E125/GLC_FCS30D_19852000_E120N15_5years_V1.1.tif  16 MB
  GLC_FCS30D_19852022maps_E120-E125/GLC_FCS30D_19852000_E120N20_5years_V1.1.tif  11 MB
  GLC_FCS30D_19852022maps_E120-E125/GLC_FCS30D_19852000_E120N25_5years_V1.1.tif  7 MB
  GLC_FCS30D_19852022maps_E120-E125/GLC_FCS30D_19852000_E120N30_5years_V1.1.tif  11 MB
  GLC_FCS30D_19852022maps_E120-E125/GLC_FCS30D_19852000_E120N35_5years_V1.1.tif  12 MB
  GLC_FCS30D_19852022maps_E120-E125/GLC_FCS30D_19852000_E120N40_5years_V1.1.tif  16 MB
  GLC_FCS30D_19852022maps_E120-E125/GLC_FCS30D_19852000_E120N45_5years_V1.1.tif  67 MB
  GLC_FCS30D_19852022maps_E120-E125/GLC_FCS30D_19852000_E120N50_5years_V1.1.tif  58 MB
  GLC_FCS30D_19852022maps_E120-E125/GLC_FCS30D_19852000_E120N55_5years_V1.1.tif  40 MB
  GLC_FCS30D_19852022map

In [ ]:
# STEP 2 - GLC_FCS30D land cover 1990-2022 for the coastal zone (Zenodo, no Earth Engine)
!pip -q install remotezip
import os
assert os.path.ismount("/content/drive"), "Google Drive is not mounted - run STEP 0 first"
os.environ["TEST_TILES"] = "0"   # full run (resumable: cached tiles are skipped)
"""STEP 2 (Colab) — GLC_FCS30D v2 land cover for the Asian 50 km coastal zone.
Reads only the needed 5x5 deg tiles straight out of the Zenodo zips (HTTP range
requests via remotezip), so no full 6 GB archives are downloaded. No Earth Engine.

Years: 1990 (5-yearly file, band 2) and 2000, 2010, 2020, 2022 (annual file).
Classes (7): 1 Cropland 2 Forest 3 Shrub/Grass 4 Wetland 5 Water 6 Impervious 7 Bare.

Per tile and year:
  * 30 m reclass -> 10x10 block class counts in the native lat/lon grid (~300 m)
  * counts reprojected to the region's 300 m EASE-Grid 2.0 (average) -> class %
  * 30 m class areas and 30 m transition areas inside the zone, area-weighted
    by cos(latitude) (exact for the lat/lon grid; no aggregation bias)
Resumable: every tile result is cached on Drive; finished tiles are skipped.

Outputs (Drive: AsiaCoastGI/step2):
  frac_<Region>_<year>.tif   7 bands, class % (0-100), 255 = outside study area
  lc_<Region>_<year>.tif     majority class 1-7, 0 = outside
  studymask_<Region>.tif     final mask = 50 km land zone + seaward cells that
                             were non-water in any epoch (reclamation/tidal flats)
  areas30m.csv, trans30m.csv
"""
import os, re, json, time, glob, gc
import numpy as np, pandas as pd
import rasterio
from rasterio.windows import Window, from_bounds
from rasterio.warp import reproject, Resampling
from rasterio.transform import Affine
from remotezip import RemoteZip

STEP1 = "/content/drive/MyDrive/AsiaCoastGI/step1"
OUT = "/content/drive/MyDrive/AsiaCoastGI/step2"
CACHE = f"{OUT}/tile_cache"
TMP = "/content/work/glc"
for d in (OUT, CACHE, TMP):
    os.makedirs(d, exist_ok=True)
REC = "https://zenodo.org/records/15063683/files/"
REGIONS = {1: "EastAsia", 2: "SouthAsia", 3: "SoutheastAsia", 4: "WesternAsia"}
YEARS = [1990, 2000, 2010, 2020, 2022]
PAIRS = [(1990, 2000), (2000, 2010), (2010, 2020), (2020, 2022), (1990, 2022)]
K = 7
TEST_TILES = int(os.environ.get("TEST_TILES", "0"))     # >0: only process this many tiles

LUT = np.zeros(256, np.uint8)
for v, c in [(10, 1), (11, 1), (12, 1), (20, 1),
             (51, 2), (52, 2), (61, 2), (62, 2), (71, 2), (72, 2), (81, 2), (82, 2), (91, 2), (92, 2),
             (120, 3), (121, 3), (122, 3), (130, 3), (140, 3),
             (181, 4), (182, 4), (183, 4), (184, 4), (185, 4), (186, 4), (187, 4),
             (210, 5), (190, 6),
             (150, 7), (152, 7), (153, 7), (200, 7), (201, 7), (202, 7), (220, 7)]:
    LUT[v] = c
R_EARTH = 6371007.2


def log(*a): print(time.strftime("%H:%M:%S"), *a, flush=True)


# ------------------------------------------------------------------ regions
reg = {}
for code, name in REGIONS.items():
    with rasterio.open(f"{STEP1}/mask_{name}.tif") as s:
        lz = s.read(1) > 0
        zone = np.where(lz, 1, np.where(s.read(2) > 0, 2, 0)).astype(np.uint8)   # 1 land zone, 2 seaward band
        reg[name] = dict(code=code, zone=zone, transform=s.transform, crs=s.crs,
                         shape=(s.height, s.width), profile=s.profile)
    log(name, "zone+sea cells:", int((zone > 0).sum()))
    del lz

from pyproj import Transformer
to_ease = Transformer.from_crs("EPSG:4326", "EPSG:6933", always_xy=True)
to_ll = Transformer.from_crs("EPSG:6933", "EPSG:4326", always_xy=True)


def tile_hits(L, T):
    """Regions whose zone intersects tile [L, L+5] x [T-5, T]; returns {name: lonlat bbox of zone}."""
    hits = {}
    xs, ys = to_ease.transform([L, L + 5], [T - 5, T])
    for name, r in reg.items():
        win = from_bounds(min(xs), min(ys), max(xs), max(ys), r["transform"]).round_offsets().round_lengths()
        r0, c0 = max(win.row_off, 0), max(win.col_off, 0)
        r1, c1 = min(win.row_off + win.height, r["shape"][0]), min(win.col_off + win.width, r["shape"][1])
        if r1 <= r0 or c1 <= c0:
            continue
        sub = r["zone"][r0:r1, c0:c1] > 0
        if not sub.any():
            continue
        rows, cols = np.flatnonzero(sub.any(1)), np.flatnonzero(sub.any(0))
        t = r["transform"]
        x0 = t.c + (c0 + cols[0] - 1) * t.a
        x1 = t.c + (c0 + cols[-1] + 2) * t.a
        y1 = t.f + (r0 + rows[0] - 1) * t.e
        y0 = t.f + (r0 + rows[-1] + 2) * t.e
        (lo0, lo1), (la0, la1) = to_ll.transform([x0, x1], [y0, y1])
        hits[name] = (max(lo0, L), max(la0, T - 5), min(lo1, L + 5), min(la1, T))
    return hits


def tile_id(L, T):
    return f"E{L}{'N' if T >= 0 else 'S'}{abs(T)}"


ZIP_INDEX = {}


def zip_members(L):
    a = (L // 10) * 10
    zname = f"GLC_FCS30D_19852022maps_E{a}-E{a + 5}.zip"
    if zname not in ZIP_INDEX:
        with RemoteZip(REC + zname + "?download=1") as z:
            ZIP_INDEX[zname] = [i.filename for i in z.infolist()]
    return zname, ZIP_INDEX[zname]


def fetch_tile(L, T):
    zname, names = zip_members(L)
    tid = tile_id(L, T)
    five = [n for n in names if f"_{tid}_" in n and "5years" in n]
    ann = [n for n in names if f"_{tid}_" in n and "5years" not in n and n.lower().endswith(".tif")]
    if not five or not ann:
        return None, None
    out = []
    with RemoteZip(REC + zname + "?download=1") as z:
        for n in (five[0], ann[0]):
            z.extract(n, TMP)
            out.append(os.path.join(TMP, n))
    return out


def block_counts(cls, f=10):
    """cls (H,W) uint8 0..7 -> counts (8, H/f, W/f) uint8 (index 0 = no data)."""
    H, W = (cls.shape[0] // f) * f, (cls.shape[1] // f) * f
    b = cls[:H, :W].reshape(H // f, f, W // f, f)
    return np.stack([(b == k).sum(axis=(1, 3), dtype=np.uint8) for k in range(K + 1)])


def process_tile(L, T, hits, paths=None):
    tid = tile_id(L, T)
    done = {n: os.path.exists(f"{CACHE}/{n}_{tid}.npz") for n in hits}
    if all(done.values()):
        return "cached"
    p5, pa = paths if paths is not None else fetch_tile(L, T)
    if p5 is None:
        return "missing"
    with rasterio.open(p5) as s5, rasterio.open(pa) as sa:
        assert sa.count == 23, f"annual file has {sa.count} bands"
        for name, (lo0, la0, lo1, la1) in hits.items():
            if done[name]:
                continue
            r = reg[name]
            win = from_bounds(lo0, la0, lo1, la1, sa.transform)
            # snap to 10-pixel blocks inside the tile
            c0 = max(int(np.floor(win.col_off / 10)) * 10, 0)
            r0 = max(int(np.floor(win.row_off / 10)) * 10, 0)
            c1 = min(int(np.ceil((win.col_off + win.width) / 10)) * 10, (sa.width // 10) * 10)
            r1 = min(int(np.ceil((win.row_off + win.height) / 10)) * 10, (sa.height // 10) * 10)
            if c1 <= c0 or r1 <= r0:
                continue
            w = Window(c0, r0, c1 - c0, r1 - r0)
            st = sa.window_transform(w)
            bt = st * Affine.scale(10)
            # zone codes on the 30 m tile window (nearest): 1 land zone, 2 seaward band
            zc = np.zeros((w.height, w.width), np.uint8)
            reproject(r["zone"], zc, src_transform=r["transform"], src_crs=r["crs"],
                      dst_transform=st, dst_crs="EPSG:4326", resampling=Resampling.nearest)
            lat = st.f + (np.arange(w.height) + 0.5) * st.e
            dpx = np.radians(abs(st.a)) * np.radians(abs(st.e))
            row_area = (R_EARTH ** 2 * dpx * np.cos(np.radians(lat)) / 1e6)      # km2 per pixel
            # destination window on the region grid
            xs, ys = to_ease.transform([st.c, st.c + st.a * w.width], [st.f + st.e * w.height, st.f])
            dw = from_bounds(min(xs), min(ys), max(xs), max(ys), r["transform"]).round_offsets().round_lengths()
            dw = Window(dw.col_off - 1, dw.row_off - 1, dw.width + 2, dw.height + 2)
            dt = rasterio.windows.transform(dw, r["transform"])
            res = dict(win=[int(dw.col_off), int(dw.row_off), int(dw.width), int(dw.height)])
            cls_years = {}
            for y in YEARS:
                band = 2 if y == 1990 else y - 1999
                src = s5 if y == 1990 else sa
                cls_years[y] = LUT[src.read(band, window=w)]
            # 30 m study mask = land zone + seaward cells mapped as non-water in any epoch
            ever = np.zeros(zc.shape, bool)
            for y in YEARS:
                ever |= (cls_years[y] != 0) & (cls_years[y] != 5)
            zm = (zc == 1) | ((zc == 2) & ever)
            del ever, zc
            for y in YEARS:
                cls = cls_years[y]
                cnt = block_counts(cls)
                frac = np.zeros((K + 1, dw.height, dw.width), np.float32)
                for k in range(K + 1):
                    reproject(cnt[k].astype(np.float32), frac[k], src_transform=bt, src_crs="EPSG:4326",
                              dst_transform=dt, dst_crs=r["crs"], resampling=Resampling.average,
                              src_nodata=None, dst_nodata=np.nan, init_dest_nodata=True)
                tot = np.nansum(frac, 0)
                cover = np.where(np.isnan(frac[0]), 0, 1).astype(bool)
                # no-data (ocean/no map) counts as water
                frac[5] = np.nan_to_num(frac[5]) + np.nan_to_num(frac[0])
                pct = np.where(tot > 0, np.nan_to_num(frac[1:]) / np.maximum(tot, 1e-6) * 100, 0)
                res[f"pct_{y}"] = np.where(cover, np.round(pct), 255).astype(np.uint8)
                # 30 m class areas in the zone
                a = np.bincount(cls[zm], weights=np.broadcast_to(row_area[:, None], cls.shape)[zm],
                                minlength=K + 1)
                res[f"area_{y}"] = a
            for (a0, a1) in PAIRS:
                code = cls_years[a0][zm].astype(np.int32) * 8 + cls_years[a1][zm]
                res[f"tr_{a0}_{a1}"] = np.bincount(code, weights=np.broadcast_to(row_area[:, None], zm.shape)[zm],
                                                   minlength=64)
            np.savez_compressed(f"{CACHE}/{name}_{tid}.npz", **res)
            del cls_years, zm
            gc.collect()
    for p in (p5, pa):
        os.remove(p)
    return "ok"


# ------------------------------------------------------------------ tile loop
tiles = []
for L in range(20, 150, 5):
    for T in range(50, -15, -5):
        h = tile_hits(L, T)
        if h:
            tiles.append((L, T, h))
log("tiles intersecting the coastal zone:", len(tiles))
json.dump([[L, T, list(h)] for L, T, h in tiles], open(f"{OUT}/tiles.json", "w"))
if TEST_TILES:
    tiles = tiles[:TEST_TILES]
t0 = time.time()
from concurrent.futures import ThreadPoolExecutor
todo = [(L, T, h) for L, T, h in tiles
        if not all(os.path.exists(f"{CACHE}/{n}_{tile_id(L, T)}.npz") for n in h)]
log(f"{len(tiles) - len(todo)} tiles already cached, {len(todo)} to do (next tile prefetched in background)")
pool = ThreadPoolExecutor(max_workers=1)
status = {}
fut = pool.submit(fetch_tile, todo[0][0], todo[0][1]) if todo else None
for i, (L, T, h) in enumerate(todo, 1):
    try:
        paths = fut.result()
    except Exception as e:
        paths = None
        log("prefetch failed, retrying inline:", repr(e))
    if i < len(todo):
        fut = pool.submit(fetch_tile, todo[i][0], todo[i][1])
    try:
        st = process_tile(L, T, h, paths)
    except Exception as e:
        st = f"ERROR {e!r}"
    status[tile_id(L, T)] = st
    log(f"[{i}/{len(todo)}] {tile_id(L, T)} {list(h)} -> {st}  ({(time.time() - t0) / 60:.1f} min)")
absent = [t for t, s in status.items() if s == "missing"]      # tile does not exist in GLC_FCS30D (open ocean / remote islets)
missing = [tile_id(L, T) for L, T, h in tiles
           if tile_id(L, T) not in absent
           and not all(os.path.exists(f"{CACHE}/{n}_{tile_id(L, T)}.npz") for n in h)]
log("tiles not in the GLC_FCS30D archive (no land-cover data exists):", absent)
log("tiles still missing:", missing)
if missing and not TEST_TILES:
    raise SystemExit("some tiles failed - simply re-run this cell; finished tiles are skipped")

if TEST_TILES:
    raise SystemExit("test run finished - set TEST_TILES=0 for the full run")

# ------------------------------------------------------------------ mosaics
CLS = ["Cropland", "Forest", "ShrubGrass", "Wetland", "Water", "Impervious", "Bare"]
area_rows, tr_rows = [], []
for name, r in reg.items():
    files = sorted(glob.glob(f"{CACHE}/{name}_*.npz"))
    log(name, "mosaicking", len(files), "tiles")
    H, W = r["shape"]
    prof = dict(r["profile"], count=K, dtype="uint8", nodata=255, compress="deflate", tiled=True,
                blockxsize=512, blockysize=512, BIGTIFF="IF_SAFER")
    with rasterio.open(f"{STEP1}/mask_{name}.tif") as s:
        land, sea = s.read(1) > 0, s.read(2) > 0
    everland = np.zeros((H, W), bool)
    areas = {y: np.zeros(K + 1) for y in YEARS}
    trans = {p: np.zeros(64) for p in PAIRS}
    for y in YEARS:
        mos = np.full((K, H, W), 255, np.uint8)
        for f in files:
            d = np.load(f)
            c, rr, w, h = d["win"]
            pr0, pc0 = max(rr, 0), max(c, 0)
            pr1, pc1 = min(rr + h, H), min(c + w, W)
            if pr1 <= pr0 or pc1 <= pc0:
                continue
            blk = d[f"pct_{y}"][:, pr0 - rr:pr1 - rr, pc0 - c:pc1 - c]
            tgt = mos[:, pr0:pr1, pc0:pc1]
            put = (tgt[0] == 255) & (blk[0] != 255)
            tgt[:, put] = blk[:, put]
            if y == YEARS[0]:
                for p in PAIRS:
                    trans[p] += d[f"tr_{p[0]}_{p[1]}"]
            areas[y] += d[f"area_{y}"]
        valid = mos[0] != 255
        maj = np.where(valid, mos.argmax(0) + 1, 0).astype(np.uint8)
        everland |= sea & valid & (maj != 5)
        mos[:, ~(land | sea)] = 255
        with rasterio.open(f"{OUT}/frac_{name}_{y}.tif", "w", **prof) as dst:
            dst.write(mos)
            for i, n in enumerate(CLS, 1):
                dst.set_band_description(i, f"{n}_pct")
        np.save(f"{TMP}/maj_{name}_{y}.npy", maj)
        del mos
        gc.collect()
    study = land | everland
    mprof = dict(r["profile"], count=1, dtype="uint8", nodata=0, compress="deflate", tiled=True,
                 blockxsize=512, blockysize=512)
    with rasterio.open(f"{OUT}/studymask_{name}.tif", "w", **mprof) as dst:
        dst.write(study.astype(np.uint8), 1)
    for y in YEARS:
        maj = np.load(f"{TMP}/maj_{name}_{y}.npy")
        maj[~study] = 0
        maj[study & (maj == 0)] = 5                       # unmapped cells inside mask = water
        with rasterio.open(f"{OUT}/lc_{name}_{y}.tif", "w", **mprof) as dst:
            dst.write(maj, 1)
        os.remove(f"{TMP}/maj_{name}_{y}.npy")
        for k in range(1, K + 1):
            area_rows.append(dict(region=name, year=y, cls=CLS[k - 1], km2_30m=areas[y][k],
                                  km2_300m_majority=float((maj == k).sum()) * 0.09))
    for (a0, a1), v in trans.items():
        M = v.reshape(8, 8)
        for i in range(1, K + 1):
            for j in range(1, K + 1):
                tr_rows.append(dict(region=name, t0=a0, t1=a1, frm=CLS[i - 1], to=CLS[j - 1], km2=M[i, j]))
    log(name, "study area km2:", round(study.sum() * 0.09))
pd.DataFrame(area_rows).to_csv(f"{OUT}/areas30m.csv", index=False)
pd.DataFrame(tr_rows).to_csv(f"{OUT}/trans30m.csv", index=False)
log("STEP 2 DONE ->", OUT)


04:42:48 EastAsia zone+sea cells: 15208038
04:42:50 SouthAsia zone+sea cells: 9160115
04:42:52 SoutheastAsia zone+sea cells: 37842144
04:42:53 WesternAsia zone+sea cells: 9731947
04:42:53 tiles intersecting the coastal zone: 127
04:42:53 121 tiles already cached, 6 to do (next tile prefetched in background)
04:42:54 [1/6] E60N25 ['SouthAsia', 'WesternAsia'] -> missing  (0.0 min)
04:42:55 [2/6] E130N30 ['EastAsia'] -> missing  (0.0 min)
04:42:55 [3/6] E130N5 ['SoutheastAsia'] -> missing  (0.0 min)
04:42:56 [4/6] E140N35 ['EastAsia'] -> missing  (0.1 min)
04:42:56 [5/6] E140N30 ['EastAsia'] -> missing  (0.1 min)
04:42:56 [6/6] E140N25 ['EastAsia'] -> missing  (0.1 min)
04:42:56 tiles not in the GLC_FCS30D archive (no land-cover data exists): ['E60N25', 'E130N30', 'E130N5', 'E140N35', 'E140N30', 'E140N25']
04:42:56 tiles still missing: []
04:42:56 EastAsia mosaicking 29 tiles
04:45:14 EastAsia study area km2: 847064
04:45:14 SouthAsia mosaicking 25 tiles
04:47:52 SouthAsia study area km2:

In [ ]:
# QC of Step 2 outputs
import pandas as pd, os
OUT = "/content/drive/MyDrive/AsiaCoastGI/step2"
print(sorted(f for f in os.listdir(OUT) if not f.startswith("tile_cache")))
a = pd.read_csv(f"{OUT}/areas30m.csv")
tot = a.groupby(["region", "year"])[["km2_30m", "km2_300m_majority"]].sum().round(0)
print("\nTotal mapped area per region/year (30 m vs 300 m majority):\n", tot.unstack("year")["km2_30m"])
pct = a.assign(pct=a.km2_30m / a.groupby(["region", "year"]).km2_30m.transform("sum") * 100)
pd.set_option("display.width", 250)
print("\nClass shares (% of zone, 30 m):")
print(pct.pivot_table(index=["region", "cls"], columns="year", values="pct").round(2))
t = pd.read_csv(f"{OUT}/trans30m.csv")
chg = t[(t.t0 == 1990) & (t.t1 == 2022) & (t.frm != t.to)].groupby("region").km2.sum()
print("\nGross change 1990->2022 (km2):", chg.round(0).to_dict())
ag = (a.groupby(["region","year"]).apply(lambda d: (abs(d.km2_30m - d.km2_300m_majority).sum() / d.km2_30m.sum() * 50)).round(2))
print("\n30 m vs 300 m majority disagreement (% of area):", ag.unstack().to_dict())

['areas30m.csv', 'frac_EastAsia_1990.tif', 'frac_EastAsia_2000.tif', 'frac_EastAsia_2010.tif', 'frac_EastAsia_2020.tif', 'frac_EastAsia_2022.tif', 'frac_SouthAsia_1990.tif', 'frac_SouthAsia_2000.tif', 'frac_SouthAsia_2010.tif', 'frac_SouthAsia_2020.tif', 'frac_SouthAsia_2022.tif', 'frac_SoutheastAsia_1990.tif', 'frac_SoutheastAsia_2000.tif', 'frac_SoutheastAsia_2010.tif', 'frac_SoutheastAsia_2020.tif', 'frac_SoutheastAsia_2022.tif', 'frac_WesternAsia_1990.tif', 'frac_WesternAsia_2000.tif', 'frac_WesternAsia_2010.tif', 'frac_WesternAsia_2020.tif', 'frac_WesternAsia_2022.tif', 'lc_EastAsia_1990.tif', 'lc_EastAsia_2000.tif', 'lc_EastAsia_2010.tif', 'lc_EastAsia_2020.tif', 'lc_EastAsia_2022.tif', 'lc_SouthAsia_1990.tif', 'lc_SouthAsia_2000.tif', 'lc_SouthAsia_2010.tif', 'lc_SouthAsia_2020.tif', 'lc_SouthAsia_2022.tif', 'lc_SoutheastAsia_1990.tif', 'lc_SoutheastAsia_2000.tif', 'lc_SoutheastAsia_2010.tif', 'lc_SoutheastAsia_2020.tif', 'lc_SoutheastAsia_2022.tif', 'lc_WesternAsia_1990.tif', '

/tmp/ipykernel_820/3434544143.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ag = (a.groupby(["region","year"]).apply(lambda d: (abs(d.km2_30m - d.km2_300m_majority).sum() / d.km2_30m.sum() * 50)).round(2))


In [ ]:
# STEP 2b - temporal-consistency filter (spike removal) for 1990-2022 land cover
import os
assert os.path.ismount("/content/drive"), "Google Drive is not mounted - run STEP 0 first"
"""STEP 2b (Colab) — temporal-consistency filter for the 1990–2022 land-cover series.

Rule (spike removal, all classes): for the interior epochs 2000, 2010 and 2020,
if class(t-1) == class(t+1) != class(t)  ->  class(t) := class(t-1).
Where a cell is corrected, its class-fraction vector for year t is replaced by
the mean of t-1 and t+1, so fractions and majority maps stay consistent.
End epochs (1990, 2022) are left unchanged (no two-sided evidence).

Outputs (Drive: AsiaCoastGI/step2_filtered):
  lc_<Region>_<year>.tif, frac_<Region>_<year>.tif   filtered maps (same format as step2)
  areas_filtered.csv       class areas per region/year from filtered fractions (km2)
  trans_filtered.csv       transition areas from filtered majority maps (km2)
  filter_log.csv           cells changed per region/year/from/to (for Methods/Supplement)
Processed in row blocks so it fits in Colab RAM.
"""
import os, gc, time
import numpy as np, pandas as pd
import rasterio
from rasterio.windows import Window

IN = "/content/drive/MyDrive/AsiaCoastGI/step2"
OUT = "/content/drive/MyDrive/AsiaCoastGI/step2_filtered"
os.makedirs(OUT, exist_ok=True)
REGIONS = ["EastAsia", "SouthAsia", "SoutheastAsia", "WesternAsia"]
YEARS = [1990, 2000, 2010, 2020, 2022]
PAIRS = [(1990, 2000), (2000, 2010), (2010, 2020), (2020, 2022), (1990, 2022)]
CLS = ["Cropland", "Forest", "ShrubGrass", "Wetland", "Water", "Impervious", "Bare"]
K, BLOCK, CELL_KM2 = 7, 1024, 0.09


def log(*a): print(time.strftime("%H:%M:%S"), *a, flush=True)


def spike_filter(lc, fr):
    """lc: (T, h, w) uint8 classes (0 = outside); fr: (T, K, h, w) uint8 % (255 = outside).
    Modifies in place; returns list of (t_index, from, to, count)."""
    changes = []
    for t in range(1, lc.shape[0] - 1):
        a, b, c = lc[t - 1], lc[t], lc[t + 1]
        sp = (a == c) & (b != a) & (a > 0) & (b > 0)
        if not sp.any():
            continue
        frm, to = b[sp], a[sp]
        cnt = np.bincount(frm.astype(np.int64) * 8 + to, minlength=64).reshape(8, 8)
        for i, j in zip(*np.nonzero(cnt)):
            changes.append((t, int(i), int(j), int(cnt[i, j])))
        lc[t][sp] = a[sp]
        fr[t][:, sp] = ((fr[t - 1][:, sp].astype(np.uint16) + fr[t + 1][:, sp]) // 2).astype(np.uint8)
    return changes


if __name__ == "__main__":
    area_rows, tr_rows, log_rows = [], [], []
    for name in REGIONS:
        srcs_lc = [rasterio.open(f"{IN}/lc_{name}_{y}.tif") for y in YEARS]
        srcs_fr = [rasterio.open(f"{IN}/frac_{name}_{y}.tif") for y in YEARS]
        H, W = srcs_lc[0].height, srcs_lc[0].width
        dst_lc = [rasterio.open(f"{OUT}/lc_{name}_{y}.tif", "w", **srcs_lc[0].profile) for y in YEARS]
        dst_fr = []
        for y, s in zip(YEARS, srcs_fr):
            d = rasterio.open(f"{OUT}/frac_{name}_{y}.tif", "w", **s.profile)
            for i, n in enumerate(CLS, 1):
                d.set_band_description(i, f"{n}_pct")
            dst_fr.append(d)
        area = np.zeros((len(YEARS), K))
        trans = {p: np.zeros((8, 8)) for p in PAIRS}
        nchg = {}
        for r0 in range(0, H, BLOCK):
            win = Window(0, r0, W, min(BLOCK, H - r0))
            lc = np.stack([s.read(1, window=win) for s in srcs_lc])
            if not lc.any():
                for d in dst_lc: d.write(lc[0] * 0, 1, window=win)
                for d in dst_fr: d.write(np.full((K, win.height, W), 255, np.uint8), window=win)
                continue
            fr = np.stack([s.read(window=win) for s in srcs_fr])
            for (t, i, j, n) in spike_filter(lc, fr):
                nchg[(YEARS[t], i, j)] = nchg.get((YEARS[t], i, j), 0) + n
            # clip fractions to the final study mask (lc == 0 outside; removes open sea of the seaward band)
            outside = lc[0] == 0
            for k in range(len(YEARS)):
                fr[k][:, outside] = 255
            for k, (d1, d2) in enumerate(zip(dst_lc, dst_fr)):
                d1.write(lc[k], 1, window=win)
                d2.write(fr[k], window=win)
                v = fr[k][0] != 255
                area[k] += fr[k][:, v].sum(1) / 100.0 * CELL_KM2
            for (a0, a1) in PAIRS:
                i0, i1 = YEARS.index(a0), YEARS.index(a1)
                m = lc[i0] > 0
                trans[(a0, a1)] += np.bincount(lc[i0][m].astype(np.int64) * 8 + lc[i1][m],
                                               minlength=64).reshape(8, 8) * CELL_KM2
            del lc, fr
            gc.collect()
        for s in srcs_lc + srcs_fr + dst_lc + dst_fr:
            s.close()
        study = trans[(1990, 2000)].sum()
        for k, y in enumerate(YEARS):
            for c in range(K):
                area_rows.append(dict(region=name, year=y, cls=CLS[c], km2=area[k, c],
                                      pct=area[k, c] / area[k].sum() * 100))
        for (a0, a1), M in trans.items():
            for i in range(1, K + 1):
                for j in range(1, K + 1):
                    tr_rows.append(dict(region=name, t0=a0, t1=a1, frm=CLS[i - 1], to=CLS[j - 1], km2=M[i, j]))
        for (y, i, j), n in sorted(nchg.items()):
            log_rows.append(dict(region=name, year=y, frm=CLS[i - 1], to=CLS[j - 1], cells=n,
                                 km2=n * CELL_KM2, pct_of_zone=n * CELL_KM2 / study * 100))
        tot = sum(n for n in nchg.values()) * CELL_KM2
        log(f"{name}: corrected {tot:,.0f} km2 cell-years ({tot / study / 3 * 100:.2f}% of zone per interior epoch)")
    pd.DataFrame(area_rows).to_csv(f"{OUT}/areas_filtered.csv", index=False)
    pd.DataFrame(tr_rows).to_csv(f"{OUT}/trans_filtered.csv", index=False)
    pd.DataFrame(log_rows).to_csv(f"{OUT}/filter_log.csv", index=False)
    a = pd.DataFrame(area_rows)
    pd.set_option("display.width", 250)
    print(a.pivot_table(index=["region", "cls"], columns="year", values="pct").round(2))
    log("STEP 2b DONE ->", OUT)


05:05:59 EastAsia: corrected 12,889 km2 cell-years (0.51% of zone per interior epoch)
05:07:43 SouthAsia: corrected 18,295 km2 cell-years (1.12% of zone per interior epoch)
05:09:47 SoutheastAsia: corrected 43,237 km2 cell-years (0.74% of zone per interior epoch)
05:11:02 WesternAsia: corrected 5,852 km2 cell-years (0.33% of zone per interior epoch)
year                       1990   2000   2010   2020   2022
region        cls                                          
EastAsia      Bare         0.01   0.03   0.05   0.05   0.05
              Cropland    34.23  31.57  29.72  29.61  29.82
              Forest      52.33  51.88  51.34  50.87  50.85
              Impervious   4.82   7.70  10.33  10.91  10.71
              ShrubGrass   0.95   0.94   0.85   0.83   0.83
              Water        4.36   3.30   3.28   3.80   3.79
              Wetland      3.30   4.57   4.44   3.93   3.95
SouthAsia     Bare        17.46  17.60  17.45  17.03  16.94
              Cropland    39.34  38.59  38.55  3

In [ ]:
# STEP 3a - inspect the Esri / Impact Observatory 10 m annual LULC bucket (public, no account)
import requests, re, xml.etree.ElementTree as ET
B = "https://io-10m-annual-lulc.s3.us-west-2.amazonaws.com/"
def ls(prefix="", delim="/", maxk=1000, token=None):
    p = {"list-type": "2", "prefix": prefix, "delimiter": delim, "max-keys": str(maxk)}
    if token: p["continuation-token"] = token
    r = requests.get(B, params=p, timeout=60); r.raise_for_status()
    ns = {"s": "http://s3.amazonaws.com/doc/2006-03-01/"}
    root = ET.fromstring(r.content)
    pre = [e.text for e in root.findall("s:CommonPrefixes/s:Prefix", ns)]
    keys = [(e.find("s:Key", ns).text, int(e.find("s:Size", ns).text)) for e in root.findall("s:Contents", ns)]
    nxt = root.find("s:NextContinuationToken", ns)
    return pre, keys, (nxt.text if nxt is not None else None)
top, keys, _ = ls()
print("top-level prefixes:", top, keys[:5])
for p in top[:3]:
    sub, k2, _ = ls(p)
    print(p, "->", sub[:20], k2[:5])

# full listing
allk, tok = [], None
while True:
    _, k, tok = ls(delim="", token=tok)
    allk += k
    if not tok: break
import pandas as pd
df = pd.DataFrame(allk, columns=["key", "size"])
df["year"] = df.key.str.extract(r"_(\d{4})")[0]
df["zone"] = df.key.str.extract(r"^(\d{2}[A-Z])")[0]
for y, g in df.groupby("year"):
    print(y, len(g), "files", round(g["size"].sum() / 1e9, 1), "GB")
print("non-matching keys:", df[df.zone.isna()].key.head(10).tolist())
z = df[df.zone.notna()].copy()
z["utm"] = z.zone.str[:2].astype(int); z["band"] = z.zone.str[2]
asia = z[z.utm.between(35, 55) & z.band.isin(list("LMNPQRST"))]
for y, g in asia.groupby("year"):
    print("Asia (UTM 35-55, bands L-T)", y, len(g), "files", round(g["size"].sum() / 1e9, 1), "GB")
print(df[df.zone == "50R"].to_string())

# metadata: STAC ndjson + COG tags for one zone across years
import json, rasterio
r = requests.get(B + "io-10m-annual-lulc.ndjson", timeout=120)
items = [json.loads(l) for l in r.text.splitlines() if l.strip()]
print(len(items), "STAC items; example properties keys:", list(items[0].get("properties", {}).keys()))
for it in items:
    if it.get("id", "").startswith("50R"):
        p = it["properties"]; print(it["id"], {k: p[k] for k in p if k not in ("proj:bbox", "proj:geometry", "proj:transform", "proj:shape")})
for y in (2021, 2022, 2023, 2024):
    with rasterio.open(f"/vsicurl/{B}50R_{y}.tif") as s:
        print(y, s.crs, s.res, s.width, s.height, "overviews:", s.overviews(1)[:6], "tags:", s.tags())

top-level prefixes: [] [('01C_2017.tif', 17219421), ('01C_2018.tif', 17211038), ('01C_2019.tif', 17217535), ('01C_2020.tif', 17214890), ('01C_2021.tif', 17258711)]
2017 717 files 66.1 GB
2018 719 files 65.3 GB
2019 737 files 66.2 GB
2020 728 files 65.9 GB
2021 728 files 66.0 GB
2022 758 files 67.2 GB
2023 757 files 52.8 GB
2024 757 files 53.3 GB
non-matching keys: ['io-10m-annual-lulc.ndjson']
Asia (UTM 35-55, bands L-T) 2017 141 files 18.2 GB
Asia (UTM 35-55, bands L-T) 2018 142 files 18.2 GB
Asia (UTM 35-55, bands L-T) 2019 143 files 18.4 GB
Asia (UTM 35-55, bands L-T) 2020 142 files 18.3 GB
Asia (UTM 35-55, bands L-T) 2021 142 files 18.1 GB
Asia (UTM 35-55, bands L-T) 2022 146 files 18.4 GB
Asia (UTM 35-55, bands L-T) 2023 146 files 15.1 GB
Asia (UTM 35-55, bands L-T) 2024 146 files 15.3 GB
               key       size  year zone
4704  50R_2017.tif  258915978  2017  50R
4705  50R_2018.tif  262103564  2018  50R
4706  50R_2019.tif  259741327  2019  50R
4707  50R_2020.tif  262570538  

In [ ]:
# STEP 3b - is there a model-version jump between IO 2022 and 2023? (read 320 m overviews)
import numpy as np, rasterio
from rasterio.enums import Resampling
B = "https://io-10m-annual-lulc.s3.us-west-2.amazonaws.com/"
ZONES = ["50R", "49Q", "52S", "43Q", "45Q", "48P", "47N", "49M", "39R", "36S"]   # Shanghai, Pearl R., Korea, Mumbai, Bengal, Bangkok, Malaysia, Java, Gulf, Turkey
def read32(z, y):
    with rasterio.open(f"/vsicurl/{B}{z}_{y}.tif") as s:
        return s.read(1, out_shape=(s.height // 32, s.width // 32), resampling=Resampling.nearest)
rows = []
for z in ZONES:
    a = {y: read32(z, y) for y in (2021, 2022, 2023, 2024)}
    valid = (a[2021] > 0) & (a[2024] > 0)
    r = {"zone": z}
    for y0, y1 in [(2021, 2022), (2022, 2023), (2023, 2024)]:
        r[f"{y0}->{y1} %changed"] = round(float(((a[y0] != a[y1]) & valid).sum() / valid.sum() * 100), 2)
    rows.append(r)
import pandas as pd
d = pd.DataFrame(rows); print(d.to_string(index=False)); print("\nmedian:", d.drop(columns="zone").median().round(2).to_dict())

zone  2021->2022 %changed  2022->2023 %changed  2023->2024 %changed
 50R                 6.44                 6.51                 5.87
 49Q                 7.42                 7.79                 8.00
 52S                 7.19                 6.85                 8.77
 43Q                 5.49                 5.45                 5.25
 45Q                 9.63                 9.82                 9.60
 48P                11.01                10.53                10.56
 47N                 6.00                 6.84                 6.39
 49M                 6.44                 8.37                 6.88
 39R                 1.94                 2.68                 1.98
 36S                 8.23                 8.41                 7.53

median: {'2021->2022 %changed': 6.82, '2022->2023 %changed': 7.32, '2023->2024 %changed': 7.2}


In [ ]:
# STEP 3 - 2024 land cover: change-only update of GLC_FCS30D 2022 with Esri/IO 10 m LULC 2022->2024
import os
assert os.path.ismount("/content/drive"), "Google Drive is not mounted - run STEP 0 first"
os.environ["REGIONS"] = "EastAsia,SouthAsia,SoutheastAsia,WesternAsia"   # test region first; set to all four for the full run
"""STEP 3 (Colab) — 2024 land cover by change-only update of the filtered
GLC_FCS30D 2022 baseline with the Esri / Impact Observatory 10 m annual LULC
(Karra et al. 2021; public bucket io-10m-annual-lulc, CC BY 4.0).

For every UTM zone touching a region's study mask:
  * read IO 2022 and 2024 at the 40 m overview (factor 4) inside the zone's
    study-mask footprint; cross-walk IO classes to our 7 classes;
  * class fractions on the region's 300 m EASE grid (Resampling.average);
  * delta = frac_2024 - frac_2022 (percentage points), kept only where
    |delta| >= DELTA_MIN and both years have >= 50 % valid (cloud-free) cover.
2024 fractions = clip(GLC2022_filtered + delta) renormalised to 100 %;
2024 majority map = argmax. Nothing changes where IO detects no robust change,
so product differences between GLC_FCS30D and IO never enter the series.

Outputs (Drive: AsiaCoastGI/step3): frac_<Region>_2024.tif, lc_<Region>_2024.tif,
areas_2024.csv, update_log.csv (km2 of cells updated, by region).
"""
import os, gc, time, json
import numpy as np, pandas as pd
import rasterio
from rasterio.windows import from_bounds, Window
from rasterio.warp import reproject, Resampling, transform_bounds
from pyproj import Transformer

BASE = "/content/drive/MyDrive/AsiaCoastGI/step2_filtered"
MASKDIR = "/content/drive/MyDrive/AsiaCoastGI/step2"
OUT = "/content/drive/MyDrive/AsiaCoastGI/step3"
CACHE = f"{OUT}/zone_cache_v2"   # v2 = persistence-confirmed change
for d in (OUT, CACHE):
    os.makedirs(d, exist_ok=True)
IO = "https://io-10m-annual-lulc.s3.us-west-2.amazonaws.com/"
Y0, Y1 = 2022, 2024
DELTA_MIN = 10                     # percentage points
NO_UPDATE = [2, 6]   # ShrubGrass, Bare (0-based)
REGIONS = os.environ.get("REGIONS", "EastAsia,SouthAsia,SoutheastAsia,WesternAsia").split(",")
CLS = ["Cropland", "Forest", "ShrubGrass", "Wetland", "Water", "Impervious", "Bare"]
K = 7
# IO 9-class -> our classes (0 = no data / clouds)
IO2US = np.zeros(256, np.uint8)
for io, us in {1: 5, 2: 2, 4: 4, 5: 1, 7: 6, 8: 7, 9: 7, 11: 3}.items():
    IO2US[io] = us
BANDS = "CDEFGHJKLMNPQRSTUVWX"
os.environ.setdefault("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR")
os.environ.setdefault("CPL_VSIL_CURL_ALLOWED_EXTENSIONS", ".tif")


def log(*a): print(time.strftime("%H:%M:%S"), *a, flush=True)


def zones_for(study, transform):
    """UTM zone+band ids whose 6x8 deg cells contain study-mask cells."""
    rr, cc = np.nonzero(study[::10, ::10])
    xs = transform.c + (cc * 10 + 0.5) * transform.a
    ys = transform.f + (rr * 10 + 0.5) * transform.e
    lon, lat = Transformer.from_crs("EPSG:6933", "EPSG:4326", always_xy=True).transform(xs, ys)
    z = np.floor((np.asarray(lon) + 180) / 6).astype(int) + 1
    b = np.floor((np.asarray(lat) + 80) / 8).astype(int)
    return sorted({f"{zz:02d}{BANDS[bb]}" for zz, bb in zip(z, b) if 0 <= bb < len(BANDS)})


def zone_fractions(zid, year, dst_shape, dst_transform, dst_crs, bounds_ll, strip=400):
    """IO class fractions (K+1 incl. valid count) on a destination window.
    Done in strips of destination rows so a full 6x8 deg zone never sits in RAM at once."""
    url = f"/vsicurl/{IO}{zid}_{year}.tif"
    out = np.full((K + 1,) + dst_shape, np.nan, np.float32)
    hit = False
    with rasterio.open(url, OVERVIEW_LEVEL=1) as s:          # factor-4 overview = 40 m
        zb = transform_bounds("EPSG:4326", s.crs, *bounds_ll, densify_pts=21)
        for r0 in range(0, dst_shape[0], strip):
            h = min(strip, dst_shape[0] - r0)
            tt = dst_transform * rasterio.Affine.translation(0, r0)
            x0, y1 = tt.c, tt.f
            x1, y0 = x0 + dst_shape[1] * tt.a, y1 + h * tt.e
            b = transform_bounds(dst_crs, s.crs, x0, y0, x1, y1, densify_pts=21)
            b = (max(b[0], zb[0]), max(b[1], zb[1]), min(b[2], zb[2]), min(b[3], zb[3]))
            if b[2] <= b[0] or b[3] <= b[1]:
                continue
            w = from_bounds(*b, s.transform).round_offsets().round_lengths()
            w = Window(w.col_off - 2, w.row_off - 2, w.width + 4, w.height + 4)
            w = w.intersection(Window(0, 0, s.width, s.height))
            if w.width <= 0 or w.height <= 0:
                continue
            cls = IO2US[s.read(1, window=w)]
            st = s.window_transform(w)
            for k in range(K + 1):
                ind = ((cls > 0) if k == 0 else (cls == k)).astype(np.float32)
                reproject(ind, out[k, r0:r0 + h], src_transform=st, src_crs=s.crs, dst_transform=tt,
                          dst_crs=dst_crs, resampling=Resampling.average, dst_nodata=np.nan, init_dest_nodata=True)
                del ind
            del cls; hit = True
    return out if hit else None


if __name__ == "__main__":
    to_ll = Transformer.from_crs("EPSG:6933", "EPSG:4326", always_xy=True)
    rows, ulog = [], []
    for name in REGIONS:
        with rasterio.open(f"{MASKDIR}/studymask_{name}.tif") as s:
            study = s.read(1) > 0
            T, crs, prof = s.transform, s.crs, s.profile
        H, W = study.shape
        delta = np.zeros((K, H, W), np.int8)
        filled = np.zeros((H, W), bool)
        zones = zones_for(study, T)
        log(name, "IO zones:", len(zones), zones)
        for zi, zid in enumerate(zones, 1):
            cf = f"{CACHE}/{name}_{zid}.npz"
            # destination window = study-mask bbox inside this zone's lon/lat cell
            zn, bd = int(zid[:2]), BANDS.index(zid[2])
            lon0, lon1 = (zn - 1) * 6 - 180, zn * 6 - 180
            lat0, lat1 = bd * 8 - 80, bd * 8 - 72
            xs, ys = Transformer.from_crs("EPSG:4326", "EPSG:6933", always_xy=True).transform([lon0, lon1], [lat0, lat1])
            dw = from_bounds(min(xs), min(ys), max(xs), max(ys), T).round_offsets().round_lengths()
            dw = dw.intersection(Window(0, 0, W, H))
            sub = study[dw.row_off:dw.row_off + dw.height, dw.col_off:dw.col_off + dw.width]
            if not sub.any():
                continue
            rows_, cols_ = np.flatnonzero(sub.any(1)), np.flatnonzero(sub.any(0))
            dw = Window(dw.col_off + cols_[0], dw.row_off + rows_[0], cols_[-1] - cols_[0] + 1, rows_[-1] - rows_[0] + 1)
            dt = rasterio.windows.transform(dw, T)
            x0, y1_ = dt.c, dt.f
            x1, y0_ = x0 + dw.width * dt.a, y1_ + dw.height * dt.e
            (la, lb), (pa, pb) = to_ll.transform([x0, x1], [y0_, y1_])
            bounds_ll = (max(min(la, lb), lon0), max(min(pa, pb), lat0), min(max(la, lb), lon1), min(max(pa, pb), lat1))
            try:
                if os.path.exists(cf):
                    d = np.load(cf)["d"]
                else:
                    # persistence-confirmed change: IO 2021, 2022 (before) and 2023, 2024 (after)
                    fy = {y: zone_fractions(zid, y, (dw.height, dw.width), dt, crs, bounds_ll) for y in (2021, 2022, 2023, 2024)}
                    if any(f is None for f in fy.values()):
                        log(f"  [{zi}/{len(zones)}] {zid}: no overlap"); continue
                    ok = np.all([np.nan_to_num(f[0]) >= 0.5 for f in fy.values()], axis=0)
                    p = {y: np.nan_to_num(f[1:]) / np.maximum(np.nan_to_num(f[0]), 1e-6) * 100 for y, f in fy.items()}
                    d23, d24, pre = p[2023] - p[2022], p[2024] - p[2022], p[2022] - p[2021]
                    confirmed = (np.sign(d23) == np.sign(d24)) & (np.abs(d23) >= DELTA_MIN) & \
                                (np.abs(d24) >= DELTA_MIN) & (np.abs(pre) < DELTA_MIN) & ok[None]
                    dd = np.where(confirmed, p[2024] - (p[2021] + p[2022]) / 2, 0)
                    dd[np.abs(dd) < DELTA_MIN] = 0
                    del fy, p
                    d = np.clip(np.round(dd), -100, 100).astype(np.int8)
                    d = np.concatenate([d, ok[None].astype(np.int8)])
                    np.savez_compressed(cf, d=d, win=[dw.col_off, dw.row_off, dw.width, dw.height])
                # shrub/grass <-> bare deltas = definitional disagreement between products in arid land
                d = d.copy(); d[NO_UPDATE] = 0
                r0, c0 = dw.row_off, dw.col_off
                okz = d[K].astype(bool) & ~filled[r0:r0 + dw.height, c0:c0 + dw.width]
                tgt = delta[:, r0:r0 + dw.height, c0:c0 + dw.width]
                tgt[:, okz] = d[:K][:, okz]
                filled[r0:r0 + dw.height, c0:c0 + dw.width] |= okz
                log(f"  [{zi}/{len(zones)}] {zid}: ok")
            except Exception as e:
                log(f"  [{zi}/{len(zones)}] {zid}: ERROR {e!r}")
            gc.collect()
        # apply the delta to the filtered 2022 baseline (row blocks to save RAM)
        f0 = rasterio.open(f"{BASE}/frac_{name}_{Y0}.tif"); l0 = rasterio.open(f"{BASE}/lc_{name}_{Y0}.tif")
        fo = rasterio.open(f"{OUT}/frac_{name}_{Y1}.tif", "w", **f0.profile)
        lo = rasterio.open(f"{OUT}/lc_{name}_{Y1}.tif", "w", **l0.profile)
        for i, n in enumerate(CLS, 1):
            fo.set_band_description(i, f"{n}_pct")
        a22, a24 = np.zeros(K), np.zeros(K)
        upd_km2 = chg_km2 = 0.0
        for r0 in range(0, H, 1024):
            win = Window(0, r0, W, min(1024, H - r0))
            sl = slice(r0, r0 + win.height)
            base = f0.read(window=win).astype(np.int16)
            lc22 = l0.read(1, window=win)
            st = study[sl]
            inside = st & (base[0] != 255)
            dl = delta[:, sl]
            new = np.clip(base + dl, 0, 100).astype(np.float32)
            tot = new.sum(0)
            new = np.where(tot > 0, new / np.maximum(tot, 1e-6) * 100, 0)
            frac24 = np.where(inside[None], np.round(new), 255).astype(np.uint8)
            lc24 = np.where(inside, frac24.argmax(0) + 1, 0).astype(np.uint8)
            keep = st & ~inside
            lc24[keep] = lc22[keep]
            fo.write(frac24, window=win)
            lo.write(lc24, 1, window=win)
            if inside.any():
                a22 += base[:, inside].sum(1) / 100 * 0.09
                a24 += frac24[:, inside].astype(np.float64).sum(1) / 100 * 0.09
            upd_km2 += float(((np.abs(dl).sum(0) > 0) & st).sum() * 0.09)
            chg_km2 += float(((lc24 != lc22) & st).sum() * 0.09)
        for f in (f0, l0, fo, lo):
            f.close()
        ulog.append(dict(region=name, zones=len(zones), io_coverage_pct=float(filled[study].mean() * 100),
                         cells_with_delta_km2=upd_km2, majority_changed_km2=chg_km2,
                         majority_changed_pct=chg_km2 / (study.sum() * 0.09) * 100))
        for k in range(K):
            rows.append(dict(region=name, cls=CLS[k], km2_2022=a22[k], km2_2024=a24[k],
                             pct_2022=a22[k] / a22.sum() * 100, pct_2024=a24[k] / a24.sum() * 100))
        log(name, ulog[-1])
        del delta, filled, study
        gc.collect()
    tag = "" if len(REGIONS) == 4 else "_" + "_".join(REGIONS)
    pd.DataFrame(rows).to_csv(f"{OUT}/areas_2024{tag}.csv", index=False)
    pd.DataFrame(ulog).to_csv(f"{OUT}/update_log{tag}.csv", index=False)
    pd.set_option("display.width", 250)
    print(pd.DataFrame(rows).round(2).to_string(index=False))
    print(pd.DataFrame(ulog).round(2).to_string(index=False))
    log("STEP 3 DONE ->", OUT)


07:04:01 EastAsia IO zones: 18 ['48Q', '49Q', '50Q', '50R', '50S', '50T', '51Q', '51R', '51S', '51T', '52R', '52S', '52T', '53S', '54R', '54S', '54T', '55T']
07:04:01   [1/18] 48Q: ok
07:04:01   [2/18] 49Q: ok
07:04:01   [3/18] 50Q: ok
07:04:02   [4/18] 50R: ok
07:04:02   [5/18] 50S: ok
07:04:02   [6/18] 50T: ok
07:04:02   [7/18] 51Q: ok
07:04:03   [8/18] 51R: ok
07:04:03   [9/18] 51S: ok
07:04:03   [10/18] 51T: ok
07:04:03   [11/18] 52R: ok
07:04:04   [12/18] 52S: ok
07:04:04   [13/18] 52T: ok
07:04:04   [14/18] 53S: ok
07:04:04   [15/18] 54R: ok
07:04:05   [16/18] 54S: ok
07:04:05   [17/18] 54T: ok
07:04:05   [18/18] 55T: ok
07:04:34 EastAsia {'region': 'EastAsia', 'zones': 18, 'io_coverage_pct': 99.11937357937988, 'cells_with_delta_km2': 36212.67, 'majority_changed_km2': 4374.8099999999995, 'majority_changed_pct': np.float64(0.5164674261298794)}
07:04:35 SouthAsia IO zones: 16 ['39R', '40R', '41R', '42P', '42Q', '42R', '43N', '43P', '43Q', '44N', '44P', '44Q', '45Q', '46N', '46P', '

In [ ]:
# STEP 4 - driving factors for FLUS / PLUS on each region's 300 m EASE grid
import os
assert os.path.ismount("/content/drive"), "Google Drive is not mounted - run STEP 0 first"
os.environ["REGIONS"] = "SouthAsia,SoutheastAsia,WesternAsia,EastAsia"   # finished layers are skipped
"""STEP 4 (Colab) - spatial driving factors for the land-use simulations.

Every layer is written on the region's 300 m EASE-Grid 2.0 (EPSG:6933) grid,
identical to the land-cover maps, and masked to the study area (nodata -9999).

  dem, slope        Copernicus GLO-90 DEM (public COGs). Slope is computed at 90 m, then averaged to 300 m.
  dist_coast        distance (km) to the sea (from the Step 1 shoreline masks).
  dist_river        distance (km) to rivers with an upstream area of at least 500 km2 (HydroRIVERS v1.0).
  dist_road         distance (km) to any GRIP4 road (Meijer et al. 2018).
  dist_urban_YYYY   distance (km) to impervious-majority cells, 2000/2010/2020/2024 (own maps).
  pop_YYYY          GHS-POP R2023A population density (persons/km2), 2000/2010/2020/2025.
  bio1, bio12       CHELSA v2.1 mean annual temperature (degC) and precipitation (mm), 1981-2010.
  protected         WDPA protected areas (1/0). Designated/inscribed/established only;
                    marine-only sites are dropped. Used as a restricted zone.

Outputs (Drive: AsiaCoastGI/step4): drv_<Region>_<layer>.tif, driver_summary.csv
Sources that cannot be reached are logged and skipped, so a rerun picks them up.
"""
import gc, glob, time, zipfile, subprocess, calendar, datetime
import numpy as np, pandas as pd, cv2
import rasterio, rasterio.features
from rasterio.windows import from_bounds, Window
from rasterio.warp import reproject, Resampling, transform_bounds
from pyproj import Transformer
from concurrent.futures import ThreadPoolExecutor

ROOT = "/content/drive/MyDrive/AsiaCoastGI"
import glob as _g
for _p in _g.glob(f"{ROOT}/step4/drv_*_bio*.tif"):   # one-off: rebuild climate layers with the 65535 no-data masked
    if os.path.getmtime(_p) < 1794994688: os.remove(_p)
MASKDIR, S1 = f"{ROOT}/step2", f"{ROOT}/step1"
LCDIR, LC24 = f"{ROOT}/step2_filtered", f"{ROOT}/step3"
OUT = f"{ROOT}/step4"; TMP = "/content/tmp_drivers"
for d in (OUT, TMP):
    os.makedirs(d, exist_ok=True)
REGIONS = os.environ.get("REGIONS", "EastAsia,SouthAsia,SoutheastAsia,WesternAsia").split(",")
LAYERS = os.environ.get("LAYERS", "all")
RES = 300.0; ND = -9999.0
os.environ.setdefault("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR")
os.environ.setdefault("CPL_VSIL_CURL_ALLOWED_EXTENSIONS", ".tif,.zip")
os.environ.setdefault("GDAL_HTTP_MAX_RETRY", "4"); os.environ.setdefault("GDAL_HTTP_RETRY_DELAY", "2")


def log(*a): print(time.strftime("%H:%M:%S"), *a, flush=True)
def want(name): return LAYERS == "all" or name.split("_")[0] in LAYERS.split(",")


def fetch(urls, dest):
    """Download the first URL that works (wget, resumable). Returns path or None."""
    if os.path.exists(dest) and os.path.getsize(dest) > 1000:
        return dest
    for u in ([urls] if isinstance(urls, str) else urls):
        r = subprocess.run(["wget", "-q", "-c", "--tries=3", "-O", dest + ".part", u])
        if r.returncode == 0 and os.path.getsize(dest + ".part") > 1000:
            os.rename(dest + ".part", dest); log("  downloaded", os.path.basename(u)); return dest
        if os.path.exists(dest + ".part"):
            os.remove(dest + ".part")
    log("  !! could not download", dest); return None


def unzip(path, sub):
    d = f"{TMP}/{sub}"
    if not os.path.isdir(d):
        with zipfile.ZipFile(path) as z:
            z.extractall(d)
    # nested zips (WDPA ships 3 inner zips)
    for inner in glob.glob(f"{d}/**/*.zip", recursive=True):
        t = inner[:-4]
        if not os.path.isdir(t):
            with zipfile.ZipFile(inner) as z:
                z.extractall(t)
    return d


class Grid:
    def __init__(self, name):
        with rasterio.open(f"{MASKDIR}/studymask_{name}.tif") as s:
            self.study = s.read(1) > 0
            self.T, self.crs, prof = s.transform, s.crs, s.profile
        self.H, self.W = self.study.shape
        self.prof = dict(driver="GTiff", height=self.H, width=self.W, count=1, dtype="float32",
                         crs=self.crs, transform=self.T, nodata=ND, compress="deflate",
                         predictor=3, tiled=True, blockxsize=512, blockysize=512, BIGTIFF="IF_SAFER")
        b = rasterio.transform.array_bounds(self.H, self.W, self.T)
        self.bounds = tuple(b)  # array_bounds -> (west, south, east, north)
        self.ll = transform_bounds(self.crs, "EPSG:4326", *self.bounds, densify_pts=101)
        self.name = name

    def path(self, layer): return f"{OUT}/drv_{self.name}_{layer}.tif"

    def save(self, layer, arr):
        a = np.where(self.study & np.isfinite(arr), arr, ND).astype(np.float32)
        with rasterio.open(self.path(layer), "w", **self.prof) as d:
            d.write(a, 1)
        v = a[self.study]; v = v[v != ND]
        log(f"  {self.name} {layer}: cover {100*v.size/self.study.sum():.1f}%  "
            f"mean {v.mean() if v.size else np.nan:.2f}  min {v.min() if v.size else np.nan:.2f}  max {v.max() if v.size else np.nan:.2f}")

    def warp(self, src_path, resampling=Resampling.average, band=1, scale=None):
        """Read the region window of a lon/lat raster and warp it to the grid."""
        out = np.full((self.H, self.W), np.nan, np.float32)
        with rasterio.open(src_path) as s:
            w = from_bounds(*transform_bounds("EPSG:4326", s.crs, *self.ll), s.transform)
            w = w.round_offsets().round_lengths().intersection(Window(0, 0, s.width, s.height))
            a = s.read(band, window=w, masked=True).astype(np.float32).filled(np.nan)
            if scale is not None:
                a = scale(a, rasterio.windows.transform(w, s.transform))
            reproject(a, out, src_transform=rasterio.windows.transform(w, s.transform), src_crs=s.crs,
                      src_nodata=np.nan, dst_transform=self.T, dst_crs=self.crs, dst_nodata=np.nan,
                      resampling=resampling, num_threads=4)
        return out

    def dist_km(self, feature):
        """Euclidean distance (km) from every cell to the nearest True cell."""
        if not feature.any():
            return np.full((self.H, self.W), np.nan, np.float32)
        return cv2.distanceTransform((~feature).astype(np.uint8), cv2.DIST_L2, cv2.DIST_MASK_PRECISE) * (RES / 1000)

    def rasterize_vectors(self, files, cols=None, keep=None, chunk=400_000, all_touched=True):
        import pyogrio
        feat = np.zeros((self.H, self.W), np.uint8)
        for f in files:
            info = pyogrio.read_info(f); n = info["features"]
            for k in range(0, n, chunk):
                g = pyogrio.read_dataframe(f, bbox=self.ll, columns=cols or [],
                                           skip_features=k, max_features=chunk)
                if keep is not None and len(g):
                    g = g[keep(g)]
                if len(g):
                    g = g.to_crs(self.crs)
                    rasterio.features.rasterize(((geom, 1) for geom in g.geometry if geom is not None),
                                                out=feat, transform=self.T, all_touched=all_touched, merge_alg=rasterio.enums.MergeAlg.replace)
                del g; gc.collect()
        return feat.astype(bool)


# ---------------------------------------------------------------- DEM + slope
CDEM = "https://copernicus-dem-90m.s3.amazonaws.com"


def dem_tiles(g):
    rr, cc = np.nonzero(g.study[::8, ::8])
    xs = g.T.c + (cc * 8 + 0.5) * g.T.a; ys = g.T.f + (rr * 8 + 0.5) * g.T.e
    lon, lat = Transformer.from_crs(g.crs, "EPSG:4326", always_xy=True).transform(xs, ys)
    return sorted({(int(np.floor(a)), int(np.floor(o))) for a, o in zip(lat, lon)})


def dem_tile(lat, lon):
    ns, ew = ("N" if lat >= 0 else "S"), ("E" if lon >= 0 else "W")
    t = f"Copernicus_DSM_COG_30_{ns}{abs(lat):02d}_00_{ew}{abs(lon):03d}_00_DEM"
    try:
        with rasterio.open(f"/vsicurl/{CDEM}/{t}/{t}.tif") as s:
            z = s.read(1).astype(np.float32); tr = s.transform
    except Exception:
        return None  # all-ocean tiles do not exist
    dy = abs(tr.e) * 111320.0
    dx = tr.a * 111320.0 * np.cos(np.radians(lat + 0.5))
    gy, gx = np.gradient(z, dy, dx)
    slope = np.degrees(np.arctan(np.hypot(gx, gy))).astype(np.float32)
    return z, slope, tr


def do_dem(g):
    tiles = dem_tiles(g); log(f"  {g.name}: {len(tiles)} DEM tiles")
    dem = np.full((g.H, g.W), np.nan, np.float32); slp = dem.copy()
    with ThreadPoolExecutor(8) as ex:
        for i, r in enumerate(ex.map(lambda t: dem_tile(*t), tiles), 1):
            if r is None:
                continue
            z, s, tr = r
            b = transform_bounds("EPSG:4326", g.crs, tr.c, tr.f + tr.e * z.shape[0], tr.c + tr.a * z.shape[1], tr.f)
            w0 = from_bounds(*b, g.T)
            c0, r0 = int(np.floor(w0.col_off)), int(np.floor(w0.row_off))
            c1, r1 = int(np.ceil(w0.col_off + w0.width)), int(np.ceil(w0.row_off + w0.height))
            c0, r0, c1, r1 = max(c0, 0), max(r0, 0), min(c1, g.W), min(r1, g.H)
            if c1 <= c0 or r1 <= r0:
                continue
            w = Window(c0, r0, c1 - c0, r1 - r0)
            wt = rasterio.windows.transform(w, g.T)
            for src, dst in ((z, dem), (s, slp)):
                o = np.full((w.height, w.width), np.nan, np.float32)
                reproject(src, o, src_transform=tr, src_crs="EPSG:4326", dst_transform=wt, dst_crs=g.crs,
                          dst_nodata=np.nan, resampling=Resampling.average)
                sub = dst[w.row_off:w.row_off + w.height, w.col_off:w.col_off + w.width]
                np.copyto(sub, o, where=np.isnan(sub) & np.isfinite(o))
            if i % 100 == 0:
                log(f"    {i}/{len(tiles)} tiles")
    # cells over open sea (no tile) -> elevation 0, slope 0
    for a in (dem, slp):
        a[np.isnan(a)] = 0
    g.save("dem", dem); g.save("slope", slp)


# ---------------------------------------------------------------- distances
def do_coast(g):
    """Distance (km) from land to the GSHHG L1 shoreline (same definition as the Step 1 zone)."""
    shp = "/content/work/gshhg/GSHHS_shp/f/GSHHS_f_L1.shp"
    if not os.path.exists(shp):
        z = fetch(["https://www.soest.hawaii.edu/pwessel/gshhg/gshhg-shp-2.3.7.zip"], f"{TMP}/gshhg.zip")
        with zipfile.ZipFile(z) as zz:
            zz.extractall("/content/work/gshhg", [m for m in zz.namelist() if "GSHHS_shp/f/GSHHS_f_L1" in m])
    land = g.rasterize_vectors([shp], all_touched=False)
    d = g.dist_km(~land); d[~land] = 0
    g.save("dist_coast", d)


def do_urban(g):
    for y in (2000, 2010, 2020, 2024):
        p = f"{LC24}/lc_{g.name}_2024.tif" if y == 2024 else f"{LCDIR}/lc_{g.name}_{y}.tif"
        if not os.path.exists(p):
            log("  missing", p); continue
        with rasterio.open(p) as s:
            imp = s.read(1) == 6
        g.save(f"dist_urban_{y}", g.dist_km(imp))


HYDRO = "https://data.hydrosheds.org/file/HydroRIVERS/HydroRIVERS_v10_{}_shp.zip"


def do_river(g):
    files = []
    for reg in ("as", "eu", "au"):
        z = fetch(HYDRO.format(reg), f"{TMP}/HydroRIVERS_{reg}.zip")
        if z:
            files += glob.glob(f"{unzip(z, 'hr_' + reg)}/**/*.shp", recursive=True)
    if not files:
        return
    g.save("dist_river", g.dist_km(g.rasterize_vectors(files, cols=["UPLAND_SKM"], keep=lambda d: d.UPLAND_SKM >= 500)))


GRIP = ["https://dataportaal.pbl.nl/downloads/GRIP4/GRIP4_Region{}_vector_shp.zip",
        "https://zenodo.org/records/6420961/files/GRIP4_Region{}_vector_shp.zip?download=1"]


def do_road(g):
    files = []
    for reg in (4, 5, 6, 7):   # Europe (Turkey), Middle East, S & E Asia, Oceania (Timor, PNG side)
        z = fetch([u.format(reg) for u in GRIP], f"{TMP}/GRIP4_R{reg}.zip")
        if z:
            files += glob.glob(f"{unzip(z, f'grip_{reg}')}/**/*.shp", recursive=True)
    if not files:
        return
    g.save("dist_road", g.dist_km(g.rasterize_vectors(files)))


# ---------------------------------------------------------------- population, climate
GHS = ("https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/GHSL/GHS_POP_GLOBE_R2023A/"
       "GHS_POP_E{y}_GLOBE_R2023A_4326_30ss/V1-0/GHS_POP_E{y}_GLOBE_R2023A_4326_30ss_V1_0.zip")


def per_km2(a, tr):
    lat = tr.f + (np.arange(a.shape[0]) + 0.5) * tr.e
    area = (abs(tr.e) * 111.32) * (tr.a * 111.32 * np.cos(np.radians(lat)))
    return a / area[:, None]


def do_pop(g):
    for y in (2000, 2010, 2020, 2025):
        z = fetch(GHS.format(y=y), f"{TMP}/ghs_pop_{y}.zip")
        if not z:
            continue
        tif = [n for n in zipfile.ZipFile(z).namelist() if n.endswith(".tif")][0]
        a = g.warp(f"/vsizip/{z}/{tif}", scale=per_km2)
        a[np.isnan(a)] = 0
        g.save(f"pop_{y}", a)


def fill_nearest(a):
    """Fill NaN cells with the nearest valid value (CHELSA is land-only; coastal cells need values)."""
    m = np.isnan(a)
    if not m.any() or m.all():
        return a
    _, lab = cv2.distanceTransformWithLabels(m.astype(np.uint8), cv2.DIST_L2, 5, labelType=cv2.DIST_LABEL_PIXEL)
    vals = np.zeros(lab.max() + 1, np.float32)
    vals[lab[~m]] = a[~m]
    return vals[lab]


CHELSA = "https://os.zhdk.cloud.switch.ch/chelsav2/GLOBAL/climatologies/1981-2010/bio/CHELSA_{}_1981-2010_V.2.1.tif"


def do_clim(g):
    for v in ("bio1", "bio12"):
        p = fetch(CHELSA.format(v), f"{TMP}/CHELSA_{v}.tif")
        if not p:
            continue
        with rasterio.open(p) as s:
            sc, off = (s.scales[0] or 1), (s.offsets[0] or 0)
        a = g.warp(p, scale=lambda x, tr: fill_nearest(np.where(x >= 65535, np.nan, x) * sc + off))   # 65535 = CHELSA no-data   # gaps filled on the 1 km source grid
        if v == "bio1" and np.nanmedian(a) > 150:      # raw Kelvin*10 without scale tags
            a = a * 0.1 - 273.15
        g.save(v, a)


# ---------------------------------------------------------------- protected areas
def wdpa_zip():
    t = datetime.date.today(); names = []
    for k in range(0, 4):
        mth = (t.month - 1 - k) % 12 + 1; yr = t.year + ((t.month - 1 - k) // 12)
        tag = f"{calendar.month_abbr[mth]}{yr}"
        names += [f"WDPA_{tag}_Public_shp.zip", f"WDPA_WDOECM_{tag}_Public_shp.zip"]
    urls = [f"https://d1gam3xoknrgr2.cloudfront.net/current/{n}" for n in names]
    return fetch(urls, f"{TMP}/wdpa_shp.zip")


def do_pa(g):
    z = wdpa_zip()
    if not z:
        return
    d = unzip(z, "wdpa")
    files = [f for f in glob.glob(f"{d}/**/*polygons.shp", recursive=True)]
    import pyogrio
    realm = "REALM" if "REALM" in list(pyogrio.read_info(files[0])["fields"]) else "MARINE"   # schema changed in 2025/26
    keep = lambda d: d.STATUS.isin(["Designated", "Inscribed", "Established"]) & \
                     ~d[realm].astype(str).isin(["Marine", "2"])
    g.save("protected", g.rasterize_vectors(files, cols=["STATUS", realm], keep=keep).astype(np.float32))


STEPS = [("dem", do_dem), ("dist_coast", do_coast), ("dist_urban", do_urban), ("pop", do_pop),
         ("bio", do_clim), ("dist_river", do_river), ("dist_road", do_road), ("protected", do_pa)]

if __name__ == "__main__":
    for name in REGIONS:
        g = Grid(name); log(name, f"grid {g.H}x{g.W}", "lon/lat", np.round(g.ll, 2))
        for key, fn in STEPS:
            if not want(key):
                continue
            first = {"dem": "dem", "bio": "bio12", "pop": "pop_2025", "dist_urban": "dist_urban_2024"}.get(key, key)
            if os.path.exists(g.path(first)):
                log(f"  {name} {key}: done already"); continue
            t0 = time.time()
            try:
                fn(g)
            except Exception as e:
                log(f"  !! {name} {key}: {e!r}")
            log(f"  {name} {key}: {time.time() - t0:.0f}s"); gc.collect()
        del g; gc.collect()
    # summary over study-area cells
    rows = []
    for p in sorted(glob.glob(f"{OUT}/drv_*.tif")):
        _, reg, layer = os.path.basename(p)[:-4].split("_", 2)
        with rasterio.open(p) as s:
            a = s.read(1); v = a[a != ND]
        if v.size == 0:
            rows.append(dict(region=reg, layer=layer, n=0)); continue
        rows.append(dict(region=reg, layer=layer, n=v.size, mean=v.mean(), p5=np.percentile(v, 5),
                         p50=np.median(v), p95=np.percentile(v, 95)))
        del a, v
    df = pd.DataFrame(rows); df.to_csv(f"{OUT}/driver_summary.csv", index=False)
    print(df.round(2).to_string(index=False)); log("STEP 4 DONE ->", OUT)


09:37:11 SouthAsia grid 16117x17690 lon/lat [43. -1. 98. 40.]
09:37:11   SouthAsia dem: done already
09:37:11   SouthAsia dist_coast: done already
09:37:11   SouthAsia dist_urban: done already
09:37:11   SouthAsia pop: done already
09:37:48   SouthAsia bio1: cover 100.0%  mean 26.62  min 14.25  max 29.55
09:38:16   SouthAsia bio12: cover 100.0%  mean 1495.44  min 98.10  max 6552.80
09:38:16   SouthAsia bio: 65s
09:38:16   SouthAsia dist_river: done already
09:38:16   SouthAsia dist_road: done already
09:38:16   SouthAsia protected: done already
09:38:16 SoutheastAsia grid 16892x16404 lon/lat [ 91. -12. 142.  29.]
09:38:16   SoutheastAsia dem: done already
09:38:16   SoutheastAsia dist_coast: done already
09:38:16   SoutheastAsia dist_urban: done already
09:38:16   SoutheastAsia pop: done already
09:38:45   SoutheastAsia bio1: cover 100.0%  mean 25.47  min 2.55  max 28.65
09:39:12   SoutheastAsia bio12: cover 100.0%  mean 2919.36  min 798.90  max 6553.40
09:39:12   SoutheastAsia bio: 56

In [ ]:
# STEP 5 - FLUS and PLUS: 2010->2020 hindcast validation and 2040 projection
import os
assert os.path.ismount("/content/drive"), "Google Drive is not mounted - run STEP 0 first"
os.environ["REGIONS"] = "SoutheastAsia"; os.environ["MODELS"] = "PLUS"   # only SEA PLUS 2040 left after disconnect
os.environ["RUNS"] = "2040"                      # hindcast already done; rerun projections with net flows
os.environ["SCENARIOS"] = "NDS,RDS,FPS,PCS"
"""STEP 5 (Colab) - land-use simulation with FLUS-type and PLUS-type models on the 300 m study grid,
following the YRD manuscript (Sections 2.3-2.4, Table 2), adapted to the Asian coast.

Classes (0-based): 0 Cropland 1 Forest 2 ShrubGrass 3 Wetland 4 Water 5 Impervious 6 Bare
Drivers (Step 4, min-max normalised): dem, slope, dist_coast, dist_river, dist_road, dist_urban_<t>,
  pop_<t>, bio1, bio12 (distances and population log1p-transformed; dem floored at -10 m).

FLUS-type : probability of occurrence from a multinomial logit (MNL) fitted to the start-year map
            (FLUS_SUIT=ANN switches to the original FLUS neural network); 3x3 neighbourhood;
            self-adaptive inertia; roulette-wheel competition.
PLUS-type : LEAS - random forest (160 trees) per class on cells that expanded to that class in the
            previous period vs non-expanded cells; CARS with multi-type random patch seeds,
            decreasing threshold and a 7x7 neighbourhood.
Both models share demand, drivers, mask and weights (neighbourhood weights from historical net
change), so they differ only in suitability and allocation. Change is allocated as transition-pair
quotas (c->k), so gross change and swaps are reproduced, not only net demand. Fixed random seed.

Runs
  hindcast : 2010 -> 2020 with the observed 2010->2020 transition quotas (isolates allocation skill);
             the Markov 2000->2010 demand error is also reported. FLUS on the 2010 map; PLUS LEAS on
             2000->2010 expansion with 2000 drivers. Accuracy vs observed 2020: OA, Kappa, FoM (+A/B/C/D),
             class FoM at 300 m, 1.5 km and 3 km, ROC-AUC of the suitability surfaces.
  2040     : 2024 -> 2040 (16 yr). Markov matrix from 2010->2022 transitions, scaled to 16 yr
             (fractional matrix power). Scenarios (manuscript Table 2):
             NDS  Markov demand applied directly, no extra restrictions
             RDS  crop/forest/grass -> impervious +20 %, water -> impervious +10 %, impervious -> other -30 %
             FPS  crop -> impervious -60 %, crop -> forest/grass/water -30 %, crop may not become grass or water
             PCS  forest, grass, water and coastal wetland protected; impervious increment = 0.65 x NDS,
                  supplied by cropland
             (adjusted rows renormalised); quotas are the NET directional flows max(Q_ck - Q_kc, 0), so
             reciprocal noise swaps are not allocated. FLUS on the 2024 map; PLUS LEAS on 2010->2022 expansion.

Outputs (Drive: AsiaCoastGI/step5): sim_<Region>_<model>_hindcast.tif, sim_<Region>_<model>_2040_<SCEN>.tif,
  accuracy.csv, areas.csv, demand.csv, leas_importance.csv, ca_log.csv
  (results of the earlier NDS-only test run are kept in step5/v1_nds/)
"""
import gc, time, json
import numpy as np, pandas as pd, rasterio
from scipy.linalg import fractional_matrix_power
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
import cv2

ROOT = "/content/drive/MyDrive/AsiaCoastGI"
LCD, LC24, DRV, MASK = f"{ROOT}/step2_filtered", f"{ROOT}/step3", f"{ROOT}/step4", f"{ROOT}/step2"
OUT = f"{ROOT}/step5"; os.makedirs(OUT, exist_ok=True)
if not os.path.isdir(f"{OUT}/v1_nds"):              # one-off: keep the NDS-only test run apart
    import shutil, glob as _g
    os.makedirs(f"{OUT}/v1_nds")
    for _p in _g.glob(f"{OUT}/*.*"):
        shutil.move(_p, f"{OUT}/v1_nds/")
REGIONS = os.environ.get("REGIONS", "WesternAsia,EastAsia,SouthAsia,SoutheastAsia").split(",")
RUNS = os.environ.get("RUNS", "hindcast,2040").split(",")
MODELS = os.environ.get("MODELS", "FLUS,PLUS").split(",")
CLS = ["Cropland", "Forest", "ShrubGrass", "Wetland", "Water", "Impervious", "Bare"]
K, CROP, IMP = 7, 0, 5
FEAT = ["dem", "slope", "dist_coast", "dist_river", "dist_road", "dist_urban", "pop", "bio1", "bio12"]
POPY = {2000: 2000, 2010: 2010, 2020: 2020, 2024: 2025}
URBY = {2000: 2000, 2010: 2010, 2020: 2020, 2024: 2024}
MAX_IT, ACCEL = 400, 0.1            # iterations, share of the remaining quota allocated per iteration
FLUS_NEIGH, PLUS_NEIGH, RF_TREES = 3, 7, 160      # manuscript Table 2
FLUS_SUIT = os.environ.get("FLUS_SUIT", "MNL")    # MNL (manuscript) or ANN (original FLUS)
PA_RESTRICT = False                 # WDPA not used as a simulation constraint (manuscript: NDS has no extra restrictions)
FOREST, SHRUB, WETLAND, WATER, BARE = 1, 2, 3, 4, 6
SCENARIOS = os.environ.get("SCENARIOS", "NDS,RDS,FPS,PCS").split(",")
PCS_PROTECT = [FOREST, SHRUB, WETLAND, WATER]     # forest, grassland, water (+ coastal wetland) protected
PCS_IMP_SCALE = 0.65
NET_FLOWS = os.environ.get("NET_FLOWS", "1") == "1"   # 2040 quotas = net directional flows (see scenario_quotas)
SEED_PCT, MU, DELTA = 1e-4, 0.5, 0.9  # PLUS: patch-seed share, patch threshold, threshold decay
SEED = 42


def log(*a): print(time.strftime("%H:%M:%S"), *a, flush=True)


def append_csv(df, name):
    p = f"{OUT}/{name}"
    old = pd.read_csv(p) if os.path.exists(p) else None
    if old is not None:
        keys = [c for c in ("region", "model", "run", "scenario") if c in df.columns and c in old.columns]
        if keys:
            m = old.set_index(keys).index.isin(df.set_index(keys).index.unique())
            old = old[~m]
        df = pd.concat([old, df], ignore_index=True)
    df.to_csv(p, index=False)


class Region:
    def __init__(self, name):
        self.name = name
        with rasterio.open(f"{MASK}/studymask_{name}.tif") as s:
            study = s.read(1) > 0
            self.prof = dict(s.profile, dtype="uint8", nodata=0, count=1, compress="deflate",
                             tiled=True, blockxsize=512, blockysize=512)
        self.H, self.W = study.shape
        r, c = np.nonzero(study); del study
        self.r, self.c = r.astype(np.int32), c.astype(np.int32)
        self.N = len(self.r)
        self.nb = self._neighbours()
        self.protected = self._read(f"{DRV}/drv_{name}_protected.tif") > 0.5
        log(f"{name}: {self.N:,} study cells, protected {self.protected.mean()*100:.1f}%")

    def _read(self, path):
        with rasterio.open(path) as s:
            return s.read(1)[self.r, self.c]

    def _neighbours(self):
        idx = np.full((self.H, self.W), -1, np.int32)
        idx[self.r, self.c] = np.arange(self.N, dtype=np.int32)
        nb = np.empty((8, self.N), np.int32)
        offs = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
        for j, (dr, dc) in enumerate(offs):
            rr, cc = self.r + dr, self.c + dc
            ok = (rr >= 0) & (rr < self.H) & (cc >= 0) & (cc < self.W)
            v = np.full(self.N, -1, np.int32); v[ok] = idx[rr[ok], cc[ok]]; nb[j] = v
        del idx; gc.collect()
        return nb

    def lc(self, year):
        p = f"{LC24}/lc_{self.name}_2024.tif" if year == 2024 else f"{LCD}/lc_{self.name}_{year}.tif"
        a = self._read(p).astype(np.int16) - 1
        a[(a < 0) | (a >= K)] = 4                     # unmapped inside mask = water (as in Step 2)
        return a.astype(np.int8)

    def drivers(self, year):
        X = np.empty((self.N, len(FEAT)), np.float32)
        for j, f in enumerate(FEAT):
            name = f"{f}_{URBY[year]}" if f == "dist_urban" else f"{f}_{POPY[year]}" if f == "pop" else f
            a = self._read(f"{DRV}/drv_{self.name}_{name}.tif").astype(np.float32)
            a[a == -9999] = np.nan
            if f == "dem":
                a = np.maximum(a, -10)
            if f.startswith("dist") or f == "pop":
                a = np.log1p(np.maximum(a, 0))
            a[np.isnan(a)] = np.nanmedian(a)
            X[:, j] = a
        return X

    def save(self, lc, tag):
        full = np.zeros((self.H, self.W), np.uint8)
        full[self.r, self.c] = lc.astype(np.uint8) + 1
        with rasterio.open(f"{OUT}/sim_{self.name}_{tag}.tif", "w", **self.prof) as d:
            d.write(full, 1)
        del full


# ------------------------------------------------------------------ suitability
def predict_chunks(fn, X, ncol, chunk=1_000_000):
    out = np.empty((ncol, len(X)), np.float32)
    for i in range(0, len(X), chunk):
        out[:, i:i + chunk] = fn(X[i:i + chunk]).T
    return out


def ann_suitability(lc, X, rng):
    """FLUS: probability of occurrence of each class from an MLP trained on the start-year map."""
    cnt = np.bincount(lc, minlength=K)
    idx = []
    for k in range(K):
        pool = np.flatnonzero(lc == k)
        n = min(len(pool), max(3000, int(150_000 * cnt[k] / cnt.sum())))
        if n:
            idx.append(rng.choice(pool, n, replace=False))
    idx = np.concatenate(idx)
    sc = MinMaxScaler().fit(X[idx])                 # manuscript: drivers min-max normalised
    if FLUS_SUIT == "ANN":
        mlp = MLPClassifier(hidden_layer_sizes=(16,), max_iter=300, early_stopping=True, random_state=SEED)
    else:                                           # multinomial logit probability of occurrence
        mlp = LogisticRegression(max_iter=1000, C=10.0)
    mlp.fit(sc.transform(X[idx]), lc[idx])
    cols = list(mlp.classes_)

    def f(x):
        p = mlp.predict_proba(sc.transform(x))
        o = np.zeros((len(x), K), np.float32); o[:, cols] = p
        return o
    P = predict_chunks(f, X, K)
    log(f"    {FLUS_SUIT} trained on {len(idx):,} cells, train acc {mlp.score(sc.transform(X[idx]), lc[idx]):.3f}")
    return P


def leas_suitability(lc_a, lc_b, Xa, Xp, rng, region, run):
    """PLUS LEAS: per-class random forest on expansion (a->b) cells vs non-expanded cells."""
    P = np.zeros((K, len(Xp)), np.float32); imp_rows = []
    for k in range(K):
        pos = np.flatnonzero((lc_b == k) & (lc_a != k))
        if len(pos) < 30:
            log(f"    LEAS {CLS[k]}: only {len(pos)} expansion cells -> no growth probability"); continue
        neg_pool = np.flatnonzero(lc_b != k)
        npos = min(len(pos), 20_000)
        pos = rng.choice(pos, npos, replace=False)
        neg = rng.choice(neg_pool, min(len(neg_pool), 2 * npos), replace=False)
        y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
        rf = RandomForestClassifier(n_estimators=RF_TREES, max_depth=14, min_samples_leaf=5, max_features="sqrt",
                                    n_jobs=-1, random_state=SEED, oob_score=True)
        rf.fit(Xa[np.r_[pos, neg]], y)
        P[k] = predict_chunks(lambda x: rf.predict_proba(x)[:, 1:2], Xp, 1)[0]
        imp_rows += [dict(region=region, run=run, cls=CLS[k], feature=f, importance=v)
                     for f, v in zip(FEAT, rf.feature_importances_)]
        log(f"    LEAS {CLS[k]}: {len(pos):,} expansion cells, OOB acc {rf.oob_score_:.3f}")
        del rf; gc.collect()
    return P, pd.DataFrame(imp_rows)


# ------------------------------------------------------------------ demand, weights
def trans_matrix(a, b):
    M = np.bincount(a.astype(np.int64) * K + b, minlength=K * K).reshape(K, K).astype(float)
    rs = M.sum(1, keepdims=True)
    P = np.where(rs > 0, M / np.maximum(rs, 1), np.eye(K))
    return P


def markov_matrix(a, b, span, ahead):
    Pn = np.real(fractional_matrix_power(trans_matrix(a, b), ahead / span))
    Pn = np.clip(Pn, 0, None); Pn /= Pn.sum(1, keepdims=True)
    return Pn


def markov(a, b, span, base, ahead):
    return base @ markov_matrix(a, b, span, ahead)


def nb_weights(a, b):
    d = np.bincount(b, minlength=K).astype(float) - np.bincount(a, minlength=K)
    return 0.1 + 0.9 * (d - d.min()) / max(d.max() - d.min(), 1)


def integer_demand(d, N):
    t = np.floor(d).astype(np.int64)
    t[np.argsort(-(d - t))[:N - t.sum()]] += 1
    return t


def scenario_quotas(Pn, base):
    """Transition-pair quotas (cells) for 2024->2040 under the manuscript scenarios (Table 2)."""
    def rows_norm(M):
        return M / M.sum(1, keepdims=True)
    def to_q(M):
        Q = np.rint(base[:, None] * M).astype(np.int64); np.fill_diagonal(Q, 0); return Q
    out = {"NDS": to_q(Pn)}
    M = Pn.copy()                                                    # rapid development
    for c in (CROP, FOREST, SHRUB):
        M[c, IMP] *= 1.20
    M[WATER, IMP] *= 1.10
    M[IMP, [k for k in range(K) if k != IMP]] *= 0.70
    out["RDS"] = to_q(rows_norm(M))
    M = Pn.copy()                                                    # farmland protection
    M[CROP, IMP] *= 0.40
    M[CROP, [FOREST, SHRUB, WATER]] *= 0.70
    M[CROP, [SHRUB, WATER]] = 0                                      # cropland may not become grassland or water
    out["FPS"] = to_q(rows_norm(M))
    Q = out["NDS"].copy()                                            # planning conservation
    Q[PCS_PROTECT, :] = 0                                            # protected classes are not converted
    nds_inc = out["NDS"][:, IMP].sum() - out["NDS"][IMP, :].sum()
    other_in = Q[:, IMP].sum() - Q[CROP, IMP]
    Q[CROP, IMP] = max(0, int(round(PCS_IMP_SCALE * nds_inc)) - (other_in - Q[IMP, :].sum()))   # rest from cropland
    out["PCS"] = Q
    # projections allocate NET directional flows only: reciprocal c<->k exchanges in the 300 m Markov
    # matrix are largely classification noise, and allocating them through the CA would smooth the
    # landscape artificially (inflating core habitat in every simulated map). Net class change and the
    # supplier structure (e.g. cropland -> impervious) are preserved exactly.
    if NET_FLOWS:
        out = {k: np.maximum(v - v.T, 0) for k, v in out.items()}
    return {k: v for k, v in out.items() if k in SCENARIOS}


# ------------------------------------------------------------------ CA engine
def neighbourhood_box(lc, region, size):
    """Class counts in a size x size window (self excluded), via box filters on the full grid."""
    full = np.full((region.H, region.W), 255, np.uint8)
    full[region.r, region.c] = lc
    cnt = np.empty((K, lc.size), np.uint8)
    for k in range(K):
        b = cv2.boxFilter((full == k).astype(np.uint8), cv2.CV_32F, (size, size), normalize=False,
                          borderType=cv2.BORDER_CONSTANT)
        cnt[k] = b[region.r, region.c] - (lc == k); del b
    del full
    return cnt


def neighbourhood(lc, nb):
    ext = np.append(lc, np.int8(-1))
    g = ext[nb]                                    # (8, N); -1 = outside study area
    cnt = np.empty((K, lc.size), np.uint8)
    for k in range(K):
        cnt[k] = (g == k).sum(0)
    del g
    return cnt


def simulate(model, lc0, P, Q, w, region, rng):
    """Q (K,K): number of cells to convert from class c (row) to k (column); diagonal ignored.
    Pair quotas reproduce gross change (swaps), not only the net demand."""
    lc = lc0.copy(); N = lc.size
    Q = Q.astype(np.int64).copy(); np.fill_diagonal(Q, 0)
    done = np.zeros((K, K), np.int64)
    target = np.bincount(lc0, minlength=K) + Q.sum(0) - Q.sum(1)
    In, D = np.ones(K), np.ones(K)                  # FLUS inertia / PLUS adaptive D_k
    hist, rows = [], []
    for it in range(1, MAX_IT + 1):
        cur = np.bincount(lc, minlength=K)
        gap = target - cur                          # >0: class still short (net)
        pg = Q - done                               # remaining pair quotas
        if pg.sum() <= max(10, 0.002 * Q.sum()):
            break
        pin, pout = pg.sum(0), pg.sum(1)            # unmet inflow / outflow per class
        if hist:                                    # self-adaptation when a class stalls
            prev_in, prev_out = hist[-1]
            D[(pin > 0) & (pin >= 0.98 * prev_in)] *= 1.25      # PLUS D_k: more attractive
            In[(pout > 0) & (pout >= 0.98 * prev_out)] *= 0.8   # FLUS inertia: easier to leave
            D, In = np.clip(D, 0.1, 50), np.clip(In, 0.02, 10)
        hist.append((pin, pout))
        size = FLUS_NEIGH if model == "FLUS" else PLUS_NEIGH
        cnt = neighbourhood(lc, region.nb) if size == 3 else neighbourhood_box(lc, region, size)
        Om = cnt.astype(np.float32) * (w[:, None] / (size * size - 1)).astype(np.float32); del cnt
        if model == "FLUS":
            TP = P * Om
            ar = np.arange(N)                       # self-adaptive inertia on the current class
            TP[lc, ar] = P[lc, ar] * np.maximum(Om[lc, ar], np.float32(1 / (size * size - 1))) * In[lc].astype(np.float32)
            del ar
        else:  # PLUS / CARS
            r = rng.random(N, dtype=np.float32)
            TP = P * Om * D[:, None].astype(np.float32)
            seed = (Om == 0) & (r[None] < P) & (rng.random((1, N), dtype=np.float32) < SEED_PCT)
            TP = np.where(seed, P * r[None] * MU * D[:, None].astype(np.float32), TP)
            del seed
        del Om
        # conversion rules
        if PA_RESTRICT:
            TP[CROP, region.protected & (lc != CROP)] = 0
            TP[IMP, region.protected & (lc != IMP)] = 0
        # roulette wheel
        cs = np.cumsum(TP, 0); tot = cs[-1]
        u = rng.random(N, dtype=np.float32) * tot
        cand = (cs < u[None]).sum(0).astype(np.int8); del cs
        cand = np.where(tot > 0, np.minimum(cand, K - 1), lc).astype(np.int8)
        ch = np.flatnonzero((cand != lc) & (pg[lc.astype(np.int64), cand.astype(np.int64)] > 0))
        if model == "PLUS" and ch.size:
            tau = DELTA ** it * rng.random()
            ch = ch[P[cand[ch], ch] > tau]
        score = TP[cand[ch].astype(np.int64), ch] / np.maximum(tot[ch], 1e-12)   # share of the winning class
        del TP
        if ch.size == 0:
            del score
            rows.append(dict(it=it, converted=0, pair_gap=int(pg.sum())))
            if it > 20 and all(r_["converted"] == 0 for r_ in rows[-10:]):
                break
            continue
        o = np.argsort(-(score + rng.random(ch.size, dtype=np.float32) * 1e-3))  # best candidates first
        ch = ch[o]; del score, o
        tk, sk = cand[ch].astype(np.int64), lc[ch].astype(np.int64)
        key = sk * K + tk
        q = np.where(pg <= 1000, pg, np.ceil(pg * ACCEL)).astype(np.int64).ravel()
        rank = pd.Series(key).groupby(key).cumcount().to_numpy()   # rank within pair (by score)
        keep = rank < q[key]
        ch, key = ch[keep], key[keep]
        lc[ch] = (key % K).astype(np.int8)
        done += np.bincount(key, minlength=K * K).reshape(K, K)
        rows.append(dict(it=it, converted=int(ch.size), pair_gap=int(pg.sum())))
    final = target - np.bincount(lc, minlength=K)
    return lc, it, final, pd.DataFrame(rows)


# ------------------------------------------------------------------ accuracy
def multires_fom(obs0, obs1, sim1, r, c, k, b):
    """FoM of class-k gains on b x b blocks (amount agreement within blocks; Pontius multiple-resolution)."""
    og, sg = (obs1 == k) & (obs0 != k), (sim1 == k) & (obs0 != k)
    key = (r // b).astype(np.int64) * (int(c.max()) // b + 1) + c // b
    O, S = np.bincount(key, og), np.bincount(key, sg)
    hit = np.minimum(O, S).sum()
    return hit / max(O.sum() + S.sum() - hit, 1)


def suit_auc(P, lc0, lc1, rng):
    from sklearn.metrics import roc_auc_score
    out = {}
    for k, nm in ((IMP, "AUC_impervious"), (CROP, "AUC_cropland")):
        cand = np.flatnonzero(lc0 != k)
        idx = rng.choice(cand, min(len(cand), 400_000), replace=False)
        y = lc1[idx] == k
        out[nm] = roc_auc_score(y, P[k, idx]) if 0 < y.sum() < len(y) else np.nan
    return out


def accuracy(obs0, obs1, sim1, r=None, c=None):
    oa = (obs1 == sim1).mean()
    C = np.bincount(obs1.astype(np.int64) * K + sim1, minlength=K * K).reshape(K, K).astype(float)
    n = C.sum(); pe = (C.sum(0) * C.sum(1)).sum() / n ** 2
    kappa = (oa - pe) / (1 - pe)
    oc, sc = obs0 != obs1, obs0 != sim1
    A_ = (oc & ~sc).sum(); B_ = (oc & sc & (sim1 == obs1)).sum()
    C_ = (oc & sc & (sim1 != obs1)).sum(); D_ = (~oc & sc).sum()
    fom = B_ / max(A_ + B_ + C_ + D_, 1)
    extra = {}
    for k, nm in ((IMP, "FoM_impervious"), (CROP, "FoM_cropland")):
        og, sg = (obs1 == k) & (obs0 != k), (sim1 == k) & (obs0 != k)
        extra[nm] = (og & sg).sum() / max((og | sg).sum(), 1)
    if r is not None:
        for b, tag in ((5, "1.5km"), (10, "3km")):
            extra[f"FoM_impervious_{tag}"] = multires_fom(obs0, obs1, sim1, r, c, IMP, b)
            extra[f"FoM_cropland_{tag}"] = multires_fom(obs0, obs1, sim1, r, c, CROP, b)
    return dict(OA=oa, Kappa=kappa, FoM=fom, **extra, A_miss=int(A_), B_hit=int(B_), C_wrong=int(C_), D_false=int(D_),
                obs_change_pct=oc.mean() * 100, sim_change_pct=sc.mean() * 100)


# ------------------------------------------------------------------ main
if __name__ == "__main__":
    for name in REGIONS:
        R = Region(name)
        for run in RUNS:
            rng = np.random.default_rng(SEED)
            if run == "hindcast":
                y0, y1 = 2010, 2020
                lc_a, lc0, lc1 = R.lc(2000), R.lc(2010), R.lc(2020)
                quotas = {"observed": np.bincount(lc0.astype(np.int64) * K + lc1, minlength=K * K).reshape(K, K)}
                mk = markov(lc_a, lc0, 10, np.bincount(lc0, minlength=K), 10)
                w = nb_weights(lc_a, lc0)
                c0, c1 = np.bincount(lc0, minlength=K), np.bincount(lc1, minlength=K)
                dem_rows = [dict(region=name, run=run, scenario="observed", cls=CLS[k], start_km2=c0[k] * 0.09,
                                 demand_km2=c1[k] * 0.09, markov_km2=mk[k] * 0.09,
                                 markov_err_pct=(mk[k] - c1[k]) / max(c1[k], 1) * 100) for k in range(K)]
                lea_a, lea_b, Xa_year = lc_a, lc0, 2000
            else:
                y0, y1 = 2024, 2040
                lc0 = R.lc(2024); lc_p, lc_q = R.lc(2010), R.lc(2022); lc1 = None
                base = np.bincount(lc0, minlength=K).astype(float)
                quotas = scenario_quotas(markov_matrix(lc_p, lc_q, 12, 16), base)
                w = nb_weights(lc_p, lc_q)
                dem_rows = []
                for sc_, Q in quotas.items():
                    tgt = base + Q.sum(0) - Q.sum(1)
                    dem_rows += [dict(region=name, run=run, scenario=sc_, cls=CLS[k], start_km2=base[k] * 0.09,
                                      demand_km2=tgt[k] * 0.09) for k in range(K)]
                lea_a, lea_b, Xa_year = lc_p, lc_q, 2010
            append_csv(pd.DataFrame(dem_rows), "demand.csv")
            log(f"== {name} {run}: {y0}->{y1}  weights {np.round(w, 2).tolist()}")
            for sc_, Q in quotas.items():
                log(f"   {sc_} net change (km2):", {CLS[k]: round(float(Q[:, k].sum() - Q[k, :].sum()) * 0.09) for k in range(K)})
            X0 = R.drivers(y0)
            acc_rows, area_rows = [], []
            for model in MODELS:
                t0 = time.time()
                if model == "FLUS":
                    P = ann_suitability(lc0, X0, rng)
                else:
                    Xa = R.drivers(Xa_year) if Xa_year != y0 else X0
                    P, imp = leas_suitability(lea_a, lea_b, Xa, X0, rng, name, run)
                    imp.insert(2, "model", "PLUS")
                    append_csv(imp, "leas_importance.csv")
                    if Xa is not X0:
                        del Xa
                    gc.collect()
                auc = suit_auc(P, lc0, lc1, np.random.default_rng(SEED)) if lc1 is not None else {}
                log(f"   {model} suitability ready ({time.time() - t0:.0f}s)")
                for sc_, Q in quotas.items():
                    t1 = time.time()
                    sim, its, final, calog = simulate(model, lc0, P, Q, w, R, np.random.default_rng(SEED))
                    tag = f"{model}_{run}" + ("" if run == "hindcast" else f"_{sc_}")
                    calog.insert(0, "scenario", sc_); calog.insert(0, "run", run)
                    calog.insert(0, "model", model); calog.insert(0, "region", name)
                    append_csv(calog, "ca_log.csv")
                    R.save(sim, tag)
                    log(f"   {model} {sc_}: {its} iterations, residual {int(np.abs(final).sum()):,} cells, {time.time() - t1:.0f}s")
                    cnt = np.bincount(sim, minlength=K)
                    area_rows += [dict(region=name, model=model, run=run, scenario=sc_, cls=CLS[k], year=y1,
                                       km2=cnt[k] * 0.09, pct=cnt[k] / R.N * 100) for k in range(K)]
                    if lc1 is not None:
                        a = accuracy(lc0, lc1, sim, R.r, R.c); a.update(region=name, model=model, run=run, **auc)
                        acc_rows.append(a)
                        log(f"   {model} accuracy: OA {a['OA']:.4f}  Kappa {a['Kappa']:.4f}  FoM {a['FoM']:.4f}  "
                            f"FoM_imp {a['FoM_impervious']:.3f} (1.5km {a['FoM_impervious_1.5km']:.3f}, 3km {a['FoM_impervious_3km']:.3f})  "
                            f"FoM_crop {a['FoM_cropland']:.3f}  AUC_imp {a['AUC_impervious']:.3f}  "
                            f"(obs change {a['obs_change_pct']:.2f}%, sim change {a['sim_change_pct']:.2f}%)")
                    del sim; gc.collect()
                del P; gc.collect()
            append_csv(pd.DataFrame(area_rows), "areas.csv")
            if acc_rows:
                append_csv(pd.DataFrame(acc_rows), "accuracy.csv")
            del X0; gc.collect()
        del R; gc.collect()
    pd.set_option("display.width", 250)
    if os.path.exists(f"{OUT}/accuracy.csv"):
        print(pd.read_csv(f"{OUT}/accuracy.csv")[["region", "model", "OA", "Kappa", "FoM", "FoM_impervious", "FoM_impervious_3km", "FoM_cropland", "AUC_impervious", "sim_change_pct"]].round(4).to_string(index=False))
    ar = pd.read_csv(f"{OUT}/areas.csv")
    ar = ar[ar.run == "2040"]
    if len(ar):
        print(ar.pivot_table(index=["region", "cls"], columns=["scenario", "model"], values="pct").round(2))
    log("STEP 5 DONE ->", OUT)


05:14:13 SoutheastAsia: 21,532,583 study cells, protected 12.7%
05:14:21 == SoutheastAsia 2040: 2024->2040  weights [1.0, 0.1, 0.76, 0.62, 0.69, 0.66, 0.65]
05:14:21    NDS net change (km2): {'Cropland': 65297, 'Forest': -91452, 'ShrubGrass': 14964, 'Wetland': -2431, 'Water': 7904, 'Impervious': 4165, 'Bare': 1552}
05:14:21    RDS net change (km2): {'Cropland': 64236, 'Forest': -91834, 'ShrubGrass': 14918, 'Wetland': -2460, 'Water': 7866, 'Impervious': 5729, 'Bare': 1545}
05:14:21    FPS net change (km2): {'Cropland': 73704, 'Forest': -93617, 'ShrubGrass': 12393, 'Wetland': -2404, 'Water': 6271, 'Impervious': 2084, 'Bare': 1570}
05:14:21    PCS net change (km2): {'Cropland': -15076, 'Forest': 9088, 'ShrubGrass': 2626, 'Wetland': 1451, 'Water': 1723, 'Impervious': 2707, 'Bare': -2518}
05:19:33     LEAS Cropland: 20,000 expansion cells, OOB acc 0.793
05:23:27     LEAS Forest: 20,000 expansion cells, OOB acc 0.813
05:26:53     LEAS ShrubGrass: 20,000 expansion cells, OOB acc 0.845
05:30:2

In [ ]:
# STEP 6 - green-infrastructure morphology, habitat quality, resistance and ecological network
import os
assert os.path.ismount("/content/drive"), "Google Drive is not mounted - run STEP 0 first"
os.environ["REGIONS"] = "WesternAsia,EastAsia,SouthAsia,SoutheastAsia"
os.environ["LANDSCAPES"] = "all"
"""STEP 6 (Colab) - GI network analysis following the YRD manuscript (Sections 2.5-2.6, Table 2),
adapted to the Asian coastal zone. Every landscape passes through the same chain:

  MSPA-type morphology  GI foreground = forest + wetland (mangrove, salt marsh, tidal flat; the YRD study
                        used forest only because wetland was not simulated there); 8-connectivity;
                        edge width 300 m (1 cell); Core/Islet/Perforation/Edge/Loop/Bridge/Branch.
  Habitat quality       InVEST formulation, k = 0.5, z = 2.5; threats impervious (10 km, exponential,
                        w 1.0), cropland (4 km, linear, 0.6), bare (2 km, linear, 0.2); fixed suitability
                        and sensitivity tables.
  Resistance            LULC 0.50 + slope 0.20 + accessibility (distance from impervious) 0.30,
                        each on a 1-1,000 scale (reported also as a 1-10 index = R/100).
  Vulnerability         EVI = 0.5 (R-1)/999 + 0.5 (1-HQ); slight < 0.40 <= moderate < 0.60 <= high
                        (identical thresholds for all years, models, scenarios and regions).
  Sources               MSPA Core patches >= 50 km2; candidates ranked by dPC; the number kept scales
                        with landscape area from the YRD rule (18 sources for ~218,000 km2).
  Corridors             least-cost paths (Dijkstra / MCP) between sources whose cost-allocation zones
                        touch (Linkage-Mapper logic), on a 1.5 km network grid.
  KDE                   quartic kernel, 20 km bandwidth, normalised 0-1, classes at 0.25/0.50/0.75.
  Connectivity          PC / dPC (intra, flux, connector), p = 0.5 at 50 km cost-weighted distance
                        (cost per km = resistance on the 1-10 index, i.e. R / 100),
                        degree, interactions I_ij (strong if >= 0.10), ECA.
  Circuit current       pairwise current for every corridor-linked pair (Circuitscape rule) on a 3 km grid.

Landscapes: 1990, 2000, 2010, 2020, 2024 (observed) and 2040 NDS / PCS for FLUS and PLUS.
Outputs (Drive: AsiaCoastGI/step6/<Region>/): mspa_, hq_, res_, vul_ (300 m); src_, corr_, kde_ (1.5 km);
  cur_ (3 km); nodes_, links_ csv per landscape; step6/summary.csv (one row per region x landscape).
Resumable: a landscape whose summary row exists is skipped.
"""
import gc, time, json
import numpy as np, pandas as pd, rasterio, cv2
from rasterio.transform import Affine
from scipy import ndimage as ndi, sparse
from scipy.signal import oaconvolve
from scipy.sparse.csgraph import dijkstra
from scipy.sparse.linalg import splu, cg
from skimage.graph import MCP_Connect

ROOT = "/content/drive/MyDrive/AsiaCoastGI"
LCD, LC24, SIM, DRV, MASK = (f"{ROOT}/step2_filtered", f"{ROOT}/step3", f"{ROOT}/step5",
                             f"{ROOT}/step4", f"{ROOT}/step2")
OUT = f"{ROOT}/step6"; os.makedirs(OUT, exist_ok=True)
REGIONS = os.environ.get("REGIONS", "WesternAsia,EastAsia,SouthAsia,SoutheastAsia").split(",")
ALL_LS = ["2020", "2024", "2040_FLUS_NDS", "2040_FLUS_PCS", "2040_PLUS_NDS", "2040_PLUS_PCS",
          "1990", "2000", "2010"]
LS = ALL_LS if os.environ.get("LANDSCAPES", "all") == "all" else os.environ["LANDSCAPES"].split(",")

CROP, FOREST, SHRUB, WETLAND, WATER, IMPERV, BARE = range(1, 8)
CELL_KM = 0.3
GI_FOREGROUND = (FOREST, WETLAND)
HQ_K, HQ_Z = 0.5, 2.5
HQ_SUIT = {CROP: 0.4, FOREST: 1.0, SHRUB: 0.8, WETLAND: 1.0, WATER: 0.7, IMPERV: 0.0, BARE: 0.2}
HQ_THREATS = {IMPERV: (10.0, 1.0, "exponential"), CROP: (4.0, 0.6, "linear"), BARE: (2.0, 0.2, "linear")}
HQ_SENS = {CROP: {IMPERV: 0.5, CROP: 0.0, BARE: 0.1}, FOREST: {IMPERV: 0.9, CROP: 0.6, BARE: 0.3},
           SHRUB: {IMPERV: 0.8, CROP: 0.6, BARE: 0.3}, WETLAND: {IMPERV: 0.9, CROP: 0.7, BARE: 0.4},
           WATER: {IMPERV: 0.8, CROP: 0.6, BARE: 0.2}, IMPERV: {}, BARE: {IMPERV: 0.3, CROP: 0.2}}
RES_LULC = {FOREST: 1, WETLAND: 5, SHRUB: 10, WATER: 100, CROP: 300, BARE: 400, IMPERV: 1000}
RES_W = dict(lulc=0.5, slope=0.2, access=0.3)
ACCESS_HALF_KM = 5.0
SRC_MIN_KM2, YRD_KM2, YRD_N = 50.0, 218_000.0, 18
DISPERSAL_KM, STRONG = 50.0, 0.10
NET_F, CUR_F = 5, 10                       # 1.5 km network grid, 3 km circuit grid
KDE_BW_KM = 20.0


def log(*a): print(time.strftime("%H:%M:%S"), *a, flush=True)


# ------------------------------------------------------------------ inputs
def read_lc(name, ls):
    if ls == "2024":
        p = f"{LC24}/lc_{name}_2024.tif"
    elif ls.startswith("2040"):
        _, model, scen = ls.split("_"); p = f"{SIM}/sim_{name}_{model}_2040_{scen}.tif"
    else:
        p = f"{LCD}/lc_{name}_{ls}.tif"
    with rasterio.open(p) as s:
        return s.read(1), s.profile


def write(path, arr, prof, dtype, nodata, transform=None):
    p = dict(prof, driver="GTiff", count=1, dtype=dtype, nodata=nodata, compress="deflate", tiled=True,
             blockxsize=256, blockysize=256, height=arr.shape[0], width=arr.shape[1], BIGTIFF="IF_SAFER")
    if transform is not None:
        p["transform"] = transform
    with rasterio.open(path, "w", **p) as d:
        d.write(arr.astype(dtype), 1)


# ------------------------------------------------------------------ morphology
S8 = np.ones((3, 3), bool)


def mspa(fg, w=1):
    """MSPA-type classes (1 Core 2 Islet 3 Perforation 4 Edge 5 Loop 6 Bridge 7 Branch), 8-connectivity."""
    out = np.zeros(fg.shape, np.uint8)
    core = ndi.binary_erosion(fg, S8, iterations=w, border_value=0)
    lab, n = ndi.label(fg, S8)
    has_core = np.zeros(n + 1, bool); has_core[np.unique(lab[core])] = True; has_core[0] = False
    islet = fg & ~has_core[lab]; del lab; gc.collect()
    near_core = ndi.binary_dilation(core, S8, iterations=w)
    boundary = fg & ~core & ~islet & near_core; del near_core
    holes = ndi.binary_fill_holes(fg) & ~fg
    perf = boundary & ndi.binary_dilation(holes, S8); del holes
    out[core] = 1; out[islet] = 2; out[perf] = 3; out[boundary & ~perf] = 4
    rest = fg & ~core & ~islet & ~boundary; del boundary, perf, islet
    if rest.any():
        # memory-lean: one int32 work array at a time (grids reach 285 M cells)
        comp, nc = ndi.label(rest, S8); idx = np.arange(1, nc + 1)
        area_lab, _ = ndi.label(core | (out == 4) | (out == 3), S8)
        big = int(area_lab.max()) + 1
        lab_max = ndi.grey_dilation(area_lab, footprint=S8)
        cmax = ndi.maximum(lab_max, comp, idx)
        contact = rest & (lab_max > 0); del lab_max; gc.collect()
        pos = area_lab > 0
        area_lab[pos] = big - area_lab[pos]; del pos            # invert labels in place
        inv = ndi.grey_dilation(area_lab, footprint=S8); del area_lab; gc.collect()
        pos = inv > 0
        inv[pos] = big - inv[pos]; inv[~pos] = big; del pos     # inv now holds the neighbouring min label
        cmin = ndi.minimum(inv, comp, idx); del inv; gc.collect()
        bridge = (cmax > 0) & (cmin < big) & (cmin != cmax)
        clab, ncl = ndi.label(contact, S8)
        seg_comp = ndi.maximum(comp, clab, np.arange(1, ncl + 1)).astype(int)
        nseg = np.bincount(seg_comp, minlength=nc + 1)[1:]
        cls = np.where(bridge, 6, np.where(nseg >= 2, 5, 7)).astype(np.uint8)
        out[rest] = np.r_[0, cls][comp[rest]]
    return out


# ------------------------------------------------------------------ habitat quality, resistance
def kernel(max_km, decay):
    r = int(np.ceil(max_km / CELL_KM)); y, x = np.mgrid[-r:r + 1, -r:r + 1]
    d = np.hypot(x, y) * CELL_KM
    k = np.exp(-2.99 * d / max_km) if decay == "exponential" else 1 - d / max_km
    k[d > max_km] = 0
    return (k / k.sum()).astype(np.float32)


def conv_rows(a, k, block=1536):
    """'same' convolution in row blocks with a halo (keeps FFT memory bounded)."""
    h = k.shape[0] // 2; out = np.empty(a.shape, np.float32)
    for r0 in range(0, a.shape[0], block):
        r1 = min(r0 + block, a.shape[0]); a0, a1 = max(r0 - h, 0), min(r1 + h, a.shape[0])
        c = oaconvolve(a[a0:a1], k, mode="same")
        out[r0:r1] = c[r0 - a0:r0 - a0 + (r1 - r0)]
    return np.clip(out, 0, None)


def habitat_quality(lc, mask):
    wsum = sum(v[1] for v in HQ_THREATS.values())
    D = np.zeros(lc.shape, np.float32)
    for r, (dmax, w, decay) in HQ_THREATS.items():
        imp = conv_rows(((lc == r) & mask).astype(np.float32), kernel(dmax, decay))
        lut = np.zeros(256, np.float32)
        for j, sd in HQ_SENS.items():
            lut[j] = sd.get(r, 0.0) * w / wsum
        imp *= lut[lc]; D += imp; del imp; gc.collect()
    np.power(D, HQ_Z, out=D)                                     # D -> D^z in place
    D /= (D + HQ_K ** HQ_Z); np.subtract(1, D, out=D)
    lut = np.zeros(256, np.float32)
    for j, h in HQ_SUIT.items():
        lut[j] = h
    D *= lut[lc]; Q = D
    Q[~mask] = np.nan
    return Q


def resistance(lc, mask, slope):
    R = np.zeros(lc.shape, np.float32)
    for c, v in RES_LULC.items():
        R[lc == c] = RES_W["lulc"] * v
    R += RES_W["slope"] * (1 + 999 * (np.minimum(slope, 45).astype(np.float32) / np.float32(45.0)))
    d = cv2.distanceTransform((lc != IMPERV).astype(np.uint8), cv2.DIST_L2, cv2.DIST_MASK_PRECISE) * CELL_KM
    R += RES_W["access"] * (1 + 999 * np.exp(-np.log(2) * d / ACCESS_HALF_KM)); del d
    R[~mask] = np.nan
    return R


# ------------------------------------------------------------------ network helpers
def block(a, f, how):
    h, w = (a.shape[0] // f) * f, (a.shape[1] // f) * f
    b = a[:h, :w].reshape(h // f, f, w // f, f)
    if how == "nanmean":
        v = np.isfinite(b); s = np.where(v, b, 0).sum((1, 3)); n = v.sum((1, 3))
        return np.where(n * 2 >= f * f, s / np.maximum(n, 1), np.nan).astype(np.float32)
    if how == "max":
        return b.max((1, 3))
    if how == "frac":
        return b.mean((1, 3), dtype=np.float32)


class Linker(MCP_Connect):
    def __init__(self, costs, start_label):
        super().__init__(costs, fully_connected=True)
        self.start_label = start_label; self.best = {}

    def create_connection(self, id1, id2, pos1, pos2, cost1, cost2):
        a, b = self.start_label[id1], self.start_label[id2]
        if a == b:
            return
        key = (min(a, b), max(a, b)); c = cost1 + cost2
        if key not in self.best or c < self.best[key][0]:
            self.best[key] = (c, pos1, pos2) if a < b else (c, pos2, pos1)


def corridors(src, R, cell_km):
    cost = np.where(np.isnan(R), np.inf, R / 100.0).astype(np.float64)   # 1-10 resistance index (manuscript)
    cost[src > 0] = 0.01
    edge = (src > 0) & ~ndi.binary_erosion(src > 0, S8)
    starts = np.argwhere(edge); L = Linker(cost, src[edge])
    L.find_costs([tuple(s) for s in starts])
    links, paths = [], np.zeros(src.shape, np.uint16)
    for (i, j), (c, p1, p2) in sorted(L.best.items()):
        path = [q for q in L.traceback(p1)[::-1] + L.traceback(p2) if src[q] == 0] or [tuple(p1)]
        arr = np.array(path)
        steps = np.hypot(*np.diff(arr, axis=0).T).sum() if len(arr) > 1 else 1.0
        links.append(dict(i=int(i), j=int(j), cwd_km=float(c * cell_km), length_km=float(steps * cell_km), path=path))
        paths[arr[:, 0], arr[:, 1]] += 1
    return links, paths


def pc_analysis(area, links, A_L):
    n = len(area); a = np.asarray(area, float)
    theta = np.log(2) / DISPERSAL_KM
    W = sparse.lil_matrix((n, n)); deg = np.zeros(n, int)
    for l in links:
        i, j = l["i"] - 1, l["j"] - 1; w = max(theta * l["cwd_km"], 1e-12)
        if W[i, j] == 0 or w < W[i, j]:
            W[i, j] = W[j, i] = w
        deg[i] += 1; deg[j] += 1
    W = W.tocsr()

    def num_of(Wm, act):
        P = np.exp(-dijkstra(Wm, directed=False)); np.fill_diagonal(P, 1.0)
        P[~act, :] = 0; P[:, ~act] = 0
        return (a[:, None] * a[None, :] * P).sum(), P
    act = np.ones(n, bool); num, P = num_of(W, act)
    dpc = np.zeros(n)
    for k in range(n):
        Wk = W.tolil(); Wk[k, :] = 0; Wk[:, k] = 0; ak = act.copy(); ak[k] = False
        dpc[k] = (num - num_of(Wk.tocsr(), ak)[0]) / num * 100
    intra = a ** 2 / num * 100
    flux = (2 * (a[:, None] * a[None, :] * P).sum(1) - 2 * a ** 2) / num * 100
    prod = a[:, None] * a[None, :] * P; np.fill_diagonal(prod, 0)
    I = prod / prod.max() if prod.max() > 0 else prod
    iu = np.triu_indices(n, 1)
    return dict(PC=num / A_L ** 2, ECA_km2=float(np.sqrt(num)), dPC=dpc, intra=intra, flux=flux,
                connector=np.clip(dpc - intra - flux, 0, None), degree=deg, I=I,
                strong=int((I[iu] >= STRONG).sum()))


def kde(paths, cell_km):
    r = int(np.ceil(KDE_BW_KM / cell_km)); y, x = np.mgrid[-r:r + 1, -r:r + 1]
    d = np.hypot(x, y) * cell_km / KDE_BW_KM
    k = np.where(d <= 1, 3 / np.pi * (1 - d ** 2) ** 2, 0).astype(np.float32)
    out = oaconvolve((paths > 0).astype(np.float32), k, mode="same").clip(0, None)
    return out / out.max() if out.max() > 0 else out


def circuit(R, src, pairs):
    valid = ~np.isnan(R); h, w = R.shape
    node = -np.ones(R.shape, np.int64); ns = int(src.max())
    free = valid & (src == 0); node[free] = ns + np.arange(free.sum()); node[src > 0] = src[src > 0] - 1
    nn = ns + int(free.sum())
    g = np.where(valid, 1.0 / np.where(src > 0, 1.0, R), 0)
    rows, cols, vals = [], [], []
    for dr, dc, dist in [(0, 1, 1), (1, 0, 1), (1, 1, np.sqrt(2)), (1, -1, np.sqrt(2))]:
        a_ = (slice(0, h - dr), slice(0, w - dc)) if dc >= 0 else (slice(0, h - dr), slice(-dc, w))
        b_ = (slice(dr, h), slice(dc, w)) if dc >= 0 else (slice(dr, h), slice(0, w + dc))
        ok = valid[a_] & valid[b_]
        na, nb = node[a_][ok], node[b_][ok]
        cond = 1.0 / ((1 / g[a_][ok] + 1 / g[b_][ok]) / 2 * dist)
        k = na != nb; rows.append(na[k]); cols.append(nb[k]); vals.append(cond[k])
    r_, c_, v_ = np.concatenate(rows), np.concatenate(cols), np.concatenate(vals)
    G = sparse.coo_matrix((np.r_[v_, v_], (np.r_[r_, c_], np.r_[c_, r_])), shape=(nn, nn)).tocsr()
    G.sum_duplicates()
    M = (sparse.diags(np.asarray(G.sum(1)).ravel()) - G + 1e-8 * sparse.identity(nn)).tocsc()
    try:
        solve = splu(M).solve
    except MemoryError:
        solve = lambda b: cg(M, b, rtol=1e-8, maxiter=5000)[0]
    Gc = G.tocoo(); cur = np.zeros(nn)
    for i, j in pairs:
        b = np.zeros(nn); b[i - 1], b[j - 1] = 1.0, -1.0
        v = solve(b)
        cur += 0.5 * np.bincount(Gc.row, weights=np.abs(v[Gc.row] - v[Gc.col]) * Gc.data, minlength=nn)
    out = np.full(R.shape, np.nan, np.float32); ok = node >= 0
    out[ok] = cur[node[ok]]; out[src > 0] = np.nan
    return out


# ------------------------------------------------------------------ one landscape
def run_landscape(name, ls, mask, slope, prof, A_L, n_keep, odir):
    t0 = time.time()
    lc, _ = read_lc(name, ls)
    lc = np.where(mask, lc, 0).astype(np.uint8)
    T = prof["transform"]
    row = dict(region=name, landscape=ls, study_km2=A_L, quota="net" if ls.startswith("2040") else "")
    for k, nm in enumerate(["cropland", "forest", "shrubgrass", "wetland", "water", "impervious", "bare"], 1):
        row[f"pct_{nm}"] = (lc == k).sum() / mask.sum() * 100
    # morphology
    fg = np.isin(lc, GI_FOREGROUND) & mask
    m = mspa(fg); ftot = max(fg.sum(), 1)
    for k, nm in enumerate(["core", "islet", "perforation", "edge", "loop", "bridge", "branch"], 1):
        row[f"mspa_{nm}_pct"] = (m == k).sum() / ftot * 100
    write(f"{odir}/mspa_{ls}.tif", m, prof, "uint8", 0); del fg; gc.collect()
    log(f"  {ls}: MSPA done ({time.time() - t0:.0f}s), core {row['mspa_core_pct']:.1f}% of GI")
    # habitat quality, resistance, vulnerability
    Q = habitat_quality(lc, mask)
    row["hq_mean"] = float(np.nanmean(Q))
    write(f"{odir}/hq_{ls}.tif", np.where(np.isnan(Q), 255, np.round(Q * 100)), prof, "uint8", 255)
    R = resistance(lc, mask, slope)
    row["res_mean_1_10"] = float(np.nanmean(R) / 100)
    write(f"{odir}/res_{ls}.tif", np.where(np.isnan(R), 0, np.round(R)), prof, "uint16", 0)
    evi = 0.5 * (R - 1) / 999 + 0.5 * (1 - np.nan_to_num(Q))
    vul = np.where(mask, np.digitize(evi, [0.40, 0.60]) + 1, 0).astype(np.uint8); del evi
    for k, nm in ((1, "slight"), (2, "moderate"), (3, "high")):
        row[f"vul_{nm}_pct"] = (vul == k).sum() / mask.sum() * 100
    write(f"{odir}/vul_{ls}.tif", vul, prof, "uint8", 0); del vul
    log(f"  {ls}: HQ {row['hq_mean']:.3f}, resistance {row['res_mean_1_10']:.2f} ({time.time() - t0:.0f}s)")
    # candidate sources: Core patches >= 50 km2
    lab, n = ndi.label(m == 1, S8); del m
    idx = np.arange(1, n + 1)
    cnt_ = np.bincount(lab.ravel(), minlength=n + 1)
    area = cnt_[1:] * CELL_KM ** 2
    hq = (np.bincount(lab.ravel(), weights=np.nan_to_num(Q).ravel(), minlength=n + 1)[1:] / np.maximum(cnt_[1:], 1)); del cnt_
    cand = idx[area >= SRC_MIN_KM2]
    cand = cand[np.argsort(-area[cand - 1])][: 3 * n_keep]           # cap the candidate set
    # network grids
    Tn = T * Affine.scale(NET_F); Rn = block(R, NET_F, "nanmean")
    lut = np.zeros(n + 1, np.int32); lut[cand] = np.arange(1, len(cand) + 1)
    S = lut[lab]; del lab

    def net_sources(Sfull):
        lab_n = block(Sfull, NET_F, "max").astype(np.int32)
        frac = block((Sfull > 0).astype(np.float32), NET_F, "frac")
        lab_n[frac < 0.5] = 0; lab_n[np.isnan(Rn)] = 0
        return lab_n

    def network(Sfull, ids_area):
        Sn = net_sources(Sfull)
        present = np.unique(Sn[Sn > 0])
        remap = np.zeros(int(Sfull.max()) + 1, np.int32); remap[present] = np.arange(1, len(present) + 1)
        Sn = remap[Sn]
        links, paths = corridors(Sn, Rn, CELL_KM * NET_F) if len(present) > 1 else ([], np.zeros(Sn.shape, np.uint16))
        pc = pc_analysis([ids_area[p - 1] for p in present], links, A_L) if len(present) else None
        return Sn, present, links, paths, pc

    cand_area = area[cand - 1]
    if len(cand) > n_keep:                                          # rank candidates by dPC, keep n_keep
        _, present, _, _, pc0 = network(S, cand_area)
        order = present[np.argsort(-pc0["dPC"])][:n_keep]
        lut2 = np.zeros(len(cand) + 1, np.int32); lut2[order] = np.arange(1, len(order) + 1)
        S = lut2[S]; keep_ids = cand[order - 1]
    else:
        keep_ids = cand
    src_area, src_hq = area[keep_ids - 1], hq[keep_ids - 1]
    row.update(n_candidates=int(len(cand)), n_sources=int(len(keep_ids)), source_km2=float(src_area.sum()),
               source_hq_mean=float(np.average(src_hq, weights=src_area)) if len(keep_ids) else np.nan)
    Sn, present, links, paths, pc = network(S, src_area); del S
    log(f"  {ls}: {len(keep_ids)} sources ({row['source_km2']:,.0f} km2), {len(links)} corridors ({time.time() - t0:.0f}s)")
    write(f"{odir}/src_{ls}.tif", Sn, prof, "int16", 0, Tn)
    write(f"{odir}/corr_{ls}.tif", paths, prof, "uint16", 0, Tn)
    K_ = kde(paths, CELL_KM * NET_F); K_[np.isnan(Rn)] = np.nan
    write(f"{odir}/kde_{ls}.tif", np.nan_to_num(K_, nan=-1), prof, "float32", -1, Tn)
    cell_n = (CELL_KM * NET_F) ** 2
    for thr, nm in ((0.25, "low"), (0.50, "mid"), (0.75, "hot")):
        row[f"kde_ge{int(thr * 100)}_km2"] = float((K_ >= thr).sum() * cell_n)
    row.update(n_corridors=len(links), corridor_km=float(sum(l["length_km"] for l in links)),
               corridor_cwd_mean=float(np.mean([l["cwd_km"] for l in links])) if links else np.nan)
    if pc is not None:
        row.update(PC=pc["PC"], ECA_km2=pc["ECA_km2"], ECA_pct=pc["ECA_km2"] / A_L * 100, strong_pairs=pc["strong"],
                   top_dPC=float(pc["dPC"].max()), mean_degree=float(pc["degree"].mean()))
        pd.DataFrame(dict(source=np.arange(1, len(present) + 1), area_km2=src_area[present - 1],
                          hq_mean=src_hq[present - 1], dPC=pc["dPC"], dPC_intra=pc["intra"], dPC_flux=pc["flux"],
                          dPC_connector=pc["connector"], degree=pc["degree"])).to_csv(f"{odir}/nodes_{ls}.csv", index=False)
        pd.DataFrame([dict(i=l["i"], j=l["j"], cwd_km=l["cwd_km"], length_km=l["length_km"],
                           I_ij=float(pc["I"][l["i"] - 1, l["j"] - 1])) for l in links]).to_csv(f"{odir}/links_{ls}.csv", index=False)
    # circuit current on the 3 km grid
    if links:
        Rc = block(R, CUR_F, "nanmean")
        Sc = block(Sn, CUR_F // NET_F, "max").astype(np.int32); Sc[np.isnan(Rc)] = 0
        pres_c = set(np.unique(Sc[Sc > 0]).tolist())
        pairs = sorted({(l["i"], l["j"]) for l in links if l["i"] in pres_c and l["j"] in pres_c})
        rm = np.zeros(int(Sc.max()) + 1, np.int64); keep = sorted(pres_c); rm[keep] = np.arange(1, len(keep) + 1)
        cur = circuit(Rc, rm[Sc], [(rm[i], rm[j]) for i, j in pairs])
        write(f"{odir}/cur_{ls}.tif", np.nan_to_num(cur, nan=-1), prof, "float32", -1, T * Affine.scale(CUR_F))
        row["current_mean"] = float(np.nanmean(cur)); row["n_circuit_pairs"] = len(pairs)
        del Rc, Sc, cur
    del Q, R, Rn, Sn, paths, K_; gc.collect()
    row["seconds"] = round(time.time() - t0)
    log(f"  {ls}: done in {row['seconds']}s  PC {row.get('PC', np.nan):.2e}  strong pairs {row.get('strong_pairs', 0)}")
    return row


if __name__ == "__main__":
    sp = f"{OUT}/summary.csv"
    summ = pd.read_csv(sp) if os.path.exists(sp) else pd.DataFrame(columns=["region", "landscape"])
    if "quota" not in summ.columns:
        summ["quota"] = np.nan
    stale = summ.landscape.astype(str).str.startswith("2040") & (summ.quota != "net")   # made from gross-swap simulations
    if stale.any():
        log(f"recomputing {int(stale.sum())} 2040 landscapes made from the earlier gross-swap simulations")
        summ = summ[~stale]
    for name in REGIONS:
        odir = f"{OUT}/{name}"; os.makedirs(odir, exist_ok=True)
        with rasterio.open(f"{MASK}/studymask_{name}.tif") as s:
            mask = s.read(1) > 0; prof = s.profile
        with rasterio.open(f"{DRV}/drv_{name}_slope.tif") as s:
            slope = s.read(1)
        slope = np.clip(np.where(slope == -9999, 0, np.round(slope)), 0, 90).astype(np.uint8)   # degrees, 1 byte
        A_L = float(mask.sum() * CELL_KM ** 2)
        n_keep = max(5, int(round(YRD_N * A_L / YRD_KM2)))
        log(f"{name}: study area {A_L:,.0f} km2 -> up to {n_keep} sources")
        for ls in LS:
            if ((summ.region == name) & (summ.landscape == ls)).any():
                log(f"  {ls}: done already"); continue
            if ls.startswith("2040") and not os.path.exists(f"{SIM}/sim_{name}_{ls.split('_')[1]}_2040_{ls.split('_')[2]}.tif"):
                log(f"  {ls}: simulation not available yet"); continue
            try:
                row = run_landscape(name, ls, mask, slope, prof, A_L, n_keep, odir)
            except Exception as e:
                import traceback; traceback.print_exc(); log(f"  !! {ls}: {e!r}"); continue
            summ = pd.concat([summ[~((summ.region == name) & (summ.landscape == ls))], pd.DataFrame([row])], ignore_index=True)
            summ.to_csv(sp, index=False)
        del mask, slope; gc.collect()
    pd.set_option("display.width", 250)
    cols = ["region", "landscape", "pct_impervious", "mspa_core_pct", "hq_mean", "res_mean_1_10", "vul_high_pct",
            "n_sources", "source_km2", "n_corridors", "corridor_km", "ECA_pct", "strong_pairs", "kde_ge75_km2"]
    print(summ[[c for c in cols if c in summ.columns]].round(3).to_string(index=False))
    log("STEP 6 DONE ->", OUT)


In [ ]:
ge
for f,n in (('Fig3_urbanisation_hotspots','v3'),('Fig5_scenarios_2040','v5')):
    im=Image.open('figs/'+f+'.png').convert('RGB'); im=im.resize((1400,int(im.size[1]*1400/im.size[0])),Image.LANCZOS); im.save('/content/'+n+'.jpg',quality=93)
    print('ok')"


In [ ]:
# STEP 8 - Figures 2-14 (Times New Roman, 600 dpi PNG + PDF)
import os, sys, glob, json, math
"""Figures 2-14 for the Asian coastal green-infrastructure study.

Two layers, as for Figure 1:
  prep_*()  Colab only (rasterio + pyproj). Reads the Drive rasters once and writes compact caches to
            figs/cache (3 km block tables, zoom-window arrays, copies of the Step 5/6 tables).
  fig*()    numpy / pandas / matplotlib only. Draws from the caches, so styling can be iterated in seconds.
Run:  python step8_figures.py prep 3 4      (prep caches, then draw Figs 3 and 4)
      python step8_figures.py 2 3 4 5 ...   (draw only)
Figure list
  2  Methodological framework
  3  Coastal urbanisation 1990-2024: hexagon map of impervious gain + ten hotspot windows coloured by epoch
  4  LULC composition 1990-2024 and 2040 scenarios by region + coast-wide alluvial flows
  5  2040 scenario maps in four hotspot windows (FLUS / PLUS) + net change by class and scenario
  6  Model validation (multi-resolution FoM, AUC) and driver importance (LEAS random forest)
  7  Landscape resistance: 2024 hexagon map, 2040 change under NDS / PCS, regional distributions
  8  Ecological sources, corridors and corridor density by region (2024, 2040 NDS, 2040 PCS)
  9  Network structure: dPC decomposition and source-interaction summaries
 10  Ecological current: coast-wide hexagons + zoom windows on pinch points (2040 NDS vs PCS)
 11  Morphology (MSPA core) and habitat quality: 2024 maps and 2040 PCS - NDS differences
 12  Trajectories 1990-2024 and 2040 scenario outcomes of key indicators
 13  Relative improvement of PCS over NDS in 2040 by region and model
 14  Conservation sequence: priority zones and their area by region
"""
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, matplotlib.ticker as mt
from matplotlib import font_manager
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Polygon as MPoly, Rectangle
from matplotlib.collections import PolyCollection, LineCollection

ROOT = os.environ.get("ROOT", "/content/drive/MyDrive/AsiaCoastGI")
FIG = f"{ROOT}/figs"; CACHE = f"{FIG}/cache"
REGIONS = {"WesternAsia": ("A", "Western Asia", "WA"), "SouthAsia": ("B", "South Asia", "SA"),
           "SoutheastAsia": ("C", "Southeast Asia", "SEA"), "EastAsia": ("D", "East Asia", "EA")}
REG_COLS = {"WesternAsia": "#B03A2E", "SouthAsia": "#B9770E", "SoutheastAsia": "#0E6655", "EastAsia": "#1A4592"}
CLS = ["Cropland", "Forest", "Shrub/grass", "Wetland", "Water", "Impervious", "Bare"]
LULC_COLS = ["#F4D03F", "#1E8449", "#A9CF54", "#48C9B0", "#2E5BDB", "#E74C3C", "#C8AD7F"]
YEARS = [1990, 2000, 2010, 2020, 2024]
SCEN = ["NDS", "RDS", "FPS", "PCS"]
SCEN_COLS = {"NDS": "#555555", "RDS": "#C0392B", "FPS": "#D4AC0D", "PCS": "#1E8449"}
SCEN_FULL = {"NDS": "Natural development", "RDS": "Rapid development", "FPS": "Farmland protection",
             "PCS": "Protection-coordinated"}
MODELS = ["FLUS", "PLUS"]
CLS5 = ["Cropland", "Forest", "ShrubGrass", "Wetland", "Water", "Impervious", "Bare"]      # names used in Step 5 tables
FEAT = ["dem", "slope", "dist_coast", "dist_river", "dist_road", "dist_urban", "pop", "bio1", "bio12"]
FEAT_LAB = ["Elevation", "Slope", "Dist. to coast", "Dist. to river", "Dist. to road", "Dist. to built-up", "Population", "Mean temperature", "Annual precipitation"]
GI_LS = ["1990", "2000", "2010", "2020", "2024", "2040_FLUS_NDS", "2040_FLUS_PCS", "2040_PLUS_NDS", "2040_PLUS_PCS"]
EXT = (18, 158, -16, 52)
DPI = 600


# ------------------------------------------------------------------ style: Times New Roman everywhere
def set_style():
    names = {f.name for f in font_manager.fontManager.ttflist}
    for p in glob.glob(f"{ROOT}/fonts/*.ttf") + glob.glob("/usr/share/fonts/truetype/msttcorefonts/*.ttf"):
        try:
            font_manager.fontManager.addfont(p)
        except Exception:
            pass
    names = {f.name for f in font_manager.fontManager.ttflist}
    serif = [n for n in ("Times New Roman", "Liberation Serif", "DejaVu Serif") if n in names]
    plt.rcParams.update({
        "font.family": "serif", "font.serif": serif, "mathtext.fontset": "stix",
        "font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8, "xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
        "legend.fontsize": 7.5, "text.color": "black", "axes.labelcolor": "black", "xtick.color": "black",
        "ytick.color": "black", "axes.edgecolor": "black", "axes.linewidth": 0.6, "xtick.major.width": 0.6,
        "ytick.major.width": 0.6, "xtick.major.size": 2.5, "ytick.major.size": 2.5, "legend.frameon": False,
        "savefig.dpi": DPI, "pdf.fonttype": 42, "ps.fonttype": 42, "axes.titleweight": "bold"})
    return serif[0]


def save(fig, name):
    os.makedirs(FIG, exist_ok=True)
    out = f"{FIG}/{name}"
    fig.savefig(out + ".png", dpi=DPI, bbox_inches="tight", pad_inches=0.04)
    fig.savefig(out + ".pdf", bbox_inches="tight", pad_inches=0.04)
    plt.close(fig)
    print("saved", out + ".png")
    return out + ".png"


def panel(ax, letter, x=-0.02, y=1.02, **kw):
    ax.text(x, y, f"({letter})", transform=ax.transAxes, fontsize=10, fontweight="bold", va="bottom", ha="right", **kw)


def clean(ax, left=True, bottom=True):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    if not left:
        ax.spines["left"].set_visible(False)
    if not bottom:
        ax.spines["bottom"].set_visible(False)


# ------------------------------------------------------------------ projection + hexagons (same as Figure 1)
R_E = 6371.0
LON0, LAT0, P1, P2 = 90.0, 15.0, 0.0, 40.0
_p1, _p2, _l0 = np.radians(P1), np.radians(P2), np.radians(LAT0)
_n = (np.sin(_p1) + np.sin(_p2)) / 2
_C = np.cos(_p1) ** 2 + 2 * _n * np.sin(_p1)
_rho0 = R_E * np.sqrt(_C - 2 * _n * np.sin(_l0)) / _n


def fwd(lon, lat):
    lon, lat = np.radians(np.asarray(lon, float)), np.radians(np.asarray(lat, float))
    th = _n * (lon - np.radians(LON0)); rho = R_E * np.sqrt(_C - 2 * _n * np.sin(lat)) / _n
    return rho * np.sin(th), _rho0 - rho * np.cos(th)


def frame_xy(ext=EXT):
    lo = np.r_[np.linspace(ext[0], ext[1], 200), np.full(200, ext[1]), np.linspace(ext[1], ext[0], 200), np.full(200, ext[0])]
    la = np.r_[np.full(200, ext[2]), np.linspace(ext[2], ext[3], 200), np.full(200, ext[3]), np.linspace(ext[3], ext[2], 200)]
    x, y = fwd(lo, la)
    return (x.min(), x.max()), (y.min(), y.max())


HEX_KM = 32.0
HEX_ANG = np.radians(np.arange(6) * 60 + 30)


def hexes(x, y, s=HEX_KM):
    q = (np.sqrt(3) / 3 * x - y / 3) / s; r = (2 / 3 * y) / s; c = -q - r
    rq, rr, rc = np.round(q), np.round(r), np.round(c)
    dq, dr, dc = np.abs(rq - q), np.abs(rr - r), np.abs(rc - c)
    fq = (dq > dr) & (dq > dc); fr = ~fq & (dr > dc)
    rq = np.where(fq, -rr - rc, rq); rr = np.where(fr, -rq - rc, rr)
    return rq.astype(int), rr.astype(int)


def hex_polys(q, r, s=HEX_KM):
    cx, cy = s * np.sqrt(3) * (q + r / 2), s * 1.5 * r
    return np.stack([np.c_[cx + s * np.cos(t), cy + s * np.sin(t)] for t in HEX_ANG], 1)


def hex_agg(df, value_cols, weight="study"):
    """Aggregate 3 km block table to hexagons: sums of value columns and of the weight."""
    x, y = fwd(df.lon.values, df.lat.values)
    q, r = hexes(x, y)
    g = df.assign(q=q, r=r).groupby(["q", "r"])
    out = g[list(value_cols) + [weight]].sum().reset_index()
    out["region"] = g.apply(lambda d: d.groupby("region")[weight].sum().idxmax(), include_groups=False).values
    return out


def basemap(ax, base, ext=EXT, grid=True, labels=True, fs=7.5):
    """White sea, natural grey shaded relief on land (clipped to coastlines), borders, graticule."""
    from matplotlib.path import Path as MPath
    from matplotlib.patches import PathPatch
    xs, ys = frame_xy(ext)
    ax.set_facecolor("white")
    if base is not None:
        e = base["extent"]; rel = base["relief"].astype(np.float32) / 255.0
        g = (0.50 + 0.50 * rel).clip(0, 1)
        im = ax.imshow(np.dstack([g, g, g]), extent=(e[0], e[1], e[2], e[3]), interpolation="bilinear", zorder=0)
        rings = [np.asarray(r_, float) for r_ in base["rings"] if len(r_) > 3]
        if rings:
            verts = np.concatenate(rings)
            codes = np.concatenate([[MPath.MOVETO] + [MPath.LINETO] * (len(r_) - 2) + [MPath.CLOSEPOLY] for r_ in rings])
            im.set_clip_path(PathPatch(MPath(verts, codes), transform=ax.transData))
        ax.add_collection(LineCollection(rings, colors="#7A7A7A", linewidths=0.3, zorder=1))
        if "borders" in getattr(base, "files", base):
            ax.add_collection(LineCollection(list(base["borders"]), colors="#9A9A9A", linewidths=0.25,
                                             linestyles=(0, (2, 1.5)), zorder=1))
    if grid:
        for lon in range(0, 181, 20):
            la = np.linspace(-40, 75, 200); gx, gy = fwd(np.full_like(la, lon), la)
            ax.plot(gx, gy, color="#C8C8C8", lw=0.3, zorder=1.5)
        for lat in (-20, 0, 20, 40, 60):
            lo = np.linspace(0, 190, 300); gx, gy = fwd(lo, np.full_like(lo, lat))
            ax.plot(gx, gy, color="#C8C8C8", lw=0.3, zorder=1.5)
    if labels:
        dx, dy = xs[1] - xs[0], ys[1] - ys[0]
        for lon in range(40, 161, 20):
            la = np.linspace(-40, 75, 4000); gx, gy = fwd(np.full_like(la, lon), la)
            k = np.argmin(np.abs(gy - ys[0]))
            if xs[0] + 0.03 * dx < gx[k] < xs[1] - 0.03 * dx:
                ax.text(gx[k], ys[0] - dy * 0.012, f"{lon}°E", ha="center", va="top", fontsize=fs)
        for lat in (-20, 0, 20, 40):
            lo = np.linspace(-20, 200, 6000); gx, gy = fwd(lo, np.full_like(lo, lat))
            k = np.argmin(np.abs(gx - xs[0]))
            if ys[0] + 0.04 * dy < gy[k] < ys[1] - 0.04 * dy:
                ax.text(xs[0] - dx * 0.006, gy[k], f"{abs(lat)}°{'S' if lat < 0 else ('N' if lat > 0 else '')}",
                        ha="right", va="center", rotation=90, fontsize=fs)
    ax.set_xlim(*xs); ax.set_ylim(*ys); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_color("#6E6E6E"); sp.set_linewidth(0.6)
    return xs, ys


def scalebar(ax, xs, ys, km=1000, fs=7):
    dx, dy = xs[1] - xs[0], ys[1] - ys[0]
    sx, sy = xs[0] + dx * 0.03, ys[0] + dy * 0.05
    for k in range(2):
        ax.add_patch(Rectangle((sx + k * km, sy), km, dy * 0.009, fc="black" if k == 0 else "white", ec="black", lw=0.4, zorder=9))
    import matplotlib.patheffects as pe
    halo = [pe.withStroke(linewidth=2, foreground="white")]
    ax.text(sx, sy + dy * 0.02, "0", ha="center", fontsize=fs, path_effects=halo, zorder=9)
    ax.text(sx + 2 * km, sy + dy * 0.02, f"{2 * km:,} km", ha="center", fontsize=fs, path_effects=halo, zorder=9)


def north(ax, xs, ys, fs=7.5):
    dx, dy = xs[1] - xs[0], ys[1] - ys[0]
    from matplotlib.patches import FancyArrow
    nx, ny = xs[1] - dx * 0.03, ys[1] - dy * 0.2
    ax.add_patch(FancyArrow(nx, ny, 0, dy * 0.08, width=dx * 0.004, head_width=dx * 0.016, head_length=dy * 0.04,
                            length_includes_head=True, fc="#222222", ec="#222222", zorder=9))
    ax.text(nx, ny + dy * 0.09, "N", ha="center", va="bottom", fontsize=fs, fontweight="bold", zorder=9)


def load_base():
    p = f"{FIG}/fig1_base.npz"
    return np.load(p, allow_pickle=True) if os.path.exists(p) else None


def halo(w=2.0):
    import matplotlib.patheffects as pe
    return [pe.withStroke(linewidth=w, foreground="white")]


# ------------------------------------------------------------------ Figure 2: methodological framework
def fig2():
    W, H = 7.2, 6.75
    fig = plt.figure(figsize=(W, H)); ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, W); ax.set_ylim(0, H); ax.axis("off")
    STAGE = [("1", "Data", "#1A4592", "#E8EEF8"), ("2", "LULC simulation", "#B9770E", "#FBF1E3"),
             ("3", "Assessment", "#1E8449", "#E7F4EC"), ("4", "Network & planning", "#7D3C98", "#F2EAF6")]
    bands = [(5.55, 1.15), (3.55, 1.78), (2.35, 1.0), (0.05, 2.08)]          # (y0, height) in inches
    X0, X1 = 0.52, W - 0.12

    def box(x, y, w, h, title, body, ec, tfs=8.2, bfs=7.4):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.0,rounding_size=0.05", fc="white", ec=ec, lw=0.9, zorder=2))
        ax.add_patch(Rectangle((x, y + h - 0.24), w, 0.24, fc=ec, ec="none", alpha=0.14, zorder=2))
        ax.text(x + w / 2, y + h - 0.12, title, ha="center", va="center", fontsize=tfs, fontweight="bold", zorder=3)
        ax.text(x + w / 2, (y + h - 0.24 + y) / 2, body, ha="center", va="center", fontsize=bfs, linespacing=1.18, zorder=3)

    def arrow(x0, y0, x1, y1, c="#333333", lw=0.9, ms=8):
        ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="-|>", mutation_scale=ms, lw=lw, color=c, zorder=4,
                                     shrinkA=0, shrinkB=0))

    def row(n, gap=0.1):
        w = (X1 - X0 - gap * (n - 1)) / n
        return w, [X0 + i * (w + gap) for i in range(n)]

    for (num, name, ec, fc), (y0, h) in zip(STAGE, bands):
        ax.add_patch(FancyBboxPatch((0.06, y0), W - 0.1, h, boxstyle="round,pad=0,rounding_size=0.08", fc=fc, ec="none", zorder=0))
        ax.add_patch(FancyBboxPatch((0.06, y0), 0.34, h, boxstyle="round,pad=0,rounding_size=0.08", fc=ec, ec="none", zorder=1))
        ax.text(0.23, y0 + h / 2, f"{num}  {name}", rotation=90, ha="center", va="center", color="white",
                fontsize=8.5, fontweight="bold", zorder=2)
    # stage 1
    y0, h = bands[0]; ec = STAGE[0][2]; bw, xs_ = row(4)
    items = [("Coastal zone", "50 km landward buffer of the\nGSHHG shoreline; 4 UN M49\nsub-regions; EASE-Grid 2.0, 300 m"),
             ("Historical LULC", "GLC_FCS30D 30 m, 7 classes;\ntemporal-consistency filter;\n1990, 2000, 2010, 2020"),
             ("LULC 2024", "Esri/IO 10 m change-only\nupdate with persistence rule;\narea-weighted fractions"),
             ("Driving factors (9)", "DEM, slope; distance to coast,\nrivers, roads, built-up; GHS-POP;\nCHELSA bio1, bio12 (+ WDPA)")]
    for i, (x, (t, b)) in enumerate(zip(xs_, items)):
        box(x, y0 + 0.1, bw, h - 0.2, t, b, ec)
        if i in (1, 2):
            arrow(x - 0.1, y0 + h / 2 - 0.08, x, y0 + h / 2 - 0.08)
    # stage 2
    y0, h = bands[1]; ec = STAGE[1][2]; hb = (h - 0.3) / 2
    yA, yB = y0 + 0.1 + hb + 0.1, y0 + 0.1
    cols = [(X0, 1.5), (X0 + 1.62, 1.72), (X0 + 3.46, 1.48), (X0 + 5.06, X1 - X0 - 5.06)]
    box(cols[0][0], yA, cols[0][1], hb, "Land demand", "Markov chain 2010 → 2022\nrescaled to 2024 → 2040\n($P^{16/12}$), net class flows", ec)
    box(cols[0][0], yB, cols[0][1], hb, "Four scenarios", "NDS natural development\nRDS rapid development\nFPS farmland protection\nPCS protection-coordinated", ec, bfs=7.1)
    box(cols[1][0], yA, cols[1][1], hb, "FLUS-type model", "multinomial-logit probability;\n3 × 3 neighbourhood, adaptive\ninertia, roulette wheel", ec)
    box(cols[1][0], yB, cols[1][1], hb, "PLUS-type model", "LEAS random forest (160 trees);\nCARS 7 × 7 neighbourhood,\npatch seeds", ec)
    box(cols[2][0], yA, cols[2][1], hb, "Validation", "hindcast 2010 → 2020;\nOA, Kappa, FoM at\n0.3, 1.5 and 3 km; AUC", ec)
    box(cols[2][0], yB, cols[2][1], hb, "Driver importance", "random-forest importance\nby class and region", ec)
    box(cols[3][0], yB, cols[3][1], h - 0.2, "2040 LULC", "2 models × 4 scenarios\n× 4 regions\n(base map 2024)\n\nNDS and PCS\npassed to stage 3", ec)
    for cx, cw in cols[:2]:
        arrow(cx + cw, yA + hb / 2 - 0.08, cx + cw + 0.12, yA + hb / 2 - 0.08)
        arrow(cx + cw, yB + hb / 2 - 0.08, cx + cw + 0.12, yB + hb / 2 - 0.08)
    arrow(cols[0][0] + cols[0][1] / 2, yA, cols[0][0] + cols[0][1] / 2, yB + hb)
    arrow(cols[2][0] + cols[2][1], y0 + h / 2, cols[3][0], y0 + h / 2)
    # stage 3
    y0, h = bands[2]; ec = STAGE[2][2]; bw, xs_ = row(4)
    items = [("Morphology (MSPA)", "forest + wetland foreground;\ncore, bridge, loop, edge ..."),
             ("Habitat quality", "InVEST, k = 0.5, z = 2.5; threats:\nimpervious, cropland, bare"),
             ("Resistance (1-10)", "0.5 LULC + 0.2 slope\n+ 0.3 accessibility"),
             ("Vulnerability", "EVI = 0.5 resistance + 0.5 (1 - HQ)\nthresholds 0.40 and 0.60")]
    for x, (t, b) in zip(xs_, items):
        box(x, y0 + 0.1, bw, h - 0.2, t, b, ec, bfs=7.2)
    # stage 4
    y0, h = bands[3]; ec = STAGE[3][2]; bw, xs_ = row(5)
    items = [("Sources", "MSPA core ≥ 50 km²;\nranked by dPC; number\nscaled to zone area"),
             ("Corridors", "least-cost paths\non the resistance\nsurface (1.5 km)"),
             ("Connectivity", "PC and dPC (intra,\nflux, connector);\n$I_{ij}$ ≥ 0.10; ECA"),
             ("Corridor density", "kernel density,\n20 km bandwidth;\nhotspots ≥ 0.75"),
             ("Ecological current", "circuit theory\n(3 km grid);\npinch points")]
    for i, (x, (t, b)) in enumerate(zip(xs_, items)):
        box(x, y0 + 0.9, bw, 1.08, t, b, ec, bfs=7.2)
        if i < 4:
            arrow(x + bw, y0 + 1.34, x + bw + 0.1, y0 + 1.34)
    box(X0, y0 + 0.1, X1 - X0, 0.66, "Planning synthesis",
        "PCS versus NDS gains in quality, resistance, vulnerability and connectivity  →  conservation sequence:\n"
        "(1) protect hub sources, (2) secure corridors, (3) restore bottlenecks, (4) extend the network to coastal wetlands",
        ec, bfs=7.2)
    arrow((X0 + X1) / 2, y0 + 0.9, (X0 + X1) / 2, y0 + 0.76)
    ax.text(X1, bands[2][0] + bands[2][1] + 0.02, "applied to 1990, 2000, 2010, 2020, 2024 and 2040 NDS / PCS (FLUS, PLUS)",
            ha="right", va="bottom", fontsize=7, style="italic")
    for (ya, _), (yb, hb_) in zip(bands[:-1], bands[1:]):
        arrow(1.6, ya, 1.6, yb + hb_, c="#222222", lw=1.2, ms=10)
    return save(fig, "Fig2_framework")

# ------------------------------------------------------------------ hotspots (zoom windows)
HOTSPOTS = [("Dubai-Sharjah", 55.38, 25.25), ("Karachi", 67.05, 24.88), ("Pearl River Delta", 113.75, 22.65),
            ("Shanghai", 121.45, 31.2), ("Mumbai", 72.9, 19.1), ("Bangkok", 100.55, 13.65),
            ("Ho Chi Minh City", 106.72, 10.78), ("Jakarta", 106.85, -6.2)]
ZOOM_N = 250                                   # 250 x 300 m = 75 km windows
MAPS = [str(y) for y in YEARS] + [f"{m}_{s}" for m in MODELS for s in SCEN]


def map_path(name, mp):
    if mp == "2024":
        return f"{ROOT}/step3/lc_{name}_2024.tif"
    if mp.isdigit():
        return f"{ROOT}/step2_filtered/lc_{name}_{mp}.tif"
    m, s = mp.split("_"); return f"{ROOT}/step5/sim_{name}_{m}_2040_{s}.tif"


def prep_lulc(strip=2000, f=10):
    """3 km block class counts for every map, 1990-2024 epoch transitions, and hotspot zoom windows."""
    import rasterio
    from rasterio.windows import Window
    from pyproj import Transformer
    os.makedirs(CACHE, exist_ok=True)
    trans, zooms, meta = [], {}, []
    for name in REGIONS:
        with rasterio.open(f"{ROOT}/step2/studymask_{name}.tif") as s:
            H, W = s.height, s.width; T = s.transform; crs = s.crs
        avail = np.array([os.path.exists(map_path(name, mp)) for mp in MAPS])
        hw = (W // f) * f
        S_rows, C_rows = [], []
        Mt = {(a, b): np.zeros(64, np.int64) for a, b in zip(YEARS[:-1], YEARS[1:])}
        srcs = {mp: rasterio.open(map_path(name, mp)) for mp, ok in zip(MAPS, avail) if ok}
        with rasterio.open(f"{ROOT}/step2/studymask_{name}.tif") as sm:
            for r0 in range(0, (H // f) * f, strip):
                hh = min(strip, (H // f) * f - r0); win = Window(0, r0, hw, hh)
                m = sm.read(1, window=win) > 0
                bl = lambda a: a.reshape(hh // f, f, hw // f, f).sum((1, 3), dtype=np.int16)
                S_rows.append(bl(m).astype(np.uint8))
                cnt = np.zeros((len(MAPS), 7, hh // f, hw // f), np.uint8); prev = None
                for i, mp in enumerate(MAPS):
                    if mp not in srcs:
                        continue
                    a = srcs[mp].read(1, window=win)
                    a = np.where(m & ((a < 1) | (a > 7)), 5, a); a = np.where(m, a, 0)
                    for k in range(1, 8):
                        cnt[i, k - 1] = bl(a == k)
                    if mp.isdigit():
                        if prev is not None:
                            y0 = YEARS[YEARS.index(int(mp)) - 1]
                            Mt[(y0, int(mp))] += np.bincount(prev[m].astype(np.int64) * 8 + a[m], minlength=64)
                        prev = a
                C_rows.append(cnt)
        for src in srcs.values():
            src.close()
        S = np.concatenate(S_rows, 0); C = np.concatenate(C_rows, 2)
        rr, cc = np.nonzero(S)
        x = T.c + (cc * f + f / 2) * T.a; y = T.f + (rr * f + f / 2) * T.e
        tr = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
        lon, lat = tr.transform(x, y)
        np.savez_compressed(f"{CACHE}/lulc_blocks_{name}.npz", lon=np.float32(lon), lat=np.float32(lat), study=S[rr, cc],
                            counts=C[:, :, rr, cc], maps=np.array(MAPS), avail=avail)
        for (a, b), v in Mt.items():
            M = v.reshape(8, 8)[1:, 1:] * 0.09
            for i in range(7):
                for j in range(7):
                    trans.append(dict(region=name, y0=a, y1=b, frm=CLS[i], to=CLS[j], km2=M[i, j]))
        print(name, len(rr), "blocks; maps available:", int(avail.sum()), "/", len(MAPS))
        # zoom windows centred on hotspots that fall in this region's grid
        inv = Transformer.from_crs("EPSG:4326", crs, always_xy=True)
        for hn, lo, la in HOTSPOTS:
            X, Y = inv.transform(lo, la)
            c0 = int((X - T.c) / T.a) - ZOOM_N // 2; r0 = int((Y - T.f) / T.e) - ZOOM_N // 2
            win = Window(c0, r0, ZOOM_N, ZOOM_N)
            with rasterio.open(f"{ROOT}/step2/studymask_{name}.tif") as sm:
                mk = sm.read(1, window=win, boundless=True, fill_value=0) > 0
            if mk.sum() < 2000 or (hn in zooms and zooms[hn]["mask"].sum() >= mk.sum()):
                continue
            stack = np.zeros((len(MAPS), ZOOM_N, ZOOM_N), np.uint8)
            for i, (mp, ok) in enumerate(zip(MAPS, avail)):
                if ok:
                    with rasterio.open(map_path(name, mp)) as s:
                        stack[i] = s.read(1, window=win, boundless=True, fill_value=0)
            stack[:, ~mk] = 0
            zooms[hn] = dict(stack=stack, mask=mk, region=name, lon=lo, lat=la, avail=avail)
    pd.DataFrame(trans).to_csv(f"{CACHE}/trans_hist.csv", index=False)
    np.savez_compressed(f"{CACHE}/zooms.npz", **{f"{k}|stack": v["stack"] for k, v in zooms.items()},
                        **{f"{k}|mask": v["mask"] for k, v in zooms.items()},
                        **{f"{k}|avail": v["avail"] for k, v in zooms.items()},
                        meta=json.dumps({k: dict(region=v["region"], lon=v["lon"], lat=v["lat"]) for k, v in zooms.items()}),
                        maps=np.array(MAPS))
    print("zoom windows:", list(zooms))


def load_blocks():
    out = []
    for name in REGIONS:
        z = np.load(f"{CACHE}/lulc_blocks_{name}.npz")
        out.append(dict(region=name, lon=z["lon"], lat=z["lat"], study=z["study"], counts=z["counts"],
                        maps=list(z["maps"]), avail=z["avail"]))
    return out


def load_zooms():
    z = np.load(f"{CACHE}/zooms.npz", allow_pickle=True)
    meta = json.loads(str(z["meta"]))
    return {k: dict(v, stack=z[f"{k}|stack"], mask=z[f"{k}|mask"], avail=z[f"{k}|avail"]) for k, v in meta.items()}, list(z["maps"])


# ------------------------------------------------------------------ Figure 3: coastal urbanisation hotspots
EPOCH_LAB = ["Before 1990", "1990–2000", "2000–2010", "2010–2020", "2020–2024"]
EPOCH_COLS = ["#BDBDBD", "#9575CD", "#26A69A", "#FDD835", "#FF5722"]
ZOOM_BG = {"land": "#383838", "water": "#1B3B5F", "out": "#F2F2F2"}
GAIN_EDGES = [0, 1, 5, 20, 50, 150, 1e9]
GAIN_LAB = ["<1", "1–5", "5–20", "20–50", "50–150", ">150"]
GAIN_COLS = ["#FEE6CE", "#FDD0A2", "#FDAE6B", "#F16913", "#D94801", "#7F2704"]


def epoch_rgb(stack, mask, maps):
    """RGB image: impervious coloured by the epoch in which it first appeared (and persists to 2024)."""
    iy = [maps.index(str(y)) for y in YEARS]
    imp = stack[iy] == 6
    ep = np.full(mask.shape, -1)
    final = imp[-1]
    for e in range(len(YEARS) - 1, -1, -1):          # earliest epoch from which impervious persists to 2024
        run = imp[e:].all(0)
        ep = np.where(final & run, e, ep)
    lc24 = stack[iy[-1]]
    hexc = lambda h: np.array([int(h[i:i + 2], 16) for i in (1, 3, 5)]) / 255
    rgb = np.empty(mask.shape + (3,)); rgb[:] = hexc(ZOOM_BG["out"])
    rgb[mask] = hexc(ZOOM_BG["land"])
    rgb[mask & np.isin(lc24, (4, 5))] = hexc(ZOOM_BG["water"])
    for e, c in enumerate(EPOCH_COLS):
        rgb[ep == e] = hexc(c)
    gain = ((ep >= 1).sum()) * 0.09
    return rgb, gain, (final.sum()) * 0.09


def fig3():
    from matplotlib.patches import ConnectionPatch
    B = load_blocks(); Z, zmaps = load_zooms(); base = load_base()
    i90, i24 = B[0]["maps"].index("1990"), B[0]["maps"].index("2024")
    df = pd.concat([pd.DataFrame(dict(region=b["region"], lon=b["lon"], lat=b["lat"], study=b["study"].astype(float),
                                      gain=(b["counts"][i24, 5].astype(float) - b["counts"][i90, 5]) * 0.09)) for b in B])
    hv = hex_agg(df, ["gain"])
    W_IN = 7.2; xs, ys = frame_xy(); dx, dy = xs[1] - xs[0], ys[1] - ys[0]
    mw = 7.0; mh = mw * dy / dx; bs = 1.66; gap = 0.42
    H_IN = bs + gap + mh + gap + bs + 0.66
    fig = plt.figure(figsize=(W_IN, H_IN))
    F = lambda x, y, w, h: [x / W_IN, y / H_IN, w / W_IN, h / H_IN]
    ax = fig.add_axes(F((W_IN - mw) / 2, 0.66 + bs + gap, mw, mh))
    basemap(ax, base, labels=False)
    cls = np.digitize(hv.gain.clip(0), GAIN_EDGES[1:-1])
    ax.add_collection(PolyCollection(hex_polys(hv.q.values, hv.r.values), facecolors=np.array(GAIN_COLS)[cls],
                                     edgecolors="#8C8C8C", linewidths=0.08, zorder=3))
    scalebar(ax, xs, ys); north(ax, xs, ys)
    ax.text(0.008, 0.985, "Impervious gain\n1990–2024 per hexagon", transform=ax.transAxes, va="top", fontsize=7.5,
            fontweight="bold", path_effects=halo(), zorder=9)
    names = [h[0] for h in HOTSPOTS if h[0] in Z]
    top = [n for n in names if n in ("Dubai-Sharjah", "Karachi", "Pearl River Delta", "Shanghai")]
    bot = [n for n in names if n not in top]
    for row, (lst, yb) in enumerate(((top, 0.66 + bs + gap + mh + gap), (bot, 0.66))):
        n = len(lst); xg = (W_IN - n * bs) / (n + 1) if n < 4 else (W_IN - n * bs) / (n - 1) * 0 + (W_IN - 4 * bs) / 3
        for i, hn in enumerate(lst):
            letter = "ABCDEFGH"[names.index(hn)]
            xb = i * (bs + xg) if n == 4 else xg + i * (bs + xg)
            axz = fig.add_axes(F(xb, yb, bs, bs))
            z = Z[hn]; rgb, gain, tot = epoch_rgb(z["stack"], z["mask"], zmaps)
            ext_km = ZOOM_N * 0.3
            axz.imshow(rgb, extent=(0, ext_km, 0, ext_km), interpolation="nearest")
            axz.set_xticks([]); axz.set_yticks([])
            for sp in axz.spines.values():
                sp.set_color("black"); sp.set_linewidth(0.7)
            axz.text(1.5, ext_km - 1.5, f"{letter}  {hn}", va="top", ha="left", fontsize=7.5, fontweight="bold",
                     bbox=dict(fc="white", ec="none", pad=1.2), zorder=5)
            axz.text(ext_km - 1.5, 1.5, f"+{gain:,.0f} km²", va="bottom", ha="right", fontsize=7.2, fontweight="bold",
                     bbox=dict(fc="white", ec="none", pad=1.0), zorder=5)
            axz.add_patch(Rectangle((0.6, 0.6), 23, 6.6, fc="black", ec="none", alpha=0.75, zorder=4))
            axz.plot([1.5, 21.5], [2.2, 2.2], color="white", lw=1.6, solid_capstyle="butt", zorder=5)
            axz.text(11.5, 3.0, "20 km", color="white", ha="center", va="bottom", fontsize=6.8, zorder=5)
            px, py = fwd(z["lon"], z["lat"])
            ax.plot(px, py, "o", ms=4.2, mfc="#D7191C", mec="black", mew=0.5, zorder=8)
            ax.text(px + dx * 0.008, py + (dy * 0.025 if row == 0 else -dy * 0.025), letter, fontsize=7.5,
                    fontweight="bold", ha="left", va="bottom" if row == 0 else "top", path_effects=halo(), zorder=8)
            con = ConnectionPatch(xyA=(px, py), coordsA=ax.transData, xyB=(0.5, 0.0 if row == 0 else 1.0),
                                  coordsB=axz.transAxes, color="#444444", lw=0.5, zorder=7)
            fig.add_artist(con)
    # legends: three rows, fixed spacing in inches
    axl = fig.add_axes(F(0.05, 0.0, W_IN - 0.1, 0.56)); axl.axis("off"); axl.set_xlim(0, W_IN - 0.1); axl.set_ylim(0, 0.56)
    rows_ = [(0.46, "Impervious surface by period of first appearance:", list(zip(EPOCH_COLS, EPOCH_LAB)), "sq", 0.9),
             (0.27, "Background in zoom windows:", [(ZOOM_BG[k], t) for k, t in (("land", "Other land"), ("water", "Water and wetland"), ("out", "Sea / outside 50 km zone"))], "sq", 1.25),
             (0.08, "Impervious gain per hexagon, 1990–2024 (km²):", list(zip(GAIN_COLS, GAIN_LAB)), "hex", 0.62)]
    for yy, title, items, kind, step in rows_:
        axl.text(0, yy, title, fontsize=7.5, va="center")
        for i, (c, t) in enumerate(items):
            x0 = 2.95 + i * step
            if kind == "sq":
                axl.add_patch(Rectangle((x0, yy - 0.055), 0.16, 0.11, fc=c, ec="black", lw=0.3))
            else:
                axl.add_patch(MPoly(np.c_[x0 + 0.08 + 0.07 * np.cos(HEX_ANG), yy + 0.07 * np.sin(HEX_ANG)], fc=c, ec="#7A7A7A", lw=0.3))
            axl.text(x0 + 0.21, yy, t, fontsize=7.3, va="center")
    return save(fig, "Fig3_urbanisation_hotspots")


# ------------------------------------------------------------------ Figure 4: composition 1990-2040 and alluvial flows
def comp_table():
    """Region x map class shares (%) from the block counts (2040 from PLUS; FLUS has identical quantities)."""
    rows = []
    for b in load_blocks():
        tot = b["counts"].sum((1, 2)).astype(float)
        for i, mp in enumerate(b["maps"]):
            if not b["avail"][i]:
                continue
            for k, c in enumerate(CLS):
                rows.append(dict(region=b["region"], map=mp, cls=c, km2=b["counts"][i, k].sum() * 0.09,
                                 pct=b["counts"][i, k].sum() / max(tot[i], 1) * 100))
    return pd.DataFrame(rows)


def alluvial(ax, T, years, min_label=4.0):
    """Coast-wide alluvial: nodes = class shares per year; ribbons = class flows between consecutive years."""
    from matplotlib.path import Path as MPath
    from matplotlib.patches import PathPatch
    gapf, wn = 0.012, 0.16
    tot = None; pos = {}
    for yi, y in enumerate(years):
        if yi == 0:
            v = T[T.y0 == years[0]].groupby("frm").km2.sum().reindex(CLS).fillna(0).values
        else:
            v = T[T.y1 == y].groupby("to").km2.sum().reindex(CLS).fillna(0).values
        tot = v.sum(); share = v / tot * (1 - gapf * 6)
        y0 = 1.0; pos[y] = {}
        for k in range(7):
            pos[y][k] = (y0 - share[k], y0); y0 -= share[k] + gapf
            ax.add_patch(Rectangle((yi - wn / 2, pos[y][k][0]), wn, share[k], fc=LULC_COLS[k], ec="white", lw=0.4, zorder=3))
            if share[k] * 100 >= min_label * (1 - gapf * 6):
                ax.text(yi, (pos[y][k][0] + pos[y][k][1]) / 2, f"{v[k] / tot * 100:.1f}", ha="center", va="center",
                        fontsize=6.6, color="white" if k in (1, 4, 5) else "black", zorder=4)
    for yi in range(len(years) - 1):
        a, b = years[yi], years[yi + 1]
        M = T[(T.y0 == a) & (T.y1 == b)].pivot_table(index="frm", columns="to", values="km2", aggfunc="sum").reindex(index=CLS, columns=CLS).fillna(0).values
        M = M / M.sum() * (1 - gapf * 6)
        outc = {k: pos[a][k][1] for k in range(7)}; inc = {k: pos[b][k][1] for k in range(7)}
        for i in range(7):
            for j in range(7):
                h = M[i, j]
                if h <= 0:
                    continue
                y0a, y0b = outc[i], inc[j]; outc[i] -= h; inc[j] -= h
                x0, x1 = yi + wn / 2, yi + 1 - wn / 2; xm = (x0 + x1) / 2
                verts = [(x0, y0a), (xm, y0a), (xm, y0b), (x1, y0b), (x1, y0b - h), (xm, y0b - h), (xm, y0a - h), (x0, y0a - h), (x0, y0a)]
                codes = [MPath.MOVETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4, MPath.LINETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4, MPath.CLOSEPOLY]
                ax.add_patch(PathPatch(MPath(verts, codes), fc=LULC_COLS[i], ec="none", alpha=0.22 if i == j else 0.75, zorder=2))
    ax.set_xlim(-0.35, len(years) - 0.65); ax.set_ylim(-0.01, 1.01)
    ax.set_xticks(range(len(years))); ax.set_xticklabels(years); ax.set_yticks([])
    clean(ax, left=False); ax.spines["bottom"].set_visible(False); ax.tick_params(axis="x", length=0)


def fig4():
    C = comp_table(); T = pd.read_csv(f"{CACHE}/trans_hist.csv")
    fig = plt.figure(figsize=(7.2, 8.4))
    cols = [str(y) for y in YEARS] + [f"PLUS_{s}" for s in SCEN]
    xl = [str(y) for y in YEARS] + SCEN
    xpos = np.r_[np.arange(5), np.arange(4) + 5.7]
    for i, name in enumerate(REGIONS):
        ax = fig.add_axes([0.07 + (i % 2) * 0.48, 0.775 - (i // 2) * 0.235, 0.41, 0.16])
        sub = C[C.region == name].pivot_table(index="map", columns="cls", values="pct").reindex(index=cols, columns=CLS)
        bottom = np.zeros(len(cols))
        for k, c in enumerate(CLS):
            v = sub[c].fillna(0).values
            ax.bar(xpos, v, bottom=bottom, width=0.78, color=LULC_COLS[k], ec="white", lw=0.3)
            bottom += v
        for x, v in zip(xpos, sub["Impervious"].values):
            if not np.isnan(v):
                ax.text(x, 101.5, f"{v:.1f}", ha="center", va="bottom", fontsize=6.6, color="#B03A2E", fontweight="bold")
        ax.axvline(5.0, color="black", lw=0.5, ls=(0, (2, 2)))
        ax.text(2.0, -17, "Observed", ha="center", va="top", fontsize=7.2, style="italic")
        ax.text(7.2, -17, "2040 scenarios", ha="center", va="top", fontsize=7.2, style="italic")
        ax.set_xticks(xpos); ax.set_xticklabels(xl, fontsize=7, rotation=0)
        ax.set_ylim(0, 100); ax.set_yticks([0, 25, 50, 75, 100]); clean(ax)
        ax.set_ylabel("Share of zone (%)" if i % 2 == 0 else "")
        ax.set_title(f"{REGIONS[name][0]}  {REGIONS[name][1]}", loc="left", fontsize=8.5, color=REG_COLS[name], pad=12)
        if i == 0:
            panel(ax, "a", x=-0.13, y=1.12)
    fig.text(0.07, 0.462, "Red numbers above bars: impervious share (%). 2040 class quantities are identical for the FLUS- and PLUS-type models.",
             fontsize=7.2)
    ax = fig.add_axes([0.07, 0.05, 0.6, 0.33])
    alluvial(ax, T.groupby(["y0", "y1", "frm", "to"]).km2.sum().reset_index(), YEARS)
    panel(ax, "b", x=-0.03, y=1.03)
    ax.set_title("Coast-wide class flows, 1990 → 2024 (node labels: % of zone)", loc="left", fontsize=8.5, pad=6)
    # (c) annual net change rates of key classes by region and period (tile heat map)
    axc = fig.add_axes([0.8, 0.075, 0.18, 0.305])
    key = ["Impervious", "Cropland", "Forest", "Wetland"]
    lab, vals = [], []
    for name in REGIONS:
        for k in key:
            r_ = []
            for a, b in zip(YEARS[:-1], YEARS[1:]):
                t = T[(T.region == name) & (T.y0 == a)]
                net = t[t.to == k].km2.sum() - t[t.frm == k].km2.sum()
                r_.append(net / (b - a))
            vals.append(r_); lab.append(k)
    V = np.array(vals)
    lim = np.nanpercentile(np.abs(V), 95)
    im = axc.imshow(V, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
    for (r, c), v in np.ndenumerate(V):
        axc.text(c, r, f"{v:,.0f}", ha="center", va="center", fontsize=5.8, color="white" if abs(v) > 0.6 * lim else "black")
    axc.set_yticks(range(len(lab))); axc.set_yticklabels(lab, fontsize=6.6)
    axc.set_xticks(range(4)); axc.set_xticklabels(["1990–\n2000", "2000–\n2010", "2010–\n2020", "2020–\n2024"], fontsize=6.4)
    for r in (4, 8, 12):
        axc.axhline(r - 0.5, color="black", lw=0.8)
    for gi, name in enumerate(REGIONS):
        axc.text(-0.62, gi * 4 + 1.5, REGIONS[name][2], transform=axc.get_yaxis_transform(), rotation=90, ha="center",
                 va="center", fontsize=7.5, fontweight="bold", color=REG_COLS[name])
    axc.tick_params(length=0)
    panel(axc, "c", x=-0.62, y=1.03)
    axc.set_title("Net change (km$^2$ yr$^{-1}$)", fontsize=8.5, pad=6)
    cb = fig.add_axes([0.8, 0.012, 0.18, 0.008]); plt.colorbar(im, cax=cb, orientation="horizontal")
    cb.tick_params(labelsize=6.5, length=2)
    # class key
    axk = fig.add_axes([0.07, 0.43, 0.9, 0.02]); axk.axis("off"); axk.set_xlim(0, 1); axk.set_ylim(0, 1)
    for k, c in enumerate(CLS):
        axk.add_patch(Rectangle((k * 0.142, 0.15), 0.022, 0.7, fc=LULC_COLS[k], ec="none"))
        axk.text(k * 0.142 + 0.03, 0.5, c, va="center", fontsize=7.5)
    return save(fig, "Fig4_LULC_composition_flows")


# ------------------------------------------------------------------ Figure 5: 2040 scenarios in hotspot windows + net change
def muted(hexs, f=0.62):
    c = np.array([[int(h[i:i + 2], 16) for i in (1, 3, 5)] for h in hexs]) / 255
    g = c.mean(1, keepdims=True)
    return (c * 0.35 + g * 0.65) * f + (1 - f)


def change_rgb(lc0, lc1, mask):
    full = np.array([[int(h[i:i + 2], 16) for i in (1, 3, 5)] for h in LULC_COLS]) / 255
    rgb = np.ones(lc0.shape + (3,))
    ok = mask & (lc0 >= 1) & (lc0 <= 7)
    rgb[ok] = (0.87, 0.87, 0.87)                                   # unchanged land: plain light grey
    rgb[ok & np.isin(lc0, (4, 5))] = (0.74, 0.82, 0.90)            # unchanged water / wetland: pale blue
    ch = ok & (lc1 != lc0) & (lc1 >= 1) & (lc1 <= 7)
    rgb[ch] = full[lc1[ch] - 1]
    return rgb, ch.sum() * 0.09


def fig5(hot=None):
    Z, zmaps = load_zooms()
    i24 = zmaps.index("2024")
    if hot is None:                                     # two windows with the most simulated change, different regions
        sc = []
        for hn, z in Z.items():
            j = zmaps.index("PLUS_NDS")
            sc.append((((z["stack"][j] != z["stack"][i24]) & z["mask"]).sum(), hn, z["region"]))
        sc.sort(reverse=True); hot, seen = [], set()
        for v, hn, rg in sc:
            if rg not in seen:
                hot.append(hn); seen.add(rg)
            if len(hot) == 2:
                break
    W_IN = 7.2; pw = 1.24; lw_ = 0.72
    H_IN = 0.3 + 4 * (pw + 0.08) + 0.5 + 2.35
    fig = plt.figure(figsize=(W_IN, H_IN))
    F = lambda x, y, w, h: [x / W_IN, y / H_IN, w / W_IN, h / H_IN]
    full = np.array([[int(h[i:i + 2], 16) for i in (1, 3, 5)] for h in LULC_COLS]) / 255
    r = 0
    for hn in hot:
        z = Z[hn]; st = z["stack"]; mk = z["mask"]; lc0 = st[i24]
        for m in MODELS:
            yb = H_IN - 0.3 - (r + 1) * (pw + 0.08)
            ax0 = fig.add_axes(F(lw_, yb, pw, pw))
            img = np.ones(lc0.shape + (3,)); ok = mk & (lc0 >= 1); img[ok] = full[lc0[ok] - 1]
            ax0.imshow(img, interpolation="nearest"); ax0.set_xticks([]); ax0.set_yticks([])
            if r == 0:
                ax0.set_title("2024 (observed)", fontsize=8)
            fig.text((lw_ - 0.2) / W_IN, (yb + pw / 2) / H_IN, f"{hn}\n{m}-type", rotation=90, ha="center", va="center",
                     fontsize=7.8, fontweight="bold")
            for j, sc_ in enumerate(SCEN):
                ax = fig.add_axes(F(lw_ + (j + 1) * (pw + 0.06), yb, pw, pw))
                lc1 = st[zmaps.index(f"{m}_{sc_}")]
                rgb, km2 = change_rgb(lc0, lc1, mk)
                ax.imshow(rgb, interpolation="nearest"); ax.set_xticks([]); ax.set_yticks([])
                ax.text(0.97, 0.03, f"{km2:,.0f} km²", transform=ax.transAxes, ha="right", va="bottom", fontsize=6.8,
                        fontweight="bold", bbox=dict(fc="white", ec="none", pad=0.8))
                if r == 0:
                    ax.set_title(f"2040 {sc_}", fontsize=8, color=SCEN_COLS[sc_])
                for sp in ax.spines.values():
                    sp.set_linewidth(0.5)
            if r == 0:
                ax0.plot([10, 76.7], [240, 240], color="black", lw=1.5); ax0.text(43, 236, "20 km", ha="center", va="bottom", fontsize=6.5)
            r += 1
    fig.text(lw_ / W_IN, (H_IN - 0.12) / H_IN, "(a)", fontsize=10, fontweight="bold", ha="right")
    fig.text((lw_ + pw + 0.06) / W_IN, (H_IN - 0.3 - 4 * (pw + 0.08) - 0.12) / H_IN,
             "2040 panels: unchanged land grey, unchanged water pale blue; cells changing class 2024 → 2040 in the colour of the new class;\n"
             "label = changed area in the 75 km window.", fontsize=7.2, va="top")
    axk = fig.add_axes(F(lw_, H_IN - 0.3 - 4 * (pw + 0.08) - 0.45, W_IN - lw_ - 0.1, 0.18)); axk.axis("off")
    axk.set_xlim(0, 1); axk.set_ylim(0, 1)
    for k, c in enumerate(CLS):
        axk.add_patch(Rectangle((k * 0.143, 0.1), 0.025, 0.8, fc=LULC_COLS[k], ec="none")); axk.text(k * 0.143 + 0.032, 0.5, c, va="center", fontsize=7.5)
    # (b) net change by class, scenario and region (km2)
    D = pd.read_csv(f"{ROOT}/step5/demand.csv"); D = D[D.run.astype(str) == "2040"]
    D["net"] = D.demand_km2 - D.start_km2
    for i, name in enumerate(REGIONS):
        ax = fig.add_axes(F(0.72 + i * 1.64, 0.35, 1.36, 1.55))
        sub = D[D.region == name]
        for k, c in enumerate(CLS5):
            ax.axhline(k, color="#EEEEEE", lw=0.6, zorder=0)
            v = sub[sub.cls == c].set_index("scenario").net.reindex(SCEN)
            ax.plot([v.min(), v.max()], [k, k], color="#9A9A9A", lw=1.0, zorder=1)
            for s_, off in zip(SCEN, (-0.18, -0.06, 0.06, 0.18)):
                ax.plot(v[s_], k + off, "o", ms=3.4, mfc=SCEN_COLS[s_], mec="white", mew=0.3, zorder=3)
        ax.axvline(0, color="black", lw=0.5)
        ax.set_ylim(6.6, -0.6); ax.set_yticks(range(7)); ax.set_yticklabels(CLS if i == 0 else [], fontsize=7.2)
        ax.xaxis.set_major_locator(mt.MaxNLocator(3, symmetric=True))
        ax.xaxis.set_major_formatter(mt.FuncFormatter(lambda v, _: (f"{v / 1000:+.0f}k" if abs(v) >= 1000 else f"{v:+.0f}" if v else "0").replace("-", "−")))
        ax.tick_params(axis="x", labelsize=6.8); clean(ax)
        ax.set_title(f"{REGIONS[name][0]}  {REGIONS[name][1]}", fontsize=8, color=REG_COLS[name])
        if i == 0:
            fig.text(0.05, 2.22 / H_IN, "(b)", fontsize=10, fontweight="bold")
    fig.text(0.5, 0.07 / H_IN, "Net change 2024 → 2040 (km²)", ha="center", fontsize=8)
    axs = fig.add_axes(F(0.72, 2.16, 6.4, 0.15)); axs.axis("off"); axs.set_xlim(0, 1); axs.set_ylim(0, 1)
    for j, s_ in enumerate(SCEN):
        axs.plot(0.05 + j * 0.24, 0.5, "o", ms=4.5, mfc=SCEN_COLS[s_], mec="white")
        axs.text(0.065 + j * 0.24, 0.5, f"{s_}  {SCEN_FULL[s_].lower()}", va="center", fontsize=7.4)
    return save(fig, "Fig5_scenarios_2040")


# ------------------------------------------------------------------ Figure 6: validation and driver importance
def fig6():
    A = pd.read_csv(f"{ROOT}/step5/accuracy.csv"); A = A[A.get("run", "hindcast").astype(str) == "hindcast"] if "run" in A else A
    I = pd.read_csv(f"{ROOT}/step5/leas_importance.csv")
    fig = plt.figure(figsize=(7.2, 6.4))
    res = [0.3, 1.5, 3.0]
    for j, (cls_, lab) in enumerate((("impervious", "Impervious"), ("cropland", "Cropland"))):
        ax = fig.add_axes([0.08 + j * 0.25, 0.6, 0.2, 0.33])
        for name in REGIONS:
            for m, ls, mk in (("FLUS", (0, (3, 2)), "s"), ("PLUS", "-", "o")):
                r_ = A[(A.region == name) & (A.model == m)]
                if not len(r_):
                    continue
                v = [r_[f"FoM_{cls_}"].iloc[0], r_[f"FoM_{cls_}_1.5km"].iloc[0], r_[f"FoM_{cls_}_3km"].iloc[0]]
                ax.plot(res, v, ls=ls, marker=mk, ms=3.2, lw=1.0, color=REG_COLS[name], mfc=REG_COLS[name] if m == "PLUS" else "white")
        ax.set_xticks(res); ax.set_xticklabels(["0.3", "1.5", "3"]); ax.set_xlabel("Assessment resolution (km)")
        ax.set_ylabel("Figure of merit" if j == 0 else ""); ax.set_title(lab, fontsize=8.5); clean(ax)
        ax.set_ylim(0, None); ax.grid(axis="y", color="#EEEEEE", lw=0.5); ax.set_axisbelow(True)
        if j == 0:
            panel(ax, "a", x=-0.2, y=1.04)
    axl = fig.add_axes([0.08, 0.5, 0.52, 0.04]); axl.axis("off"); axl.set_xlim(0, 1); axl.set_ylim(0, 1)
    for i, name in enumerate(REGIONS):
        axl.plot([i * 0.27, i * 0.27 + 0.05], [0.75, 0.75], color=REG_COLS[name], lw=1.4)
        axl.text(i * 0.27 + 0.065, 0.75, REGIONS[name][1], va="center", fontsize=7.2)
    axl.plot([0.0, 0.05], [0.1, 0.1], color="black", lw=1, ls=(0, (3, 2)), marker="s", ms=3, mfc="white"); axl.text(0.065, 0.1, "FLUS-type", va="center", fontsize=7.2)
    axl.plot([0.25, 0.30], [0.1, 0.1], color="black", lw=1, marker="o", ms=3); axl.text(0.315, 0.1, "PLUS-type", va="center", fontsize=7.2)
    # (b) accuracy table as heat tiles
    ax = fig.add_axes([0.66, 0.53, 0.32, 0.4])
    mets = [("OA", "OA"), ("Kappa", "Kappa"), ("FoM", "FoM"), ("AUC_impervious", "AUC\nimperv."), ("AUC_cropland", "AUC\ncropland")]
    rows_ = [(n, m) for n in REGIONS for m in MODELS]
    V = np.array([[A[(A.region == n) & (A.model == m)][c].iloc[0] if len(A[(A.region == n) & (A.model == m)]) and c in A else np.nan
                   for c, _ in mets] for n, m in rows_])
    Vn = (V - np.nanmin(V, 0)) / np.maximum(np.nanmax(V, 0) - np.nanmin(V, 0), 1e-9)
    ax.imshow(Vn, cmap="Greens", vmin=-0.3, vmax=1.3, aspect="auto")
    for (r, c), v in np.ndenumerate(V):
        ax.text(c, r, "–" if np.isnan(v) else f"{v:.3f}", ha="center", va="center", fontsize=6.8)
    ax.set_xticks(range(len(mets))); ax.set_xticklabels([l for _, l in mets], fontsize=7, rotation=35, ha="right")
    ax.set_yticks(range(len(rows_))); ax.set_yticklabels([f"{REGIONS[n][2]}  {m}" for n, m in rows_], fontsize=7)
    ax.xaxis.tick_top(); plt.setp(ax.get_xticklabels(), rotation=0, ha="center")
    for r in (2, 4, 6):
        ax.axhline(r - 0.5, color="white", lw=1.5)
    ax.tick_params(length=0)
    panel(ax, "b", x=-0.3, y=1.1)
    # (c) driver importance for impervious expansion (heat map) and (d) mean over classes (dot matrix)
    Ir = I[I.run.astype(str) == "2040"] if (I.run.astype(str) == "2040").any() else I
    piv = Ir[Ir.cls == "Impervious"].pivot_table(index="feature", columns="region", values="importance").reindex(index=FEAT, columns=list(REGIONS))
    ax = fig.add_axes([0.19, 0.08, 0.3, 0.34])
    im = ax.imshow(piv.values * 100, cmap="YlOrBr", aspect="auto", vmin=0)
    for (r, c), v in np.ndenumerate(piv.values * 100):
        ax.text(c, r, f"{v:.1f}", ha="center", va="center", fontsize=6.8, color="white" if v > np.nanmax(piv.values) * 70 else "black")
    ax.set_yticks(range(len(FEAT))); ax.set_yticklabels(FEAT_LAB, fontsize=7.2)
    ax.set_xticks(range(4)); ax.set_xticklabels([REGIONS[n][2] for n in REGIONS], fontsize=7.2); ax.tick_params(length=0)
    ax.set_title("Impervious expansion", fontsize=8.5)
    cb = fig.add_axes([0.19, 0.035, 0.3, 0.01]); plt.colorbar(im, cax=cb, orientation="horizontal"); cb.tick_params(labelsize=6.5, length=2)
    cb.set_xlabel("Importance (%)", fontsize=7, labelpad=1)
    panel(ax, "c", x=-0.48, y=1.03)
    ax = fig.add_axes([0.53, 0.08, 0.45, 0.34])
    M = Ir.groupby(["feature", "cls"]).importance.mean().unstack().reindex(index=FEAT, columns=CLS5) * 100
    xx, yy = np.meshgrid(range(M.shape[1]), range(M.shape[0]))
    ax.scatter(xx.ravel(), yy.ravel(), s=(M.values.ravel() ** 1.3) * 3.2, c=M.values.ravel(), cmap="YlOrBr", vmin=0,
               vmax=np.nanmax(M.values), edgecolors="#555555", linewidths=0.3)
    ax.set_xticks(range(7)); ax.set_xticklabels(CLS, rotation=35, ha="right", fontsize=7.2)
    ax.set_yticks(range(len(FEAT))); ax.set_yticklabels([]); ax.set_ylim(len(FEAT) - 0.5, -0.5); ax.set_xlim(-0.6, 6.6)
    ax.grid(color="#EEEEEE", lw=0.5); ax.set_axisbelow(True); clean(ax, left=False)
    ax.tick_params(axis="y", length=0)
    ax.set_title("All classes (mean of four regions)", fontsize=8.5)
    for v in (5, 10, 20):
        ax.scatter([], [], s=(v ** 1.3) * 3.2, c="#F0B060", edgecolors="#555555", linewidths=0.3, label=f"{v}%")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=3, fontsize=7, handletextpad=0.2, columnspacing=1.2)
    panel(ax, "d", x=0.02, y=1.03)
    return save(fig, "Fig6_validation_drivers")


# ------------------------------------------------------------------ Step 6 caches
NET_LS = ["2024", "2040_PLUS_NDS", "2040_PLUS_PCS", "2040_FLUS_NDS", "2040_FLUS_PCS"]
EASE_R, EASE_C30 = 6371007.181, np.cos(np.radians(30.0))


def ease_lonlat(x, y):
    """EASE-Grid 2.0 global (EPSG:6933) metres -> lon/lat (exact inverse of the cylindrical equal-area)."""
    return np.degrees(x / (EASE_R * EASE_C30)), np.degrees(np.arcsin(np.clip(y * EASE_C30 / EASE_R, -1, 1)))


def pool(a, f, how):
    h, w = (a.shape[0] // f) * f, (a.shape[1] // f) * f
    b = a[:h, :w].reshape(h // f, f, w // f, f)
    if how == "max":
        return b.max((1, 3))
    with np.errstate(invalid="ignore"):
        return np.nanmean(b, (1, 3))


def prep6(strip=2000, f=10):
    """3 km block sums of resistance, habitat quality, MSPA core / foreground and high vulnerability for every
    landscape; pooled source / corridor / density grids and 3 km current for the network landscapes."""
    import rasterio
    from rasterio.windows import Window
    from scipy import ndimage as ndi
    os.makedirs(CACHE, exist_ok=True)
    VARS = ["n", "res", "hq", "hq_n", "hq_high", "fg", "core", "vul_high"]
    extra = []
    for name in REGIONS:
        d6 = f"{ROOT}/step6/{name}"
        lss = [ls for ls in GI_LS if os.path.exists(f"{d6}/res_{ls}.tif")]
        with rasterio.open(f"{ROOT}/step2/studymask_{name}.tif") as sm:
            H, W = sm.height, sm.width; T = sm.transform
            hw = (W // f) * f
            S_rows, V_rows = [], []
            src = {(ls, v): rasterio.open(f"{d6}/{v}_{ls}.tif") for ls in lss for v in ("res", "hq", "mspa", "vul")}
            for r0 in range(0, (H // f) * f, strip):
                hh = min(strip, (H // f) * f - r0); win = Window(0, r0, hw, hh)
                m = sm.read(1, window=win) > 0
                bl = lambda a: a.reshape(hh // f, f, hw // f, f).sum((1, 3), dtype=np.float64).astype(np.float32)
                S_rows.append(bl(m.astype(np.float32)))
                V = np.zeros((len(lss), len(VARS), hh // f, hw // f), np.float32)
                for i, ls in enumerate(lss):
                    res = src[(ls, "res")].read(1, window=win).astype(np.float32)
                    hq = src[(ls, "hq")].read(1, window=win)
                    ms = src[(ls, "mspa")].read(1, window=win)
                    vu = src[(ls, "vul")].read(1, window=win)
                    ok = m & (res > 0); okq = m & (hq != 255)
                    V[i, 0] = bl(ok.astype(np.float32)); V[i, 1] = bl(np.where(ok, res / 100, 0))
                    V[i, 2] = bl(np.where(okq, hq / 100, 0)); V[i, 3] = bl(okq.astype(np.float32))
                    V[i, 4] = bl((okq & (hq >= 70)).astype(np.float32))
                    V[i, 5] = bl((m & (ms > 0)).astype(np.float32)); V[i, 6] = bl((m & (ms == 1)).astype(np.float32))
                    V[i, 7] = bl((m & (vu == 3)).astype(np.float32))
                V_rows.append(V)
            for v in src.values():
                v.close()
        S = np.concatenate(S_rows, 0); V = np.concatenate(V_rows, 2)
        rr, cc = np.nonzero(S)
        x = T.c + (cc * f + f / 2) * T.a; y = T.f + (rr * f + f / 2) * T.e
        lon, lat = ease_lonlat(x, y)
        np.savez_compressed(f"{CACHE}/gi_blocks_{name}.npz", lon=np.float32(lon), lat=np.float32(lat),
                            study=S[rr, cc], vals=V[:, :, rr, cc], ls=np.array(lss), vars=np.array(VARS))
        for i, ls in enumerate(lss):
            tot = V[i].sum((1, 2))
            extra.append(dict(region=name, landscape=ls, hq_high_pct=tot[4] / max(tot[3], 1) * 100))
        # network layers for the map figures
        net = {}
        for ls in [l for l in NET_LS if os.path.exists(f"{d6}/src_{l}.tif")]:
            with rasterio.open(f"{d6}/src_{ls}.tif") as s_:
                Sn = s_.read(1); Tn = s_.transform
            with rasterio.open(f"{d6}/corr_{ls}.tif") as s_:
                Cr = s_.read(1)
            with rasterio.open(f"{d6}/kde_{ls}.tif") as s_:
                Kd = s_.read(1).astype(np.float32); Kd[Kd < 0] = np.nan
            pf = max(1, int(np.ceil(max(Sn.shape) / 1400)))
            ids = np.arange(1, int(Sn.max()) + 1)
            cen = ndi.center_of_mass(Sn > 0, Sn, ids) if len(ids) else []
            cy = np.array([c[0] for c in cen]); cx = np.array([c[1] for c in cen])
            X = Tn.c + (cx + 0.5) * Tn.a; Y = Tn.f + (cy + 0.5) * Tn.e
            clo, cla = ease_lonlat(X, Y)
            net[f"{ls}|src"] = pool(Sn, pf, "max").astype(np.int16)
            net[f"{ls}|corr"] = (pool(Cr, pf, "max") > 0)
            net[f"{ls}|kde"] = pool(Kd, pf, "mean").astype(np.float16)
            net[f"{ls}|cen"] = np.c_[X, Y, clo, cla].astype(np.float64)
            net[f"{ls}|tf"] = np.array([Tn.c, Tn.f, Tn.a * pf, Tn.e * pf])
            cp = f"{d6}/cur_{ls}.tif"
            if os.path.exists(cp):
                with rasterio.open(cp) as s_:
                    cu = s_.read(1).astype(np.float32); Tc = s_.transform
                cu[cu < 0] = np.nan
                net[f"{ls}|cur"] = cu.astype(np.float16); net[f"{ls}|curtf"] = np.array([Tc.c, Tc.f, Tc.a, Tc.e])
        with rasterio.open(f"{ROOT}/step2/studymask_{name}.tif") as sm:
            pf = max(1, int(np.ceil(max(sm.height, sm.width) / 1400)))
            mk = sm.read(1, out_shape=(sm.height // pf, sm.width // pf)) > 0
            net["mask"] = mk; net["masktf"] = np.array([T.c, T.f, T.a * sm.width / mk.shape[1], T.e * sm.height / mk.shape[0]])
        np.savez_compressed(f"{CACHE}/net_{name}.npz", **net)
        print(name, "GI blocks", len(rr), "landscapes", lss)
    pd.DataFrame(extra).to_csv(f"{CACHE}/gi_extra.csv", index=False)


def load_gi():
    out = {}
    for name in REGIONS:
        z = np.load(f"{CACHE}/gi_blocks_{name}.npz")
        out[name] = dict(lon=z["lon"], lat=z["lat"], study=z["study"], vals=z["vals"], ls=list(z["ls"]), vars=list(z["vars"]))
    return out


def gi_hex(G, ls, var, how="mean"):
    """Hexagon table of a GI variable for one landscape: mean = sum(var)/sum(count)."""
    den = {"res": "n", "hq": "hq_n", "hq_high": "hq_n", "core": "fg", "fg": "n", "vul_high": "n"}[var]
    rows = []
    for name, g in G.items():
        if ls not in g["ls"]:
            continue
        i = g["ls"].index(ls)
        rows.append(pd.DataFrame(dict(region=name, lon=g["lon"], lat=g["lat"], study=g["study"],
                                      num=g["vals"][i, g["vars"].index(var)], den=g["vals"][i, g["vars"].index(den)])))
    h = hex_agg(pd.concat(rows), ["num", "den"])
    h["val"] = h.num / h.den.replace(0, np.nan)
    return h


def load_summary():
    S = pd.read_csv(f"{ROOT}/step6/summary.csv")
    if os.path.exists(f"{CACHE}/gi_extra.csv"):
        S = S.merge(pd.read_csv(f"{CACHE}/gi_extra.csv"), on=["region", "landscape"], how="left")
    S["landscape"] = S.landscape.astype(str)
    return S


def hexmap(ax, base, h, col, cmap, norm, title=None, letter=None, labels=False, fs=7):
    xs, ys = basemap(ax, base, labels=labels, fs=fs)
    ok = h[col].notna().values
    pc = PolyCollection(hex_polys(h.q.values[ok], h.r.values[ok]), array=h[col].values[ok], cmap=cmap, norm=norm,
                        edgecolors="none", linewidths=0, zorder=3)
    ax.add_collection(pc)
    if title:
        ax.text(0.012, 0.975, title, transform=ax.transAxes, va="top", ha="left", fontsize=7.8, fontweight="bold",
                path_effects=halo(), zorder=9)
    if letter:
        panel(ax, letter, x=-0.005, y=1.01)
    return pc, xs, ys


def hbar(fig, rect, pc, label, ticks=None, fmt=None):
    cax = fig.add_axes(rect); cb = plt.colorbar(pc, cax=cax, orientation="horizontal", ticks=ticks, format=fmt)
    cb.ax.tick_params(labelsize=7, length=2); cb.outline.set_linewidth(0.4)
    cb.set_label(label, fontsize=7.5, labelpad=2)
    return cb


# ------------------------------------------------------------------ Figure 7: landscape resistance
def fig7():
    from matplotlib.colors import Normalize, TwoSlopeNorm, BoundaryNorm
    G = load_gi(); base = load_base()
    xs, ys = frame_xy(); asp = (ys[1] - ys[0]) / (xs[1] - xs[0])
    W_IN = 7.2; mw = 7.0; mh = mw * asp; sw = 3.47; sh = sw * asp
    H_IN = 0.15 + mh + 0.82 + 2 * sh + 0.12 + 0.62 + 0.3 + 1.25 + 0.4
    fig = plt.figure(figsize=(W_IN, H_IN)); F = lambda x, y, w, h: [x / W_IN, y / H_IN, w / W_IN, h / H_IN]
    y_top = H_IN - 0.15 - mh
    ax = fig.add_axes(F(0.1, y_top, mw, mh))
    h24 = gi_hex(G, "2024", "res")
    bnds = [1, 1.5, 2, 3, 4, 5, 6, 7, 8, 10]
    pc, _, _ = hexmap(ax, base, h24, "val", plt.get_cmap("YlOrRd", len(bnds) - 1), BoundaryNorm(bnds, len(bnds) - 1),
                      "(a)  Mean resistance, 2024", None, labels=True)
    scalebar(ax, xs, ys); north(ax, xs, ys)
    hbar(fig, F(0.1 + mw * 0.3, y_top - 0.42, mw * 0.4, 0.07), pc, "Resistance index (1–10)", ticks=bnds)
    lim = 1.5; dn = TwoSlopeNorm(vcenter=0, vmin=-lim, vmax=lim)
    y_small = y_top - 0.82
    for k, (m, sc_) in enumerate((("FLUS", "NDS"), ("FLUS", "PCS"), ("PLUS", "NDS"), ("PLUS", "PCS"))):
        h = gi_hex(G, f"2040_{m}_{sc_}", "res")
        h = h.merge(h24[["q", "r", "val"]].rename(columns={"val": "v24"}), on=["q", "r"], how="left"); h["d"] = h.val - h.v24
        ax2 = fig.add_axes(F(0.1 + (k % 2) * (sw + 0.06), y_small - (k // 2 + 1) * sh - (k // 2) * 0.12, sw, sh))
        pcd, _, _ = hexmap(ax2, base, h, "d", "RdBu_r", dn, f"({'bcde'[k]})  {m}-type, 2040 {sc_} − 2024")
    y_cb = y_small - 2 * sh - 0.12 - 0.3
    hbar(fig, F(0.1 + mw * 0.3, y_cb, mw * 0.4, 0.07), pcd, "Change in mean resistance (2040 − 2024)",
         ticks=[-1.5, -1, -0.5, 0, 0.5, 1, 1.5])
    # (f) distributions of block-level resistance by region (2024 vs 2040 NDS / PCS, PLUS)
    axf = fig.add_axes(F(0.62, 0.4, W_IN - 0.75, 1.25))
    cases = [("2024", "#7F7F7F", "2024"), ("2040_PLUS_NDS", SCEN_COLS["NDS"], "2040 NDS"), ("2040_PLUS_PCS", SCEN_COLS["PCS"], "2040 PCS")]
    for i, name in enumerate(REGIONS):
        g = G[name]
        for j, (ls, c, lab) in enumerate(cases):
            if ls not in g["ls"]:
                continue
            k_ = g["ls"].index(ls); n_ = g["vals"][k_, 0]; v = g["vals"][k_, 1] / np.maximum(n_, 1)
            v = v[n_ > 20]; xpos = i * 4 + j
            vp = axf.violinplot([v], positions=[xpos], widths=0.85, showextrema=False)
            for b_ in vp["bodies"]:
                b_.set_facecolor(c); b_.set_edgecolor("black"); b_.set_linewidth(0.3); b_.set_alpha(0.85)
            q1, md, q3 = np.percentile(v, [25, 50, 75])
            axf.plot([xpos, xpos], [q1, q3], color="black", lw=1.4); axf.plot(xpos, md, "o", ms=2.6, mfc="white", mec="black", mew=0.5)
        axf.text(i * 4 + 1, 10.6, f"{REGIONS[name][0]}  {REGIONS[name][1]}", ha="center", fontsize=7.8, fontweight="bold", color=REG_COLS[name])
    axf.set_xticks([]); axf.set_ylim(0.8, 10.2); axf.set_ylabel("Block resistance (3 km)"); clean(axf, bottom=False)
    axf.set_xlim(-0.7, 14.7)
    for j, (_, c, lab) in enumerate(cases):
        axf.add_patch(Rectangle((0.0 + j * 3.2, -1.25), 0.35, 0.55, fc=c, ec="black", lw=0.3, clip_on=False, transform=axf.transData))
        axf.text(0.5 + j * 3.2, -0.97, lab + (" (PLUS-type)" if j else ""), fontsize=7.2, va="center", clip_on=False)
    panel(axf, "f", x=-0.06, y=1.06)
    return save(fig, "Fig7_resistance")


# ------------------------------------------------------------------ Figure 8: sources, corridors, corridor density
DPC_CLS = ["Hub (top 10%)", "Important (10–30%)", "Moderate (30–60%)", "General (>60%)"]
DPC_COLS = ["#7B1E1E", "#D35400", "#F5B041", "#F9E79F"]


def dpc_class(nodes):
    r = nodes.dPC.rank(ascending=False, pct=True).values
    return np.digitize(r, [0.1, 0.3, 0.6], right=True)


def region_window(net, pad=0.03):
    mk = net["mask"]; tf = net["masktf"]
    rr, cc = np.nonzero(mk)
    x0, x1 = tf[0] + cc.min() * tf[2], tf[0] + (cc.max() + 1) * tf[2]
    y1, y0 = tf[1] + rr.min() * tf[3], tf[1] + (rr.max() + 1) * tf[3]
    dx, dy = x1 - x0, y1 - y0
    return (x0 - pad * dx, x1 + pad * dx, y0 - pad * dy, y1 + pad * dy)


def ext_of(a, tf):
    return (tf[0], tf[0] + a.shape[1] * tf[2], tf[1] + a.shape[0] * tf[3], tf[1])


def fig8(model="PLUS"):
    from matplotlib.colors import ListedColormap
    cols_ls = ["2024", f"2040_{model}_NDS", f"2040_{model}_PCS"]
    col_lab = ["2024", "2040 NDS", "2040 PCS"]
    W_IN = 7.2; cw = 2.25; lw_ = 0.42
    nets = {n: dict(np.load(f"{CACHE}/net_{n}.npz")) for n in REGIONS}
    wins = {n: region_window(nets[n]) for n in REGIONS}
    rh = {n: cw * (w[3] - w[2]) / (w[1] - w[0]) for n, w in wins.items()}
    rh = {n: min(max(v, 1.0), 2.3) for n, v in rh.items()}
    H_IN = 0.3 + sum(rh.values()) + 0.1 * 4 + 0.9
    fig = plt.figure(figsize=(W_IN, H_IN)); F = lambda x, y, w, h: [x / W_IN, y / H_IN, w / W_IN, h / H_IN]
    kcm = plt.get_cmap("BuPu"); kcm = ListedColormap(kcm(np.linspace(0.08, 0.85, 256)))
    y = H_IN - 0.3
    for name in REGIONS:
        net = nets[name]; w = wins[name]; h_ = rh[name]; y -= h_
        fig.text((lw_ - 0.16) / W_IN, (y + h_ / 2) / H_IN, f"{REGIONS[name][0]}  {REGIONS[name][1]}", rotation=90, ha="center",
                 va="center", fontsize=8, fontweight="bold", color=REG_COLS[name])
        for j, ls in enumerate(cols_ls):
            ax = fig.add_axes(F(lw_ + j * (cw + 0.04), y, cw, h_))
            ax.set_facecolor("white")
            mk = net["mask"]
            ax.imshow(np.where(mk, 0.88, np.nan), cmap="gray", vmin=0, vmax=1, extent=ext_of(mk, net["masktf"]), interpolation="nearest", zorder=0)
            if f"{ls}|kde" in net:
                kd = net[f"{ls}|kde"].astype(np.float32); kd[kd < 0.02] = np.nan
                ax.imshow(kd, cmap=kcm, vmin=0, vmax=1, extent=ext_of(kd, net[f"{ls}|tf"]), interpolation="bilinear", zorder=1, alpha=0.9)
                from scipy.ndimage import binary_dilation, minimum_filter
                cr = binary_dilation(net[f"{ls}|corr"], iterations=1)
                ax.imshow(np.where(cr, 1.0, np.nan), cmap=ListedColormap(["#111111"]), extent=ext_of(cr, net[f"{ls}|tf"]),
                          interpolation="nearest", zorder=3)
                src = net[f"{ls}|src"]; npth = f"{ROOT}/step6/{name}/nodes_{ls}.csv"
                if os.path.exists(npth):
                    nd = pd.read_csv(npth); cl = np.full(int(src.max()) + 1, -1); cl[nd.source.values] = dpc_class(nd)
                    sc = np.where(src > 0, cl[np.clip(src, 0, len(cl) - 1)], 9).astype(float); sc[sc < 0] = 9
                    sc = minimum_filter(sc, size=5); sc[sc > 3] = np.nan          # enlarge patches for legibility
                    ax.imshow(sc, cmap=ListedColormap(DPC_COLS), vmin=-0.5, vmax=3.5, extent=ext_of(src, net[f"{ls}|tf"]),
                              interpolation="nearest", zorder=2)
                    ax.text(0.98, 0.03, f"{len(nd)} sources", transform=ax.transAxes, ha="right", va="bottom", fontsize=6.8,
                            bbox=dict(fc="white", ec="none", pad=0.8), zorder=5)
            ax.set_xlim(w[0], w[1]); ax.set_ylim(w[2], w[3]); ax.set_aspect("auto")
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_linewidth(0.5); sp.set_color("#555555")
            if name == "WesternAsia":
                ax.set_title(col_lab[j], fontsize=8.5)
        y -= 0.1
    # legend
    axl = fig.add_axes(F(lw_, 0.25, W_IN - lw_ - 0.1, 0.5)); axl.axis("off"); axl.set_xlim(0, W_IN - lw_ - 0.1); axl.set_ylim(0, 0.5)
    axl.text(0, 0.38, "Sources by dPC rank:", fontsize=7.5, va="center")
    for i, (c, t) in enumerate(zip(DPC_COLS, DPC_CLS)):
        axl.add_patch(Rectangle((1.3 + i * 1.3, 0.32), 0.15, 0.12, fc=c, ec="black", lw=0.3)); axl.text(1.5 + i * 1.3, 0.38, t, fontsize=7.2, va="center")
    axl.text(0, 0.1, "Least-cost corridor:", fontsize=7.5, va="center")
    axl.plot([1.3, 1.55], [0.1, 0.1], color="#111111", lw=1.6)
    axl.text(2.1, 0.1, "Corridor density (20 km kernel):", fontsize=7.5, va="center")
    for i in range(6):
        axl.add_patch(Rectangle((4.05 + i * 0.22, 0.04), 0.22, 0.12, fc=kcm(i / 5), ec="none"))
    axl.text(4.05, 0.0, "low", fontsize=6.8, va="top"); axl.text(5.37, 0.0, "high", fontsize=6.8, va="top", ha="right")
    axl.add_patch(Rectangle((5.6, 0.04), 0.22, 0.12, fc="#E0E0E0", ec="black", lw=0.3)); axl.text(5.87, 0.1, "Coastal zone", fontsize=7.2, va="center")
    fig.text(0.5, 0.02, f"{model}-type simulations; maps in EASE-Grid 2.0 (equal-area) coordinates, each region at its own scale; source patches enlarged for legibility.",
             ha="center", fontsize=7)
    return save(fig, f"Fig8_sources_corridors_{model}")


# ------------------------------------------------------------------ Figure 9: network structure
def fig9(model="PLUS"):
    S = load_summary(); LSs = ["1990", "2000", "2010", "2020", "2024", f"2040_{model}_NDS", f"2040_{model}_PCS"]
    LSlab = ["1990", "2000", "2010", "2020", "2024", "NDS", "PCS"]
    fig = plt.figure(figsize=(7.2, 6.6))
    nets = {n: dict(np.load(f"{CACHE}/net_{n}.npz")) for n in REGIONS}
    # (a) network graphs, 2024
    for i, name in enumerate(REGIONS):
        ax = fig.add_axes([0.01 + i * 0.247, 0.6, 0.235, 0.33]); net = nets[name]; ls = "2024"
        mk = net["mask"]; w = region_window(net)
        ax.imshow(np.where(mk, 0.9, np.nan), cmap="gray", vmin=0, vmax=1, extent=ext_of(mk, net["masktf"]), interpolation="nearest")
        npth, lpth = f"{ROOT}/step6/{name}/nodes_{ls}.csv", f"{ROOT}/step6/{name}/links_{ls}.csv"
        if os.path.exists(npth) and f"{ls}|cen" in net:
            nd = pd.read_csv(npth); cen = net[f"{ls}|cen"]; cls_ = dpc_class(nd)
            if os.path.exists(lpth):
                L = pd.read_csv(lpth)
                for _, l in L.sort_values("I_ij").iterrows():
                    a, b = int(l.i) - 1, int(l.j) - 1
                    strong = l.I_ij >= 0.10
                    ax.plot(cen[[a, b], 0], cen[[a, b], 1], color="#C0392B" if strong else "#5D6D7E",
                            lw=0.5 + 2.2 * min(l.I_ij / 0.3, 1), alpha=0.9 if strong else 0.55, zorder=2, solid_capstyle="round")
            sz = 4 + 60 * np.sqrt(nd.area_km2.values / nd.area_km2.max())
            ax.scatter(cen[:, 0], cen[:, 1], s=sz, c=np.array(DPC_COLS)[cls_], edgecolors="black", linewidths=0.4, zorder=3)
        ax.set_xlim(w[0], w[1]); ax.set_ylim(w[2], w[3]); ax.set_aspect("equal", adjustable="datalim")
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_linewidth(0.5)
        ax.set_title(f"{REGIONS[name][0]}  {REGIONS[name][1]}", fontsize=8.2, color=REG_COLS[name])
        if i == 0:
            panel(ax, "a", x=0.04, y=1.08)
    axl = fig.add_axes([0.03, 0.555, 0.95, 0.03]); axl.axis("off"); axl.set_xlim(0, 1); axl.set_ylim(0, 1)
    for i, (c, t) in enumerate(zip(DPC_COLS, DPC_CLS)):
        axl.scatter(0.01 + i * 0.15, 0.5, s=30, c=c, edgecolors="black", linewidths=0.4); axl.text(0.022 + i * 0.15, 0.5, t, fontsize=7, va="center")
    axl.plot([0.62, 0.66], [0.5, 0.5], color="#C0392B", lw=2); axl.text(0.665, 0.5, "Strong interaction (I$_{ij}$ ≥ 0.10)", fontsize=7, va="center")
    axl.plot([0.86, 0.9], [0.5, 0.5], color="#5D6D7E", lw=0.8); axl.text(0.905, 0.5, "Weaker", fontsize=7, va="center")
    fig.text(0.5, 0.522, "Ecological networks in 2024: node size scaled to source area, line width to interaction I$_{ij}$.",
             ha="center", fontsize=7.3, style="italic")
    # (b) dPC decomposition
    ax = fig.add_axes([0.08, 0.08, 0.4, 0.38])
    xt, xl = [], []
    for i, name in enumerate(REGIONS):
        for j, ls in enumerate(LSs):
            npth = f"{ROOT}/step6/{name}/nodes_{ls}.csv"
            if not os.path.exists(npth):
                continue
            nd = pd.read_csv(npth); tot = nd[["dPC_intra", "dPC_flux", "dPC_connector"]].sum()
            sh = tot / max(tot.sum(), 1e-12) * 100; x = i * 8 + j
            b0 = 0
            for k, (c_, col) in enumerate((("dPC_intra", "#1E8449"), ("dPC_flux", "#5DADE2"), ("dPC_connector", "#E67E22"))):
                ax.bar(x, sh[c_], bottom=b0, width=0.8, color=col, ec="white", lw=0.3); b0 += sh[c_]
            xt.append(x); xl.append(LSlab[j])
        ax.text(i * 8 + 3, 104, REGIONS[name][2], ha="center", fontsize=7.8, fontweight="bold", color=REG_COLS[name])
    ax.set_xticks(xt); ax.set_xticklabels(xl, rotation=90, fontsize=6.4); ax.set_ylim(0, 100); ax.set_ylabel("Share of total dPC (%)")
    clean(ax); ax.tick_params(axis="x", length=0)
    for k, (t, col) in enumerate((("intra (habitat area)", "#1E8449"), ("flux (dispersal)", "#5DADE2"), ("connector (stepping stone)", "#E67E22"))):
        ax.add_patch(Rectangle((0.0 + k * 0.34, -0.33), 0.04, 0.05, fc=col, transform=ax.transAxes, clip_on=False))
        ax.text(0.05 + k * 0.34, -0.305, t, transform=ax.transAxes, fontsize=6.8, va="center")
    panel(ax, "b", x=-0.1, y=1.06)
    # (c) interaction summaries (heat tiles)
    mets = [("strong", "Strong pairs\n(I$_{ij}$ ≥ 0.10)"), ("dom", "Mean I of\ndominant source"), ("sum", "Sum of\nall I$_{ij}$")]
    for k, (key, lab) in enumerate(mets):
        ax = fig.add_axes([0.6 + k * 0.135, 0.08, 0.12, 0.38])
        V = np.full((len(LSs), 4), np.nan)
        for i, name in enumerate(REGIONS):
            for j, ls in enumerate(LSs):
                lp = f"{ROOT}/step6/{name}/links_{ls}.csv"
                if not os.path.exists(lp):
                    continue
                L = pd.read_csv(lp)
                if key == "strong":
                    V[j, i] = (L.I_ij >= 0.10).sum()
                elif key == "sum":
                    V[j, i] = L.I_ij.sum()
                else:
                    nd = pd.read_csv(f"{ROOT}/step6/{name}/nodes_{ls}.csv"); d = int(nd.loc[nd.dPC.idxmax(), "source"])
                    V[j, i] = L[(L.i == d) | (L.j == d)].I_ij.mean() if len(L) else np.nan
        Vn = V / np.nanmax(V, 0)
        ax.imshow(Vn, cmap="PuRd", vmin=-0.1, vmax=1.15, aspect="auto")
        for (r, c), v in np.ndenumerate(V):
            ax.text(c, r, "–" if np.isnan(v) else (f"{v:.0f}" if key == "strong" else f"{v:.1f}" if key == "sum" else f"{v:.2f}"),
                    ha="center", va="center", fontsize=6.2, color="white" if Vn[r, c] > 0.7 else "black")
        ax.set_xticks(range(4)); ax.set_xticklabels([REGIONS[n][2] for n in REGIONS], fontsize=6.6, rotation=90)
        ax.set_yticks(range(len(LSs))); ax.set_yticklabels(LSlab if k == 0 else [], fontsize=7)
        ax.axhline(4.5, color="black", lw=0.8); ax.tick_params(length=0)
        ax.set_title(lab, fontsize=7.2, pad=4, linespacing=1.1)
        if k == 0:
            panel(ax, "c", x=-0.45, y=1.06)
    fig.text(0.6, 0.015, f"2040 rows: {model}-type simulations; columns normalised for shading.", fontsize=6.8)
    return save(fig, f"Fig9_network_structure_{model}")


# ------------------------------------------------------------------ Figure 10: ecological current with zoom windows
def cur_points(nets, ls, n_per=(1, 2, 2, 2), size=110):
    from scipy.ndimage import uniform_filter
    pts = []
    for (name, net), k in zip(nets.items(), n_per):
        if f"{ls}|cur" not in net:
            continue
        cu = np.nan_to_num(net[f"{ls}|cur"].astype(np.float32)); sm = uniform_filter(cu, 9)
        tf = net[f"{ls}|curtf"]
        for _ in range(k):
            r, c = np.unravel_index(np.argmax(sm), sm.shape)
            pts.append((name, r, c, tf)); sm[max(0, r - size):r + size, max(0, c - size):c + size] = 0
    return pts


def fig10(model="PLUS", win=100):
    from matplotlib.patches import ConnectionPatch
    from matplotlib.colors import LogNorm
    nets = {n: dict(np.load(f"{CACHE}/net_{n}.npz")) for n in REGIONS}; base = load_base()
    l0, l1, l2 = "2024", f"2040_{model}_NDS", f"2040_{model}_PCS"
    # hexagon mean current (2040 PCS) from the 3 km grids
    rows = []
    for name, net in nets.items():
        if f"{l2}|cur" not in net:
            continue
        cu = net[f"{l2}|cur"].astype(np.float32); tf = net[f"{l2}|curtf"]
        rr, cc = np.nonzero(np.isfinite(cu) & (cu > 0))
        lo, la = ease_lonlat(tf[0] + (cc + 0.5) * tf[2], tf[1] + (rr + 0.5) * tf[3])
        rows.append(pd.DataFrame(dict(region=name, lon=lo, lat=la, study=1.0, cur=cu[rr, cc])))
    h = hex_agg(pd.concat(rows), ["cur"]); h["val"] = h.cur / h.study
    vals = h.val[h.val > 0]; norm = LogNorm(vmin=np.percentile(vals, 2), vmax=np.percentile(vals, 99.5))
    pts = cur_points(nets, l2)
    W_IN = 7.2; xs, ys = frame_xy(); dx, dy = xs[1] - xs[0], ys[1] - ys[0]
    mw = 7.0; mh = mw * dy / dx; bw = 2.3; bh = 1.2; gap = 0.45
    H_IN = bh + 0.2 + gap + mh + gap + bh + 0.2 + 0.6
    fig = plt.figure(figsize=(W_IN, H_IN)); F = lambda x, y, w, h_: [x / W_IN, y / H_IN, w / W_IN, h_ / H_IN]
    ax = fig.add_axes(F(0.1, 0.6 + bh + 0.2 + gap, mw, mh))
    pc, _, _ = hexmap(ax, base, h, "val", "inferno_r", norm, f"Mean ecological current, 2040 PCS ({model}-type)")
    scalebar(ax, xs, ys); north(ax, xs, ys)
    order = sorted(range(len(pts)), key=lambda i: ease_lonlat(pts[i][3][0] + pts[i][2] * pts[i][3][2], 0)[0])
    lat_of = lambda p: ease_lonlat(0, p[3][1] + p[1] * p[3][3])[1]
    top = sorted([i for i in order if lat_of(pts[i]) >= np.median([lat_of(p) for p in pts])][:3],
                 key=lambda i: pts[i][3][0] + pts[i][2] * pts[i][3][2])
    bot = sorted([i for i in order if i not in top][:3], key=lambda i: pts[i][3][0] + pts[i][2] * pts[i][3][2])
    letters = iter("ABCDEF")
    for row, (lst, yb) in enumerate(((top, 0.6 + bh + 0.2 + gap + mh + gap), (bot, 0.6))):
        xg = (W_IN - 3 * bw) / 2
        for k, i in enumerate(lst):
            name, r, c, tf = pts[i]; L = next(letters); net = nets[name]
            r0, c0 = max(0, r - win // 2), max(0, c - win // 2)
            for j, (ls, lab) in enumerate(((l1, "NDS"), (l2, "PCS"))):
                axz = fig.add_axes(F(k * (bw + xg) + j * (bw / 2), yb, bw / 2 - 0.02, bh))
                cu = net[f"{ls}|cur"].astype(np.float32)[r0:r0 + win, c0:c0 + win]
                axz.set_facecolor("white")
                axz.imshow(np.where(np.isfinite(cu), 0.86, np.nan), cmap="gray", vmin=0, vmax=1, interpolation="nearest", aspect="auto")
                axz.imshow(np.where(cu > 0, cu, np.nan), cmap="inferno_r", norm=norm, interpolation="nearest", aspect="auto")
                axz.set_xticks([]); axz.set_yticks([])
                for sp in axz.spines.values():
                    sp.set_linewidth(0.6)
                axz.text(0.04, 0.96, f"{L}  {lab}" if j == 0 else lab, transform=axz.transAxes, va="top", fontsize=7.3,
                         fontweight="bold", bbox=dict(fc="white", ec="none", pad=0.8))
                mcur = np.nanmean(np.where(cu > 0, cu, np.nan))
                axz.text(0.96, 0.04, f"{mcur:.2f}", transform=axz.transAxes, ha="right", va="bottom", fontsize=6.8,
                         bbox=dict(fc="white", ec="none", pad=0.6))
                if j == 0:
                    axz.plot([4, 4 + 100 / 3], [win - 6, win - 6], color="black", lw=1.4)
                    axz.text(4 + 50 / 3, win - 8, "100 km", ha="center", va="bottom", fontsize=6.3,
                             bbox=dict(fc="white", ec="none", pad=0.5))
                    anchor = axz
            lo, la = ease_lonlat(tf[0] + (c + 0.5) * tf[2], tf[1] + (r + 0.5) * tf[3])
            px, py = fwd(lo, la)
            ax.plot(px, py, "s", ms=4.5, mfc="none", mec="#0B5345", mew=1.2, zorder=9)
            ax.text(px + dx * 0.009, py, L, fontsize=7.5, fontweight="bold", va="center", path_effects=halo(), zorder=9)
            fig.add_artist(ConnectionPatch(xyA=(px, py), coordsA=ax.transData, xyB=(1.0, 0.0 if row == 0 else 1.0),
                                           coordsB=anchor.transAxes, color="#444444", lw=0.5))
    cb = hbar(fig, F(1.6, 0.18, 4.0, 0.08), pc, "Ecological current (log scale); window labels: mean current in the window")
    cb.ax.xaxis.set_major_formatter(mt.FormatStrFormatter("%g")); cb.ax.xaxis.set_minor_formatter(mt.FormatStrFormatter("%g"))
    cb.ax.tick_params(which="minor", labelsize=6.5)
    return save(fig, f"Fig10_current_{model}")


# ------------------------------------------------------------------ Figure 11: morphology and habitat quality
MSPA_N = ["Core", "Islet", "Perforation", "Edge", "Loop", "Bridge", "Branch"]
MSPA_C = ["#1E8449", "#F4D03F", "#5DADE2", "#1B2631", "#E74C3C", "#C0392B", "#E59866"]


def fig11(model="PLUS"):
    from matplotlib.colors import Normalize, TwoSlopeNorm
    G = load_gi(); base = load_base(); S = load_summary()
    xs, ys = frame_xy(); asp = (ys[1] - ys[0]) / (xs[1] - xs[0])
    W_IN = 7.2; sw = 3.5; sh = sw * asp
    H_IN = 0.1 + 2 * sh + 0.8 + 0.75 + 1.9 + 0.1
    fig = plt.figure(figsize=(W_IN, H_IN)); F = lambda x, y, w, h: [x / W_IN, y / H_IN, w / W_IN, h / H_IN]
    yt = H_IN - 0.1 - sh
    c24 = gi_hex(G, "2024", "core"); q24 = gi_hex(G, "2024", "hq")
    ax = fig.add_axes(F(0.08, yt, sw, sh)); pa, _, _ = hexmap(ax, base, c24, "val", "Greens", Normalize(0, 1), "(a)  MSPA core share of GI, 2024")
    ax = fig.add_axes(F(0.08 + sw + 0.06, yt, sw, sh)); pb, _, _ = hexmap(ax, base, q24, "val", "viridis", Normalize(0.2, 0.9), "(b)  Mean habitat quality, 2024")
    dn = TwoSlopeNorm(vcenter=0, vmin=-0.15, vmax=0.15)
    dd = []
    for var in ("core", "hq"):
        a = gi_hex(G, f"2040_{model}_PCS", var); b = gi_hex(G, f"2040_{model}_NDS", var)
        a = a.merge(b[["q", "r", "val"]].rename(columns={"val": "vb"}), on=["q", "r"]); a["d"] = a.val - a.vb; dd.append(a)
    y2 = yt - sh - 0.8
    ax = fig.add_axes(F(0.08, y2, sw, sh)); pc_, _, _ = hexmap(ax, base, dd[0], "d", "PiYG", dn, "(c)  Core share, 2040 PCS − NDS")
    ax = fig.add_axes(F(0.08 + sw + 0.06, y2, sw, sh)); hexmap(ax, base, dd[1], "d", "PiYG", dn, "(d)  Habitat quality, 2040 PCS − NDS")
    hbar(fig, F(0.4, yt - 0.42, 2.8, 0.07), pa, "Core share of forest + wetland", ticks=[0, 0.25, 0.5, 0.75, 1])
    hbar(fig, F(0.4 + sw + 0.06, yt - 0.42, 2.8, 0.07), pb, "Habitat quality (0–1)")
    hbar(fig, F(1.9, y2 - 0.42, 3.4, 0.07), pc_, f"Difference PCS − NDS ({model}-type)", ticks=[-0.15, -0.1, -0.05, 0, 0.05, 0.1, 0.15])
    # (e) MSPA composition by region and landscape
    LSs = ["1990", "2000", "2010", "2020", "2024", f"2040_{model}_NDS", f"2040_{model}_PCS"]; LSlab = ["1990", "2000", "2010", "2020", "2024", "NDS", "PCS"]
    ax = fig.add_axes(F(0.55, 0.42, W_IN - 1.6, 1.5))
    for i, name in enumerate(REGIONS):
        for j, ls in enumerate(LSs):
            r_ = S[(S.region == name) & (S.landscape == ls)]
            if not len(r_):
                continue
            b0 = 0; x = i * 8 + j
            for nm, col in zip(MSPA_N, MSPA_C):
                v = r_[f"mspa_{nm.lower()}_pct"].iloc[0]; ax.bar(x, v, bottom=b0, width=0.8, color=col, ec="white", lw=0.25); b0 += v
        ax.text(i * 8 + 3, 103, f"{REGIONS[name][0]}  {REGIONS[name][1]}", ha="center", fontsize=7.6, fontweight="bold", color=REG_COLS[name])
    ax.set_xticks([i * 8 + j for i in range(4) for j in range(7)]); ax.set_xticklabels(LSlab * 4, rotation=90, fontsize=6.5)
    ax.set_ylim(0, 100); ax.set_ylabel("Share of GI (%)"); clean(ax); ax.tick_params(axis="x", length=0)
    panel(ax, "e", x=-0.06, y=1.04)
    axl = fig.add_axes(F(W_IN - 1.0, 0.5, 0.95, 1.35)); axl.axis("off"); axl.set_xlim(0, 1); axl.set_ylim(0, 7.3)
    for k, (nm, col) in enumerate(zip(MSPA_N, MSPA_C)):
        axl.add_patch(Rectangle((0.02, 6.6 - k), 0.18, 0.55, fc=col, ec="none")); axl.text(0.27, 6.87 - k, nm, fontsize=7.2, va="center")
    return save(fig, f"Fig11_mspa_habitat_{model}")


# ------------------------------------------------------------------ Figure 12: indicator trajectories
IND = [("hq_mean", "Mean habitat quality", 1), ("res_mean_1_10", "Mean resistance (1–10)", 1),
       ("vul_high_pct", "Highly vulnerable land (%)", 1), ("mspa_core_pct", "MSPA core (% of GI)", 1),
       ("ECA_pct", "Equivalent connected area (%)", 1), ("hq_high_pct", "High-quality habitat (%)", 1)]


def fig12():
    S = load_summary()
    fig, axs = plt.subplots(2, 3, figsize=(7.2, 4.9))
    plt.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.16, wspace=0.38, hspace=0.42)
    hist = [str(y) for y in YEARS]
    for k, (ax, (col, lab, _)) in enumerate(zip(axs.ravel(), IND)):
        if col not in S:
            ax.axis("off"); continue
        for name in REGIONS:
            r_ = S[S.region == name].set_index("landscape")
            yv = [r_[col].get(l, np.nan) for l in hist]
            ax.plot(YEARS, yv, "-o", color=REG_COLS[name], ms=2.8, lw=1.1)
            for m, off in (("FLUS", -1.0), ("PLUS", 1.0)):
                for sc_, mk in (("NDS", "s"), ("PCS", "^")):
                    v = r_[col].get(f"2040_{m}_{sc_}", np.nan)
                    if np.isnan(v):
                        continue
                    x = 2040 + off
                    ax.plot([2024, x], [yv[-1], v], color=REG_COLS[name], lw=0.5, ls=(0, (2, 2)), alpha=0.7)
                    ax.plot(x, v, mk, ms=3.6, color=REG_COLS[name], mfc=REG_COLS[name] if sc_ == "PCS" else "white", mew=0.8)
        ax.axvspan(2026, 2044, color="#F2F2F2", zorder=0)
        ax.set_xticks([1990, 2000, 2010, 2020, 2040]); ax.set_xlim(1986, 2044)
        ax.set_title(lab, fontsize=8.2); clean(ax); ax.grid(axis="y", color="#EEEEEE", lw=0.5); ax.set_axisbelow(True)
        ax.yaxis.set_major_locator(mt.MaxNLocator(4))
        panel(ax, "abcdef"[k], x=-0.12, y=1.04)
    axl = fig.add_axes([0.08, 0.01, 0.9, 0.06]); axl.axis("off"); axl.set_xlim(0, 1); axl.set_ylim(0, 1)
    for i, name in enumerate(REGIONS):
        axl.plot([i * 0.15, i * 0.15 + 0.03], [0.7, 0.7], "-o", color=REG_COLS[name], ms=2.8, lw=1.1)
        axl.text(i * 0.15 + 0.037, 0.7, REGIONS[name][1], va="center", fontsize=7.2)
    for i, (mk, f_, t) in enumerate((("s", "white", "2040 NDS"), ("^", "#333333", "2040 PCS"))):
        axl.plot(0.62 + i * 0.13, 0.7, mk, ms=4, color="#333333", mfc=f_); axl.text(0.635 + i * 0.13, 0.7, t, va="center", fontsize=7.2)
    axl.text(0.0, 0.12, "2040 markers: left = FLUS-type, right = PLUS-type; shaded band = projection period.", fontsize=7)
    return save(fig, "Fig12_trajectories")


# ------------------------------------------------------------------ Figure 13: PCS gains over NDS
GAINS = [("hq_mean", "Mean habitat quality", 1), ("hq_high_pct", "High-quality habitat", 1), ("mspa_core_pct", "MSPA core share", 1),
         ("source_km2", "Source area", 1), ("ECA_pct", "Equivalent connected area", 1), ("strong_pairs", "Strong source pairs", 1),
         ("res_mean_1_10", "Mean resistance", -1), ("vul_high_pct", "Highly vulnerable land", -1), ("current_mean", "Mean current density", 1)]


def fig13():
    S = load_summary()
    rows = [g for g in GAINS if g[0] in S]
    fig = plt.figure(figsize=(7.2, 3.9))
    for i, name in enumerate(REGIONS):
        ax = fig.add_axes([0.22 + i * 0.195, 0.16, 0.17, 0.72])
        r_ = S[S.region == name].set_index("landscape")
        for k, (col, lab, sign) in enumerate(rows):
            ax.axhline(k, color="#F0F0F0", lw=0.6, zorder=0)
            for m, off, mk in (("FLUS", -0.15, "s"), ("PLUS", 0.15, "o")):
                a, b = r_[col].get(f"2040_{m}_PCS", np.nan), r_[col].get(f"2040_{m}_NDS", np.nan)
                v = (a - b) / abs(b) * 100 * sign if b not in (0, np.nan) and np.isfinite(b) and b != 0 else np.nan
                if np.isnan(v):
                    continue
                c = "#1E8449" if v >= 0 else "#B03A2E"
                ax.plot([0, v], [k + off, k + off], color=c, lw=1.3, alpha=0.8)
                ax.plot(v, k + off, mk, ms=3.6, color=c, mfc=c if m == "PLUS" else "white", mew=0.8)
        ax.axvline(0, color="black", lw=0.6)
        ax.set_ylim(len(rows) - 0.5, -0.5); ax.set_yticks(range(len(rows)))
        ax.set_yticklabels([l + (" (↓)" if s_ < 0 else "") for _, l, s_ in rows] if i == 0 else [], fontsize=7.4)
        ax.xaxis.set_major_locator(mt.MaxNLocator(4, symmetric=True)); ax.tick_params(axis="x", labelsize=7)
        clean(ax); ax.set_title(f"{REGIONS[name][0]}  {REGIONS[name][1]}", fontsize=8.2, color=REG_COLS[name])
    fig.text(0.6, 0.04, "Relative improvement of PCS over NDS in 2040 (%)", ha="center", fontsize=8)
    fig.text(0.01, 0.005, "(↓) lower is better: improvement = reduction. Open squares FLUS-type, filled circles PLUS-type; green = PCS better.",
             fontsize=6.8)
    return save(fig, "Fig13_PCS_gains")


# ------------------------------------------------------------------ Figure 14: conservation sequence
PRI = ["1  Protect hub sources", "2  Secure corridors", "3  Restore bottlenecks", "4  Extend to coastal wetlands"]
PRI_C = ["#1E8449", "#2E86C1", "#C0392B", "#17A589"]


def fig14(model="PLUS"):
    """Priority zones per hexagon (2040 PCS): hub = hub/important sources present; corridor = high corridor density;
    bottleneck = high current with high resistance; coastal wetland extension = wetland/water-rich hexagons outside
    the network with low habitat quality. First matching rule wins (sequence order)."""
    G = load_gi(); base = load_base(); nets = {n: dict(np.load(f"{CACHE}/net_{n}.npz")) for n in REGIONS}
    ls = f"2040_{model}_PCS"
    B = load_blocks(); i24 = B[0]["maps"].index("2024")
    wet = pd.concat([pd.DataFrame(dict(region=b["region"], lon=b["lon"], lat=b["lat"], study=b["study"].astype(float),
                                       wet=(b["counts"][i24, 3].astype(float) + b["counts"][i24, 4]))) for b in B])
    hw = hex_agg(wet, ["wet"]); hw["wet_share"] = hw.wet / hw.study
    hq = gi_hex(G, ls, "hq")[["q", "r", "val", "region"]].rename(columns={"val": "hq"})
    rs = gi_hex(G, ls, "res")[["q", "r", "val"]].rename(columns={"val": "res"})
    pts = []
    for name, net in nets.items():
        tf = net.get(f"{ls}|tf")
        if tf is None:
            continue
        nd_p = f"{ROOT}/step6/{name}/nodes_{ls}.csv"
        src = net[f"{ls}|src"]; kd = net[f"{ls}|kde"].astype(np.float32)
        cls_ = np.full(int(src.max()) + 1, 9)
        if os.path.exists(nd_p):
            nd = pd.read_csv(nd_p); cls_[nd.source.values] = dpc_class(nd)
        rr, cc = np.nonzero(src > 0)
        lo, la = ease_lonlat(tf[0] + (cc + 0.5) * tf[2], tf[1] + (rr + 0.5) * tf[3])
        pts.append(pd.DataFrame(dict(region=name, lon=lo, lat=la, study=1.0, hub=(cls_[src[rr, cc]] <= 1).astype(float), kde=0.0, cur=0.0)))
        rr, cc = np.nonzero(np.nan_to_num(kd) >= 0.5)
        lo, la = ease_lonlat(tf[0] + (cc + 0.5) * tf[2], tf[1] + (rr + 0.5) * tf[3])
        pts.append(pd.DataFrame(dict(region=name, lon=lo, lat=la, study=1.0, hub=0.0, kde=1.0, cur=0.0)))
        if f"{ls}|cur" in net:
            cu = net[f"{ls}|cur"].astype(np.float32); ct = net[f"{ls}|curtf"]
            thr = np.nanpercentile(cu[cu > 0], 90) if np.any(cu > 0) else np.inf
            rr, cc = np.nonzero(np.nan_to_num(cu) >= thr)
            lo, la = ease_lonlat(ct[0] + (cc + 0.5) * ct[2], ct[1] + (rr + 0.5) * ct[3])
            pts.append(pd.DataFrame(dict(region=name, lon=lo, lat=la, study=1.0, hub=0.0, kde=0.0, cur=1.0)))
    hp = hex_agg(pd.concat(pts), ["hub", "kde", "cur"])
    h = hq.merge(rs, on=["q", "r"]).merge(hw[["q", "r", "wet_share"]], on=["q", "r"], how="left").merge(
        hp[["q", "r", "hub", "kde", "cur"]], on=["q", "r"], how="left").fillna({"hub": 0, "kde": 0, "cur": 0, "wet_share": 0})
    rq = h.res.quantile(0.6)
    pri = np.full(len(h), -1)
    pri[(pri < 0) & (h.hub > 0)] = 0
    pri[(pri < 0) & (h.kde > 0)] = 1
    pri[(pri < 0) & (h.cur > 0) & (h.res >= rq)] = 2
    pri[(pri < 0) & (h.wet_share >= 0.2) & (h.hq < h.hq.median())] = 3
    h["pri"] = pri
    xs, ys = frame_xy(); dx, dy = xs[1] - xs[0], ys[1] - ys[0]
    W_IN = 7.2; mw = 7.0; mh = mw * dy / dx
    H_IN = 0.1 + mh + 0.6 + 2.1
    fig = plt.figure(figsize=(W_IN, H_IN)); F = lambda x, y, w, h_: [x / W_IN, y / H_IN, w / W_IN, h_ / H_IN]
    ax = fig.add_axes(F(0.1, H_IN - 0.1 - mh, mw, mh)); basemap(ax, base, labels=True)
    other = h[h.pri < 0]
    ax.add_collection(PolyCollection(hex_polys(other.q.values, other.r.values), facecolors="#D5D8DC", edgecolors="none", zorder=3))
    for k in range(4):
        s_ = h[h.pri == k]
        ax.add_collection(PolyCollection(hex_polys(s_.q.values, s_.r.values), facecolors=PRI_C[k], edgecolors="none", zorder=4))
    scalebar(ax, xs, ys); north(ax, xs, ys); panel(ax, "a", x=-0.005, y=1.01)
    ax.text(0.012, 0.975, f"Conservation priorities, 2040 PCS ({model}-type)", transform=ax.transAxes, va="top", fontsize=7.8,
            fontweight="bold", path_effects=halo(), zorder=9)
    axl = fig.add_axes(F(0.1, H_IN - 0.1 - mh - 0.5, mw, 0.2)); axl.axis("off"); axl.set_xlim(0, mw); axl.set_ylim(0, 1)
    for k in range(4):
        axl.add_patch(MPoly(np.c_[0.1 + k * 1.62 + 0.07 * np.cos(HEX_ANG), 0.5 + 0.35 * np.sin(HEX_ANG)], fc=PRI_C[k], ec="none"))
        axl.text(0.22 + k * 1.62, 0.5, PRI[k], va="center", fontsize=7.4)
    # (b) hexagon share by priority and region
    ax = fig.add_axes(F(0.75, 0.35, 3.0, 1.55))
    reg_tot = h.groupby("region").size()
    for i, name in enumerate(REGIONS):
        left = 0
        for k in range(4):
            v = ((h.region == name) & (h.pri == k)).sum() / max(reg_tot.get(name, 1), 1) * 100
            ax.barh(i, v, left=left, color=PRI_C[k], height=0.6, ec="white", lw=0.4); left += v
            if v >= 6:
                ax.text(left - v / 2, i, f"{v:.0f}", ha="center", va="center", fontsize=6.8, color="white")
    ax.set_yticks(range(4)); ax.set_yticklabels([REGIONS[n][1] for n in REGIONS]); ax.set_ylim(3.6, -0.6)
    for t_, n in zip(ax.get_yticklabels(), REGIONS):
        t_.set_color(REG_COLS[n]); t_.set_fontweight("bold")
    ax.set_xlabel("Share of coastal hexagons (%)"); clean(ax); ax.tick_params(axis="y", length=0)
    panel(ax, "b", x=-0.33, y=1.04)
    # (c) the sequence as a staged arrow diagram
    ax = fig.add_axes(F(4.25, 0.25, 2.85, 1.7)); ax.axis("off"); ax.set_xlim(0, 4); ax.set_ylim(0, 2.4)
    txt = ["Protect hub\nsources", "Secure\ncorridors", "Restore\nbottlenecks", "Extend to\ncoastal\nwetlands"]
    for k in range(4):
        x0 = k * 1.0; y0 = 0.25 + k * 0.45
        ax.add_patch(FancyBboxPatch((x0 + 0.04, y0), 0.92, 0.62, boxstyle="round,pad=0,rounding_size=0.06", fc=PRI_C[k], ec="none"))
        ax.text(x0 + 0.5, y0 + 0.31, txt[k], ha="center", va="center", fontsize=7, color="white", fontweight="bold")
        ax.text(x0 + 0.08, y0 + 0.66, str(k + 1), fontsize=9, fontweight="bold", color=PRI_C[k], va="bottom")
        if k < 3:
            ax.add_patch(FancyArrowPatch((x0 + 0.96, y0 + 0.31), (x0 + 1.04, y0 + 0.45 + 0.31), arrowstyle="-|>", mutation_scale=8,
                                         lw=0.9, color="#333333"))
    ax.text(2.0, 0.05, "short term  →  long term", ha="center", fontsize=7.2, style="italic")
    panel(ax, "c", x=0.0, y=1.0)
    return save(fig, f"Fig14_conservation_sequence_{model}")


FIGS = {"2": fig2, "3": fig3, "4": fig4, "5": fig5, "6": fig6, "7": fig7, "8": fig8, "9": fig9, "10": fig10,
        "11": fig11, "12": fig12, "13": fig13, "14": fig14}
if __name__ == "__main__":
    set_style()
    args = sys.argv[1:] or ["2"]
    if "prep" in args:
        args.remove("prep")
        if any(a in ("3", "4", "5", "14") for a in args) or not args:
            prep_lulc()
        if any(a in ("7", "8", "9", "10", "11", "12", "13", "14") for a in args):
            prep6()
        if not args:
            args = list(FIGS)
    for a in args:
        FIGS[a]()


In [ ]:
# STEP 7a - Figure 1: study area (hexagon map + longitude/latitude profiles + regional and country panels)
import os
"""Figure 1 (study area):
 (a) Albers equal-area conic map of the 50 km Asian coastal zone on grey shaded relief (white sea, coastlines,
     country borders); ~55 km hexagons summarise 2024 impervious-surface area (6 classes, yellow -> purple);
     each UN M49 sub-region (A-D) is outlined along its own hexagons, so outlines never overlap. Marginal
     profiles: impervious area by map x (top) and map y (right). Legend sits below the map, outside the frame.
 (b) which region is most affected: share of the coastal zone that changed LULC class 1990 -> 2024.
 (c) LULC transition chord diagrams 1990 -> 2024 for the four regions (ribbons coloured by source class,
     outer ring = partner classes, ticks at a common interval).
Part 1 (Colab only, needs the Drive rasters): figs/fig1_blocks.csv (3 km blocks), figs/fig1_trans.csv
         (300 m transition matrices), figs/fig1_base.npz (relief, coastlines, borders).
Part 2 (numpy/pandas/matplotlib only): figs/Fig1_study_area.png (600 dpi) and .pdf.
"""
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, matplotlib.ticker
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Polygon as MPoly, Wedge, FancyArrow

ROOT = os.environ.get("ROOT", "/content/drive/MyDrive/AsiaCoastGI")
FIG = f"{ROOT}/figs"
REGIONS = {"WesternAsia": ("A", "Western Asia", (24, 11, 64, 44)),
           "SouthAsia": ("B", "South Asia", (43, -1, 98, 40)),
           "SoutheastAsia": ("C", "Southeast Asia", (91, -12, 142, 29)),
           "EastAsia": ("D", "East Asia", (97, 17, 150, 47))}
SHORT = {"WesternAsia": "WA", "SouthAsia": "SA", "SoutheastAsia": "SEA", "EastAsia": "EA"}
EXT = (18, 158, -16, 52)                         # lon0, lon1, lat0, lat1 of the map frame
IMPERV = 6
CLS = ["Cropland", "Forest", "Shrub/grass", "Wetland", "Water", "Impervious", "Bare"]

# ------------------------------------------------------------------ projection (spherical Albers)
R_E = 6371.0
LON0, LAT0, P1, P2 = 90.0, 15.0, 0.0, 40.0
_p1, _p2, _l0 = np.radians(P1), np.radians(P2), np.radians(LAT0)
_n = (np.sin(_p1) + np.sin(_p2)) / 2
_C = np.cos(_p1) ** 2 + 2 * _n * np.sin(_p1)
_rho0 = R_E * np.sqrt(_C - 2 * _n * np.sin(_l0)) / _n


def fwd(lon, lat):
    lon, lat = np.radians(np.asarray(lon, float)), np.radians(np.asarray(lat, float))
    th = _n * (lon - np.radians(LON0))
    rho = R_E * np.sqrt(_C - 2 * _n * np.sin(lat)) / _n
    return rho * np.sin(th), _rho0 - rho * np.cos(th)


def inv(x, y):
    rho = np.hypot(x, _rho0 - y) * np.sign(_n)
    th = np.arctan2(x, _rho0 - y)
    lat = np.arcsin(np.clip((_C - (rho * _n / R_E) ** 2) / (2 * _n), -1, 1))
    return np.degrees(np.radians(LON0) + th / _n), np.degrees(lat)


def lonlat_box(lon0, lat0, lon1, lat1, n=60):
    lo = np.r_[np.linspace(lon0, lon1, n), np.full(n, lon1), np.linspace(lon1, lon0, n), np.full(n, lon0)]
    la = np.r_[np.full(n, lat0), np.linspace(lat0, lat1, n), np.full(n, lat1), np.linspace(lat1, lat0, n)]
    return np.c_[fwd(lo, la)]


# ------------------------------------------------------------------ Part 1: data (Colab)
def prepare():
    import rasterio, geopandas as gpd, zipfile, subprocess
    from pyproj import Transformer
    os.makedirs(FIG, exist_ok=True)
    tmp = "/content/tmp_fig"; os.makedirs(tmp, exist_ok=True)
    rows, trans = [], []
    for name in REGIONS:
        with rasterio.open(f"{ROOT}/step2/studymask_{name}.tif") as s:
            m = s.read(1) > 0; T = s.transform; crs = s.crs
        with rasterio.open(f"{ROOT}/step2_filtered/lc_{name}_1990.tif") as s:
            i90 = s.read(1) == IMPERV
        with rasterio.open(f"{ROOT}/step3/lc_{name}_2024.tif") as s:
            i24 = s.read(1) == IMPERV
        f = 10
        h, w = (m.shape[0] // f) * f, (m.shape[1] // f) * f
        bl = lambda a: a[:h, :w].reshape(h // f, f, w // f, f).sum((1, 3), dtype=np.int32)
        S, A, B = bl(m), bl(i90 & m), bl(i24 & m)
        del i90, i24
        with rasterio.open(f"{ROOT}/step2_filtered/lc_{name}_1990.tif") as s:
            a = s.read(1)[m].astype(np.int64)
        with rasterio.open(f"{ROOT}/step3/lc_{name}_2024.tif") as s:
            b = s.read(1)[m].astype(np.int64)
        M = np.bincount(a * 8 + b, minlength=64).reshape(8, 8)[1:, 1:] * 0.09
        trans.append(pd.DataFrame(M, index=CLS, columns=CLS).stack().rename("km2").reset_index()
                     .rename(columns={"level_0": "frm", "level_1": "to"}).assign(region=name))
        del a, b, m
        rr, cc = np.nonzero(S)
        x = T.c + (cc * f + f / 2) * T.a; y = T.f + (rr * f + f / 2) * T.e
        lon, lat = Transformer.from_crs(crs, "EPSG:4326", always_xy=True).transform(x, y)
        rows.append(pd.DataFrame(dict(region=name, lon=lon, lat=lat, study_km2=S[rr, cc] * 0.09,
                                      imp1990_km2=A[rr, cc] * 0.09, imp2024_km2=B[rr, cc] * 0.09)))
        print(name, len(rr), "blocks")
    df = pd.concat(rows, ignore_index=True)
    ne = f"{tmp}/ne_10m_admin_0_countries.zip"
    if not os.path.exists(ne):
        subprocess.run(["wget", "-q", "-O", ne, "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip"])
    cty = gpd.read_file(f"zip://{ne}")[["ADMIN", "geometry"]]
    pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326")
    j = gpd.sjoin(pts, cty, how="left", predicate="within")
    j = j[~j.index.duplicated()]
    miss = j.ADMIN.isna()
    if miss.any():                                               # coastal blocks just off the NE polygons
        near = gpd.sjoin_nearest(pts[miss].to_crs(6933), cty.to_crs(6933), how="left")
        j.loc[miss, "ADMIN"] = near[~near.index.duplicated()].ADMIN.values
    df["country"] = j.ADMIN.values
    df.to_csv(f"{FIG}/fig1_blocks.csv", index=False)
    pd.concat(trans).to_csv(f"{FIG}/fig1_trans.csv", index=False)
    # basemap: Natural Earth grey shaded relief, reprojected to the map grid; coastline from NE 50 m land
    sr = f"{tmp}/GRAY_50M_SR_W.zip"
    if not os.path.exists(sr):
        subprocess.run(["wget", "-q", "-O", sr, "https://naciscdn.org/naturalearth/50m/raster/GRAY_50M_SR_W.zip"])
    tif = [n for n in zipfile.ZipFile(sr).namelist() if n.lower().endswith(".tif")][0]
    with rasterio.open(f"/vsizip/{sr}/{tif}") as s:
        rel = s.read(1); rT = s.transform
    xs, ys = frame_xy()
    X, Y = np.meshgrid(np.linspace(xs[0], xs[1], 3400), np.linspace(ys[1], ys[0], 1800))
    lo, la = inv(X, Y)
    cidx = ((lo - rT.c) / rT.a).astype(int).clip(0, rel.shape[1] - 1)
    ridx = ((la - rT.f) / rT.e).astype(int).clip(0, rel.shape[0] - 1)
    relief = rel[ridx, cidx].astype(np.uint8)
    land = f"{tmp}/ne_50m_land.zip"
    if not os.path.exists(land):
        subprocess.run(["wget", "-q", "-O", land, "https://naciscdn.org/naturalearth/50m/physical/ne_50m_land.zip"])
    g = gpd.read_file(f"zip://{land}").clip((EXT[0] - 15, EXT[2] - 15, EXT[1] + 25, EXT[3] + 15)).explode(index_parts=False)
    rings = [np.c_[fwd(*np.asarray(p.exterior.coords).T)] for p in g.geometry if p.area > 0.05]
    bd = f"{tmp}/ne_50m_admin_0_boundary_lines_land.zip"
    if not os.path.exists(bd):
        subprocess.run(["wget", "-q", "-O", bd, "https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_boundary_lines_land.zip"])
    gb = gpd.read_file(f"zip://{bd}").clip((EXT[0] - 15, EXT[2] - 15, EXT[1] + 25, EXT[3] + 15)).explode(index_parts=False)
    borders = [np.c_[fwd(*np.asarray(l.coords).T)] for l in gb.geometry if l.length > 0]
    np.savez_compressed(f"{FIG}/fig1_base.npz", relief=relief, extent=np.r_[xs, ys],
                        rings=np.array(rings, dtype=object), borders=np.array(borders, dtype=object))
    print("saved", f"{FIG}/fig1_blocks.csv", f"{FIG}/fig1_base.npz")


def frame_xy():
    b = lonlat_box(EXT[0], EXT[2], EXT[1], EXT[3], 200)
    return (b[:, 0].min(), b[:, 0].max()), (b[:, 1].min(), b[:, 1].max())


# ------------------------------------------------------------------ Part 2: figure
BLUE, GRID = "#1F96D4", "#BDBDBD"
CLASS_EDGES = [0, 1, 10, 50, 100, 500, 1e9]
CLASS_LABELS = ["<1", "1–10", "10–50", "50–100", "100–500", ">500"]
CLASS_COLS = ["#FDE725", "#7AD151", "#22A884", "#2A788E", "#414487", "#440154"]   # viridis, reversed
LULC_COLS = {"Cropland": "#F4D03F", "Forest": "#1E8449", "Shrub/grass": "#A9CF54", "Wetland": "#48C9B0",
             "Water": "#2E5BDB", "Impervious": "#E74C3C", "Bare": "#C8AD7F"}
REG_COLS = {"WesternAsia": "#B03A2E", "SouthAsia": "#B9770E", "SoutheastAsia": "#0E6655", "EastAsia": "#1A4592"}
REG_LABEL_AT = {"WesternAsia": (41, 29), "SouthAsia": (70, 13), "SoutheastAsia": (114, -12), "EastAsia": (139, 29)}
HEX_KM = 32.0                                   # hexagon circumradius (~55 km across)


def hexes(x, y, s=HEX_KM):
    """Pointy-top hexagon axial index (q, r) of map points (cube rounding)."""
    q = (np.sqrt(3) / 3 * x - y / 3) / s; r = (2 / 3 * y) / s; c = -q - r
    rq, rr, rc = np.round(q), np.round(r), np.round(c)
    dq, dr, dc = np.abs(rq - q), np.abs(rr - r), np.abs(rc - c)
    fq = (dq > dr) & (dq > dc); fr = ~fq & (dr > dc)
    rq = np.where(fq, -rr - rc, rq); rr = np.where(fr, -rq - rc, rr)
    return rq.astype(int), rr.astype(int)


def hex_center(q, r, s=HEX_KM):
    return s * np.sqrt(3) * (q + r / 2), s * 1.5 * r


HEX_ANG = np.radians(np.arange(6) * 60 + 30)                    # vertex angles, pointy-top
NB = [(1, 0), (0, 1), (-1, 1), (-1, 0), (0, -1), (1, -1)]      # neighbour across edge (vertex k, k+1)


CHORD_LAB = {"Cropland": "Crop", "Forest": "Forest", "Shrub/grass": "Shrub", "Wetland": "Wetland",
             "Water": "Water", "Impervious": "Imperv.", "Bare": "Bare"}


def chord(ax, M, title, scale_step):
    """Chord diagram of class transitions (off-diagonal km2). Arc length = outflow + inflow."""
    from matplotlib.path import Path
    from matplotlib.patches import PathPatch
    n = len(CLS); M = np.array(M, float); np.fill_diagonal(M, 0)
    tot = M.sum(1) + M.sum(0)
    keep = tot > 0
    gap = 2 * np.pi * 0.012; span = 2 * np.pi - gap * keep.sum()
    k = span / max(tot.sum(), 1e-9)
    start = {}; th = np.pi / 2
    for i in range(n):                                              # clockwise from 12 o'clock
        if not keep[i]:
            continue
        start[i] = th; th -= tot[i] * k + gap
    R0, R1 = 1.0, 1.07
    def arc(t0, t1, r, m=30):
        t = np.linspace(t0, t1, m); return np.c_[r * np.cos(t), r * np.sin(t)]
    out_pos = {i: start[i] for i in start}                          # cursor for outgoing segments
    in_pos = {i: start[i] - M[i].sum() * k for i in start}          # incoming after outgoing
    for i in start:                                                 # ribbons, largest last
        for j in np.argsort(M[i]):
            v = M[i, j]
            if v <= 0 or j not in start:
                continue
            a0 = out_pos[i]; a1 = a0 - v * k; out_pos[i] = a1
            b0 = in_pos[j]; b1 = b0 - v * k; in_pos[j] = b1
            p1, p2 = arc(a0, a1, R0 - 0.005), arc(b0, b1, R0 - 0.005)
            verts = [*p1, (0, 0), p2[0], *p2[1:], (0, 0), p1[0]]
            codes = ([Path.MOVETO] + [Path.LINETO] * (len(p1) - 1) + [Path.CURVE3, Path.CURVE3]
                     + [Path.LINETO] * (len(p2) - 1) + [Path.CURVE3, Path.CURVE3])
            ax.add_patch(PathPatch(Path(verts, codes), fc=LULC_COLS[CLS[i]], ec="none", alpha=0.62, zorder=2))
    for i in start:                                                 # class arcs, composition ring, ticks
        t0, t1 = start[i], start[i] - tot[i] * k
        ax.add_patch(MPoly(np.r_[arc(t0, t1, R0), arc(t1, t0, R1)], fc=LULC_COLS[CLS[i]], ec="white", lw=0.3, zorder=3))
        c = t0
        segs = [(j, M[i, j]) for j in range(n)] + [(j, M[j, i]) for j in range(n)]
        for j, v in segs:
            if v <= 0:
                continue
            ax.add_patch(MPoly(np.r_[arc(c, c - v * k, R1 + 0.025), arc(c - v * k, c, R1 + 0.06)],
                               fc=LULC_COLS[CLS[j]], ec="none", zorder=3))
            c -= v * k
        for tv in np.arange(0, tot[i] + 1e-9, scale_step):          # ticks every scale_step km2
            tt = t0 - tv * k
            ax.plot([1.095 * np.cos(tt), 1.12 * np.cos(tt)], [1.095 * np.sin(tt), 1.12 * np.sin(tt)], color="#222222", lw=0.4)
        mid = (t0 + t1) / 2; deg = np.degrees(mid) % 360
        left = 90 < deg < 270                                       # radial labels, never upside down
        if abs(t0 - t1) < 0.22:                                     # tiny arcs: colour key only
            pass
        elif 235 < deg < 305 or 55 < deg < 125:                     # bottom / top: horizontal labels
            ax.text(1.17 * np.cos(mid), 1.17 * np.sin(mid), CHORD_LAB[CLS[i]], ha="center",
                    va="top" if deg > 180 else "bottom", fontsize=6.3)
        else:                                                       # sides: radial labels, never upside down
            ax.text(1.17 * np.cos(mid), 1.17 * np.sin(mid), CHORD_LAB[CLS[i]], ha="right" if left else "left", va="center",
                    rotation=deg - 180 if left else deg, rotation_mode="anchor", fontsize=6.3)
    ax.set_xlim(-1.75, 1.75); ax.set_ylim(-1.85, 1.6); ax.set_aspect("equal"); ax.axis("off")
    ax.text(0, -1.6, title, ha="center", va="top", fontsize=7.5, fontweight="bold")


def draw(df, tr, base=None, out=None):
    plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],
                         "font.size": 7.5, "text.color": "black", "axes.labelcolor": "black", "xtick.color": "black",
                         "ytick.color": "black", "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.linewidth": 0.7,
                         "xtick.major.width": 0.7, "ytick.major.width": 0.7, "xtick.major.size": 2.8,
                         "ytick.major.size": 2.8, "font.weight": "normal"})
    import matplotlib.patheffects as pe
    from matplotlib.collections import PolyCollection, LineCollection
    xs, ys = frame_xy(); dx, dy = xs[1] - xs[0], ys[1] - ys[0]
    W_IN, H_IN = 7.2, 8.6
    fig = plt.figure(figsize=(W_IN, H_IN), dpi=300)
    mx0, mw = 0.07, 0.785
    mh = mw * W_IN * dy / dx / H_IN
    my0 = 1 - 0.105 - mh
    ax = fig.add_axes([mx0, my0, mw, mh])
    axt = fig.add_axes([mx0, my0 + mh + 0.027, mw, 0.06])
    axr = fig.add_axes([mx0 + mw + 0.03, my0, 0.085, mh])
    halo = [pe.withStroke(linewidth=2.2, foreground="white")]
    # basemap: grey hillshade on land, white sea, coastlines and borders
    ax.set_facecolor("white")
    if base is not None:
        e = base["extent"]
        from matplotlib.path import Path as MPath
        from matplotlib.patches import PathPatch
        rel = base["relief"].astype(np.float32) / 255.0
        g = (0.50 + 0.50 * rel).clip(0, 1)                          # natural grey terrain, light tone
        im = ax.imshow(np.dstack([g, g, g]), extent=(e[0], e[1], e[2], e[3]), interpolation="bilinear", zorder=0)
        rings = [np.asarray(r_, float) for r_ in base["rings"] if len(r_) > 3]
        if rings:
          verts = np.concatenate(rings); codes = np.concatenate(
            [[MPath.MOVETO] + [MPath.LINETO] * (len(r_) - 2) + [MPath.CLOSEPOLY] for r_ in rings])
          im.set_clip_path(PathPatch(MPath(verts, codes), transform=ax.transData))   # relief on land only, white sea
        ax.add_collection(LineCollection(list(base["rings"]), colors="#6E6E6E", linewidths=0.3, zorder=1))
        if "borders" in base.files:
            ax.add_collection(LineCollection(list(base["borders"]), colors="#8C8C8C", linewidths=0.3,
                                             linestyles=(0, (2, 1.5)), zorder=1))
    for lon in range(20, 181, 20):
        la = np.linspace(-30, 70, 200); gx, gy = fwd(np.full_like(la, lon), la)
        ax.plot(gx, gy, color=GRID, lw=0.35, zorder=1.5)
    for lat in (-20, 0, 20, 40, 60):
        lo = np.linspace(0, 190, 300); gx, gy = fwd(lo, np.full_like(lo, lat))
        ax.plot(gx, gy, color=GRID, lw=0.35, zorder=1.5)
    for lon in range(40, 161, 20):
        la = np.linspace(-30, 70, 4000); gx, gy = fwd(np.full_like(la, lon), la)
        k = np.argmin(np.abs(gy - ys[1]))
        if xs[0] + 0.02 * dx < gx[k] < xs[1] - 0.02 * dx:
            ax.text(gx[k], ys[1] + dy * 0.012, f"{lon}°E", ha="center", va="bottom", fontsize=7)
    for lat in (-20, 0, 20, 40):
        for side, ha, rot in ((0, "right", 90),):        # left edge only (right side holds the profile)
            lo = np.linspace(-20, 200, 6000); gx, gy = fwd(lo, np.full_like(lo, lat))
            k = np.argmin(np.abs(gx - xs[side]))
            if ys[0] + 0.03 * dy < gy[k] < ys[1] - 0.03 * dy:
                off = -dx * 0.005 if side == 0 else dx * 0.005
                ax.text(xs[side] + off, gy[k], f"{abs(lat)}°{'S' if lat < 0 else ('N' if lat > 0 else '')}",
                        ha=ha, va="center", rotation=rot, fontsize=7, clip_on=False)
    # hexagons (own lattice so region outlines can be traced exactly)
    x, y = fwd(df.lon.values, df.lat.values)
    q, r = hexes(x, y)
    h = pd.DataFrame(dict(q=q, r=r, region=df.region.values, imp=df.imp2024_km2.values, st=df.study_km2.values))
    reg_of = h.groupby(["q", "r", "region"]).st.sum().reset_index().sort_values("st").drop_duplicates(["q", "r"], keep="last")
    hv = h.groupby(["q", "r"]).imp.sum().reset_index().merge(reg_of[["q", "r", "region"]], on=["q", "r"])
    cx, cy = hex_center(hv.q.values, hv.r.values)
    polys = np.stack([np.c_[cx + HEX_KM * np.cos(t), cy + HEX_KM * np.sin(t)] for t in HEX_ANG], 1)
    cls = np.digitize(hv.imp.values, CLASS_EDGES[1:-1])
    ax.add_collection(PolyCollection(polys, facecolors=np.array(CLASS_COLS)[cls], edgecolors="white",
                                     linewidths=0.08, zorder=3))
    # region outlines = hexagon edges whose neighbour belongs to another region (or is empty)
    key = dict(zip(zip(hv.q, hv.r), hv.region))
    for name in REGIONS:
        segs = []
        for (qq, rr_), rg in key.items():
            if rg != name:
                continue
            ccx, ccy = hex_center(qq, rr_)
            for e_ in range(6):
                nq, nr = qq + NB[e_][0], rr_ + NB[e_][1]
                if key.get((nq, nr)) != name:
                    t0, t1 = HEX_ANG[e_], HEX_ANG[(e_ + 1) % 6]
                    segs.append([(ccx + HEX_KM * np.cos(t0), ccy + HEX_KM * np.sin(t0)),
                                 (ccx + HEX_KM * np.cos(t1), ccy + HEX_KM * np.sin(t1))])
        ax.add_collection(LineCollection(segs, colors=REG_COLS[name], linewidths=0.9, zorder=4))
        lx_, ly_ = fwd(*REG_LABEL_AT[name])
        ax.text(lx_, ly_, f"{REGIONS[name][0]}  {REGIONS[name][1]}", color=REG_COLS[name], fontsize=8.5,
                fontweight="bold", ha="center", va="center", path_effects=halo, zorder=6)
    ax.set_xlim(*xs); ax.set_ylim(*ys); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_color("#8C8C8C"); sp.set_linewidth(0.6)
    sx, sy, seg = xs[0] + dx * 0.035, ys[0] + dy * 0.05, 1000.0
    for k_ in range(2):
        ax.add_patch(plt.Rectangle((sx + k_ * seg, sy), seg, dy * 0.008, fc="black" if k_ == 0 else "white",
                                   ec="black", lw=0.4, zorder=7))
    for d in (0, 2000):
        ax.text(sx + d, sy + dy * 0.018, f"{d:,}" + (" km" if d == 2000 else ""), ha="center", fontsize=7,
                path_effects=halo, zorder=7)
    nx_, ny_ = xs[1] - dx * 0.035, ys[1] - dy * 0.17
    ax.add_patch(FancyArrow(nx_, ny_, 0, dy * 0.075, width=dx * 0.005, head_width=dx * 0.02, head_length=dy * 0.04,
                            length_includes_head=True, fc="#333333", ec="#333333", zorder=7))
    ax.text(nx_, ny_ + dy * 0.085, "N", ha="center", va="bottom", fontsize=7, zorder=7)
    ax.text(0.006, 0.988, "(a)", transform=ax.transAxes, fontsize=9, fontweight="bold", va="top", zorder=8,
            path_effects=halo)
    # marginal profiles aligned with the map
    from scipy.ndimage import gaussian_filter1d
    xb = np.arange(xs[0], xs[1] + 50, 50.0)
    hx = gaussian_filter1d(np.histogram(x, xb, weights=df.imp2024_km2)[0] / 1e3, 1.0)
    axt.fill_between(xb[:-1] + 25, hx, color=BLUE, alpha=0.25, lw=0); axt.plot(xb[:-1] + 25, hx, color=BLUE, lw=0.8)
    axt.set_xlim(*xs); axt.set_ylim(0, None); axt.set_xticks([])
    axt.set_ylabel("Area\n(10³ km²)", fontsize=7, labelpad=2); axt.grid(axis="y", color="#EEEEEE", lw=0.5)
    axt.yaxis.set_major_locator(matplotlib.ticker.MaxNLocator(3))
    yb_ = np.arange(ys[0], ys[1] + 50, 50.0)
    hy = gaussian_filter1d(np.histogram(y, yb_, weights=df.imp2024_km2)[0] / 1e3, 1.0)
    axr.fill_betweenx(yb_[:-1] + 25, hy, color=BLUE, alpha=0.25, lw=0); axr.plot(hy, yb_[:-1] + 25, color=BLUE, lw=0.8)
    axr.set_ylim(*ys); axr.set_xlim(0, None); axr.set_yticks([])
    axr.set_xlabel("Area (10³ km²)", fontsize=7, labelpad=2); axr.grid(axis="x", color="#EEEEEE", lw=0.5)
    axr.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(2))
    for a_ in (axt, axr):
        for sp in ("top", "right"):
            a_.spines[sp].set_visible(False)
    # legend strip below the map (outside the map frame): row 1 hexagon classes, row 2 region outlines
    axl = fig.add_axes([mx0, my0 - 0.072, mw, 0.05]); axl.axis("off")
    axl.set_xlim(0, mw * W_IN); axl.set_ylim(0, 0.05 * H_IN)                     # inches -> true shapes
    hh = 0.05 * H_IN
    axl.text(0.0, hh * 0.72, "Impervious area per hexagon, 2024 (km²):", va="center", fontsize=7)
    for i, (c, t) in enumerate(zip(CLASS_COLS, CLASS_LABELS)):
        x0 = 2.05 + i * 0.6
        axl.add_patch(MPoly(np.c_[x0 + 0.055 * np.cos(HEX_ANG), hh * 0.72 + 0.055 * np.sin(HEX_ANG)], fc=c, ec="none"))
        axl.text(x0 + 0.09, hh * 0.72, t, va="center", fontsize=7)
    axl.text(0.0, hh * 0.25, "Region (UN M49 sub-region) outline:", va="center", fontsize=7)
    for i, name in enumerate(REGIONS):
        x0 = 2.05 + i * 1.08
        axl.plot([x0 - 0.06, x0 + 0.1], [hh * 0.25] * 2, color=REG_COLS[name], lw=1.4)
        axl.text(x0 + 0.14, hh * 0.25, f"{REGIONS[name][0]} {REGIONS[name][1]}", va="center", fontsize=7)
    # (b) which region is most affected: share of the coastal zone that changed class 1990-2024,
    #     stacked by the class the land changed INTO (same colours as c), ranked
    chg = tr[tr.frm != tr.to]
    allr = tr.groupby("region").km2.sum()
    G = (chg.groupby(["region", "to"]).km2.sum().unstack().reindex(columns=CLS).fillna(0)
         .div(allr, axis=0) * 100).reindex(list(REGIONS))
    G = G.loc[G.sum(1).sort_values().index]                      # most affected on top
    axb = fig.add_axes([0.085, 0.255, 0.185, 0.165])
    left = np.zeros(len(G)); yy = np.arange(len(G))
    for cn in CLS:
        axb.barh(yy, G[cn].values, left=left, height=0.62, color=LULC_COLS[cn], ec="white", lw=0.4)
        left += G[cn].values
    for y_, v in zip(yy, left):
        axb.text(v + left.max() * 0.02, y_, f"{v:.1f}%", va="center", ha="left", fontsize=7, fontweight="bold")
    axb.set_yticks(yy); axb.set_yticklabels([REGIONS[n][0] + " " + SHORT[n] for n in G.index], fontsize=7)
    for t_, n in zip(axb.get_yticklabels(), G.index):
        t_.set_color(REG_COLS[n]); t_.set_fontweight("bold")
    axb.set_xlim(0, left.max() * 1.22); axb.set_ylim(-0.6, len(G) - 0.4)
    axb.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(4))
    axb.set_xlabel("Coastal zone changed 1990–2024 (%)", fontsize=7)
    axb.grid(axis="x", color="#E6E6E6", lw=0.5); axb.set_axisbelow(True)
    for sp in ("top", "right"):
        axb.spines[sp].set_visible(False)
    axb.tick_params(axis="y", length=0)
    fig.text(0.015, 0.455, "(b)", fontsize=9, fontweight="bold", va="top")
    axb.set_title("Most affected regions\n(bar segments = LULC class gained)", fontsize=7, pad=6)
    # (c) LULC transition chord diagrams 1990 -> 2024, one per region
    step = 10 ** np.floor(np.log10(max(tr[tr.frm != tr.to].groupby("region").km2.sum().max() / 40, 1)))
    w_c, x_c0 = 0.172, 0.305
    for i, name in enumerate(REGIONS):
        sub = tr[tr.region == name].pivot(index="frm", columns="to", values="km2").reindex(index=CLS, columns=CLS).fillna(0)
        axc = fig.add_axes([x_c0 + i * w_c, 0.245, w_c, w_c * W_IN / H_IN * 3.45 / 3.5])
        chord(axc, sub.values, f"{REGIONS[name][0]}  {REGIONS[name][1]}", step)
    fig.text(x_c0 - 0.005, 0.455, "(c)", fontsize=9, fontweight="bold", va="top")
    fig.text(x_c0 + 2 * w_c, 0.445, f"LULC transitions 1990→2024 (ribbon width proportional to area; ticks every {step:,.0f} km²)",
             ha="center", fontsize=7)
    axk = fig.add_axes([x_c0, 0.195, 4 * w_c, 0.02]); axk.axis("off"); axk.set_xlim(0, 1); axk.set_ylim(0, 1)
    for i, cn in enumerate(CLS):
        axk.add_patch(plt.Rectangle((0.02 + i * 0.14, 0.25), 0.02, 0.5, fc=LULC_COLS[cn], ec="none"))
        axk.text(0.045 + i * 0.14, 0.5, cn, va="center", fontsize=7)
    out = out or f"{FIG}/Fig1_study_area"
    fig.savefig(out + ".png", dpi=600, bbox_inches="tight", pad_inches=0.03)
    fig.savefig(out + ".pdf", bbox_inches="tight", pad_inches=0.03)
    plt.close(fig)
    return out + ".png"


if __name__ == "__main__":
    if not all(os.path.exists(f"{FIG}/{f}") for f in ("fig1_blocks.csv", "fig1_trans.csv", "fig1_base.npz")):
        prepare()
    df = pd.read_csv(f"{FIG}/fig1_blocks.csv")
    tr = pd.read_csv(f"{FIG}/fig1_trans.csv")
    base = np.load(f"{FIG}/fig1_base.npz", allow_pickle=True)
    p = draw(df, tr, base)
    print("saved", p)
    try:
        from IPython.display import Image, display
        display(Image(p, width=1100))
    except Exception:
        pass


Output hidden; open in https://colab.research.google.com to view.

In [3]:
# status check after reconnect
import os, glob, time
R='/content/drive/MyDrive/AsiaCoastGI'
print('mounted', os.path.ismount('/content/drive'))
for p in sorted(glob.glob(R+'/step5/*2040*'))+sorted(glob.glob(R+'/step5/*.csv')):
    print(time.strftime('%m-%d %H:%M', time.gmtime(os.path.getmtime(p))), os.path.basename(p))
print('--- step6');
for p in sorted(glob.glob(R+'/step6/*.csv')): print(time.strftime('%m-%d %H:%M', time.gmtime(os.path.getmtime(p))), os.path.basename(p))

mounted True
11-18 17:18 sim_EastAsia_FLUS_2040_FPS.tif
11-18 17:13 sim_EastAsia_FLUS_2040_NDS.tif
11-18 17:20 sim_EastAsia_FLUS_2040_PCS.tif
11-18 17:15 sim_EastAsia_FLUS_2040_RDS.tif
11-18 17:49 sim_EastAsia_PLUS_2040_FPS.tif
11-18 17:38 sim_EastAsia_PLUS_2040_NDS.tif
11-18 17:55 sim_EastAsia_PLUS_2040_PCS.tif
11-18 17:44 sim_EastAsia_PLUS_2040_RDS.tif
11-18 18:00 sim_SouthAsia_FLUS_2040_FPS.tif
11-18 17:58 sim_SouthAsia_FLUS_2040_NDS.tif
11-18 18:01 sim_SouthAsia_FLUS_2040_PCS.tif
11-18 17:59 sim_SouthAsia_FLUS_2040_RDS.tif
11-18 18:32 sim_SouthAsia_PLUS_2040_FPS.tif
11-18 18:22 sim_SouthAsia_PLUS_2040_NDS.tif
11-18 18:37 sim_SouthAsia_PLUS_2040_PCS.tif
11-18 18:27 sim_SouthAsia_PLUS_2040_RDS.tif
11-18 19:03 sim_SoutheastAsia_FLUS_2040_FPS.tif
11-18 18:47 sim_SoutheastAsia_FLUS_2040_NDS.tif
11-18 19:09 sim_SoutheastAsia_FLUS_2040_PCS.tif
11-18 18:55 sim_SoutheastAsia_FLUS_2040_RDS.tif
11-18 13:51 sim_SoutheastAsia_PLUS_2040_FPS.tif
11-18 13:33 sim_SoutheastAsia_PLUS_2040_NDS.tif
11-

In [ ]:
# check: impervious share - majority-class maps (simulation) vs area-weighted fractions (Step 3)
import pandas as pd
d = pd.read_csv("/content/drive/MyDrive/AsiaCoastGI/step5/demand.csv")
d = d[d.cls == "Impervious"]
d["start_pct_majority"] = d.start_km2 / d.groupby(["region", "run", "scenario"]).start_km2.transform("sum") if False else d.start_km2
print(d[["region", "run", "scenario", "start_km2", "demand_km2"]].round(0).to_string(index=False))
a = pd.read_csv("/content/drive/MyDrive/AsiaCoastGI/step5/areas.csv"); a = a[(a.cls == "Impervious")]
print(a.pivot_table(index="region", columns=["run", "scenario", "model"], values="km2").round(0).T.to_string())

       region      run scenario  start_km2  demand_km2
  WesternAsia hindcast observed    13088.0     14375.0
  WesternAsia     2040      NDS    14959.0     16522.0
  WesternAsia     2040      RDS    14959.0     16946.0
  WesternAsia     2040      FPS    14959.0     16024.0
  WesternAsia     2040      PCS    14959.0     15975.0
     EastAsia hindcast observed    69192.0     75853.0
     EastAsia     2040      NDS    76050.0     83177.0
     EastAsia     2040      RDS    76050.0     86029.0
     EastAsia     2040      FPS    76050.0     78220.0
     EastAsia     2040      PCS    76050.0     80683.0
    SouthAsia hindcast observed     6825.0      8035.0
    SouthAsia     2040      NDS     8561.0     10128.0
    SouthAsia     2040      RDS     8561.0     10531.0
    SouthAsia     2040      FPS     8561.0      9402.0
    SouthAsia     2040      PCS     8561.0      9579.0
SoutheastAsia hindcast observed    28555.0     31770.0
SoutheastAsia     2040      NDS    33353.0     37519.0
SoutheastA

In [ ]:
# diagnostics 2: how predictable is 2010->2020 impervious growth? (uses Step 5 functions)
from sklearn.metrics import roc_auc_score
R = Region("WesternAsia"); lc_a, lc0, lc1 = R.lc(2000), R.lc(2010), R.lc(2020)
new = (lc1 == IMP) & (lc0 != IMP); cand = lc0 != IMP
nbimp = neighbourhood(lc0, R.nb)[IMP]
print("new impervious cells:", int(new.sum()), "| lost impervious:", int(((lc0 == IMP) & (lc1 != IMP)).sum()))
print("share of new with >=1 imp neighbour in 2010:", round(float((nbimp[new] > 0).mean()), 3),
      "| share of all candidate cells with imp neighbour:", round(float((nbimp[cand] > 0).mean()), 3))
print("P(new | n imp neighbours):", {n: round(float(new[cand & (nbimp == n)].mean()), 4) for n in range(9)})
X0 = R.drivers(2010); rng = np.random.default_rng(1)
idx = rng.choice(np.flatnonzero(cand), 400000, replace=False)
P = ann_suitability(lc0, X0, rng)
print("AUC ANN P_imp:", round(roc_auc_score(new[idx], P[IMP, idx]), 3), "| AUC neighbours:", round(roc_auc_score(new[idx], nbimp[idx]), 3),
      "| AUC ANN*nb:", round(roc_auc_score(new[idx], P[IMP, idx] * (nbimp[idx] + 0.5)), 3))
del P; Xa = R.drivers(2000)
Pl, _ = leas_suitability(lc_a, lc0, Xa, X0, rng, "WA", "diag")
print("AUC LEAS P_imp:", round(roc_auc_score(new[idx], Pl[IMP, idx]), 3), "| AUC LEAS*nb:", round(roc_auc_score(new[idx], Pl[IMP, idx] * (nbimp[idx] + 0.5)), 3))
# upper bound: allocate the observed number of new impervious cells to the top LEAS*nb scores
sc = np.where(cand & ~R.protected, Pl[IMP] * (nbimp + 0.5), -1); n = int(new.sum())
top = np.argpartition(-sc, n)[:n]; hit = new[top].sum()
print("greedy top-n hit rate:", round(hit / n, 3), "-> FoM_imp upper-ish:", round(hit / (2 * n - hit), 3))
del Pl, Xa, X0

10:18:12 WesternAsia: 6,522,170 study cells, protected 12.0%
new impervious cells: 17439 | lost impervious: 3130
share of new with >=1 imp neighbour in 2010: 0.704 | share of all candidate cells with imp neighbour: 0.023
P(new | n imp neighbours): {0: 0.0008, 1: 0.0384, 2: 0.0749, 3: 0.112, 4: 0.1854, 5: 0.242, 6: 0.2874, 7: 0.2969, 8: 0.3079}
10:18:47     ANN trained on 153,026 cells, train acc 0.862
AUC ANN P_imp: 0.952 | AUC neighbours: 0.838 | AUC ANN*nb: 0.953
10:19:17     LEAS Cropland: 20,000 expansion cells, OOB acc 0.880
10:19:29     LEAS Forest: 10,728 expansion cells, OOB acc 0.900
10:19:42     LEAS ShrubGrass: 13,923 expansion cells, OOB acc 0.889
10:19:50     LEAS Wetland: 2,868 expansion cells, OOB acc 0.954
10:19:56     LEAS Water: 2,993 expansion cells, OOB acc 0.953
10:20:11     LEAS Impervious: 20,000 expansion cells, OOB acc 0.955
10:20:19     LEAS Bare: 3,894 expansion cells, OOB acc 0.909
AUC LEAS P_imp: 0.98 | AUC LEAS*nb: 0.98
greedy top-n hit rate: 0.227 -> FoM_

In [ ]:
# diagnostics: how predictable is 2010->2020 impervious growth? (uses Step 5 functions)
from sklearn.metrics import roc_auc_score
R = Region("WesternAsia"); lc_a, lc0, lc1 = R.lc(2000), R.lc(2010), R.lc(2020)
new = (lc1 == IMP) & (lc0 != IMP); cand = lc0 != IMP
nbimp = neighbourhood(lc0, R.nb)[IMP]
print("new impervious cells:", int(new.sum()), "| lost impervious:", int(((lc0 == IMP) & (lc1 != IMP)).sum()))
print("share of new with >=1 imp neighbour in 2010:", round(float((nbimp[new] > 0).mean()), 3),
      "| share of all candidate cells with imp neighbour:", round(float((nbimp[cand] > 0).mean()), 3))
print("P(new | n imp neighbours):", {n: round(float(new[cand & (nbimp == n)].mean()), 4) for n in range(9)})
X0 = R.drivers(2010); rng = np.random.default_rng(1)
idx = rng.choice(np.flatnonzero(cand), 400000, replace=False)
P = ann_suitability(lc0, X0, rng)
print("AUC ANN P_imp:", round(roc_auc_score(new[idx], P[IMP, idx]), 3), "| AUC neighbours:", round(roc_auc_score(new[idx], nbimp[idx]), 3),
      "| AUC ANN*nb:", round(roc_auc_score(new[idx], P[IMP, idx] * (nbimp[idx] + 0.5)), 3))
del P; Xa = R.drivers(2000)
Pl, _ = leas_suitability(lc_a, lc0, Xa, X0, rng, "WA", "diag")
print("AUC LEAS P_imp:", round(roc_auc_score(new[idx], Pl[IMP, idx]), 3), "| AUC LEAS*nb:", round(roc_auc_score(new[idx], Pl[IMP, idx] * (nbimp[idx] + 0.5)), 3))
# upper bound: allocate the observed number of new impervious cells to the top LEAS*nb scores
sc = np.where(cand & ~R.protected, Pl[IMP] * (nbimp + 0.5), -1); n = int(new.sum())
top = np.argpartition(-sc, n)[:n]; hit = new[top].sum()
print("greedy top-n hit rate:", round(hit / n, 3), "-> FoM_imp upper-ish:", round(hit / (2 * n - hit), 3))
del Pl, Xa, X0

['/content/tmp_drivers/wdpa/WDPA_Nov2026_Public_shp_0/WDPA_Nov2026_Public_shp-polygons.shp', '/content/tmp_drivers/wdpa/WDPA_Nov2026_Public_shp_1/WDPA_Nov2026_Public_shp-polygons.shp', '/content/tmp_drivers/wdpa/WDPA_Nov2026_Public_shp_2/WDPA_Nov2026_Public_shp-polygons.shp']
166783 ['SITE_ID', 'SITE_PID', 'SITE_TYPE', 'NAME_ENG', 'NAME', 'DESIG', 'DESIG_ENG', 'DESIG_TYPE', 'IUCN_CAT', 'INT_CRIT', 'REALM', 'REP_M_AREA', 'GIS_M_AREA', 'REP_AREA', 'GIS_AREA', 'NO_TAKE', 'NO_TK_AREA', 'STATUS', 'STATUS_YR', 'GOV_TYPE', 'GOVSUBTYPE', 'OWN_TYPE', 'OWNSUBTYPE', 'MANG_AUTH', 'MANG_PLAN', 'VERIF', 'METADATAID', 'PRNT_ISO3', 'ISO3', 'SUPP_INFO', 'CONS_OBJ', 'INLND_WTRS', 'OECM_ASMT']
SITE_TYPE {'PA': 20000}
DESIG_TYPE {'National': 18987, 'International': 1010, 'Not Applicable': 2, 'Regional': 1}
IUCN_CAT {'IV': 6198, 'Not Reported': 4111, 'II': 2350, 'Ia': 2173, 'V': 1668, 'VI': 1115, 'III': 887, 'Ib': 756, 'Not Assigned': 491, 'Not Applicable': 251}
REALM {'Terrestrial': 17990, 'Coastal': 1362

In [ ]:
# Check: are the 6 "missing" tiles really absent from the GLC_FCS30D archive?
for tid, L in [("E60N25", 60), ("E130N30", 130), ("E130N5", 130), ("E140N35", 140), ("E140N30", 140), ("E140N25", 140)]:
    zname, names = zip_members(L)
    hits = [n for n in names if tid in n]
    near = sorted({re.search(r"_(E\d+[NS]\d+)_", n).group(1) for n in names if re.search(r"_(E\d+[NS]\d+)_", n)})
    print(tid, "in", zname, "->", hits or "NOT IN ARCHIVE")
print("tiles present in E140-E145 zip:", [t for t in near])

E60N25 in GLC_FCS30D_19852022maps_E60-E65.zip -> NOT IN ARCHIVE
E130N30 in GLC_FCS30D_19852022maps_E130-E135.zip -> NOT IN ARCHIVE
E130N5 in GLC_FCS30D_19852022maps_E130-E135.zip -> ['GLC_FCS30D_19852022maps_E130-E135/GLC_FCS30D_19852000_E130N50_5years_V1.1.tif', 'GLC_FCS30D_19852022maps_E130-E135/GLC_FCS30D_19852000_E130N55_5years_V1.1.tif', 'GLC_FCS30D_19852022maps_E130-E135/GLC_FCS30D_20002022_E130N50_Annual_V1.1.tif', 'GLC_FCS30D_19852022maps_E130-E135/GLC_FCS30D_20002022_E130N55_Annual_V1.1.tif']
E140N35 in GLC_FCS30D_19852022maps_E140-E145.zip -> NOT IN ARCHIVE
E140N30 in GLC_FCS30D_19852022maps_E140-E145.zip -> NOT IN ARCHIVE
E140N25 in GLC_FCS30D_19852022maps_E140-E145.zip -> NOT IN ARCHIVE
tiles present in E140-E145 zip: ['E140N0', 'E140N40', 'E140N45', 'E140N50', 'E140N55', 'E140N60', 'E140N65', 'E140N70', 'E140N75', 'E140N80', 'E140S10', 'E140S15', 'E140S20', 'E140S25', 'E140S30', 'E140S35', 'E140S40', 'E140S5', 'E145N0', 'E145N45', 'E145N50', 'E145N60', 'E145N65', 'E145N70'

In [ ]:
# QC of the smoke-test tiles
import numpy as np
CLS = ["Crop","Forest","Shrub","Wetland","Water","Imperv","Bare"]
for t in ["E25N45", "E25N40"]:
    d = np.load(f"{CACHE}/WesternAsia_{t}.npz")
    print("==", t, "dest window", d["win"])
    for y in YEARS:
        a = d[f"area_{y}"]; p = d[f"pct_{y}"]; v = p[0] != 255
        print(y, "30m km2:", {c: round(x) for c, x in zip(CLS, a[1:])}, "| nodata km2:", round(a[0]),
              "| 300m cells:", int(v.sum()), "pct-sum ok:", bool(np.all(np.abs(p[:, v].astype(int).sum(0) - 100) <= 4)))
    tr = d["tr_1990_2022"].reshape(8, 8)
    print("1990->2022 transitions total km2:", round(tr.sum()), "= 1990 area:", round(d["area_1990"].sum()))
    print("  cropland->impervious:", round(tr[1, 6], 1), "km2 | forest->cropland:", round(tr[2, 1], 1), "km2")

== E25N45 dest window [ 589  795 1342  477]
1990 30m km2: {'Crop': 6163, 'Forest': 7597, 'Shrub': 3861, 'Wetland': 104, 'Water': 811, 'Imperv': 737, 'Bare': 11} | nodata km2: 0 | 300m cells: 638316 pct-sum ok: True
2000 30m km2: {'Crop': 6162, 'Forest': 7873, 'Shrub': 3088, 'Wetland': 116, 'Water': 801, 'Imperv': 1233, 'Bare': 12} | nodata km2: 0 | 300m cells: 638316 pct-sum ok: True
2010 30m km2: {'Crop': 6228, 'Forest': 7695, 'Shrub': 2990, 'Wetland': 113, 'Water': 816, 'Imperv': 1428, 'Bare': 15} | nodata km2: 0 | 300m cells: 638316 pct-sum ok: True
2020 30m km2: {'Crop': 6255, 'Forest': 7463, 'Shrub': 3076, 'Wetland': 130, 'Water': 801, 'Imperv': 1542, 'Bare': 18} | nodata km2: 0 | 300m cells: 638316 pct-sum ok: True
2022 30m km2: {'Crop': 6281, 'Forest': 7436, 'Shrub': 3094, 'Wetland': 128, 'Water': 802, 'Imperv': 1525, 'Bare': 20} | nodata km2: 0 | 300m cells: 638316 pct-sum ok: True
1990->2022 transitions total km2: 19285 = 1990 area: 19285
  cropland->impervious: 433.2 km2 | fo